<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/15_ML_Utility_TSTR_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [65]:
# ==================================================================================================
# NOTEBOOK 15 — ML UTILITY / TSTR EVALUATION
# Research: SPP-GAN
#
# Scope:
#   1. Evaluate synthetic-data utility using Train-on-Synthetic, Test-on-Real (TSTR).
#   2. Evaluate the real-data reference using Train-on-Real, Test-on-Real (TRTR).
#   3. Report Accuracy, Macro-F1, and Weighted-F1 as primary classification metrics.
#   4. Report Precision, Recall, and ROC-AUC where supported.
#   5. Evaluate Adult Income, Bank Marketing, and Diabetes 130-US.
#   6. Use five predetermined repetition seeds: 3026–3030.
#   7. Preserve consistent target definitions and held-out real test sets across methods.
#   8. Save evaluation results and validation artifacts for reproducibility.
#
# Methodology reference:
# SPP-GAN Methodology, Sections 4.7, 4.12, and 4.13.
#
# Important:
#   The real test set is reserved for final evaluation and must not be used for model selection.
#   TSTR utility results do not, by themselves, establish privacy or statistical fidelity.
# ==================================================================================================

NOTEBOOK_ID = "notebook_15"
NOTEBOOK_TITLE = "ML Utility / TSTR Evaluation"
NOTEBOOK_SCOPE = "TSTR and TRTR classification utility evaluation"

DATASETS = {
    "adult_income": {"target": "income"},
    "bank_marketing": {"target": "y"},
    "diabetes_130us": {"target": "readmitted"},
}

MASTER_SEED = 2025
REPETITION_SEEDS = [3026, 3027, 3028, 3029, 3030]
N_REPETITIONS = len(REPETITION_SEEDS)

PRIMARY_METRICS = ["accuracy", "macro_f1", "weighted_f1"]
SECONDARY_METRICS = ["precision", "recall", "roc_auc"]

print("=" * 90)
print(NOTEBOOK_TITLE.upper())
print("=" * 90)
print("Research framework :", "SPP-GAN")
print("Evaluation protocols:", "TSTR, TRTR")
print("Datasets            :", ", ".join(DATASETS.keys()))
print("Primary metrics     :", ", ".join(PRIMARY_METRICS))
print("Secondary metrics   :", ", ".join(SECONDARY_METRICS))
print("Repetitions         :", N_REPETITIONS)
print("Repetition seeds    :", REPETITION_SEEDS)
print("Scope status        :", "INITIALIZED")

ML UTILITY / TSTR EVALUATION
Research framework : SPP-GAN
Evaluation protocols: TSTR, TRTR
Datasets            : adult_income, bank_marketing, diabetes_130us
Primary metrics     : accuracy, macro_f1, weighted_f1
Secondary metrics   : precision, recall, roc_auc
Repetitions         : 5
Repetition seeds    : [3026, 3027, 3028, 3029, 3030]
Scope status        : INITIALIZED


In [66]:
# ==================================================================================================
# SECTION 2 — CONFIGURATION & ARTIFACT DISCOVERY
# SPP-GAN | Notebook 15 — ML Utility / TSTR Evaluation
#
# Frozen responsibilities:
# 1. Define reproducible experiment configuration.
# 2. Discover and validate canonical Notebook 02 and Notebook 13 inputs.
# 3. Validate real split schemas and synthetic feature schemas.
# 4. Report exact schema differences without silently modifying columns.
# 5. Validate upstream metadata artifacts.
# 6. Persist configuration, validation reports, and artifact fingerprints.
#
# Safety:
# - No model training or TSTR metric calculation.
# - No upstream notebook modifications.
# - No automatic column dropping, renaming, or reordering.
# - Schema mismatches remain BLOCKED until explicitly resolved.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import pandas as pd
import numpy as np

# --------------------------------------------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Research")

NB02_ROOT = PROJECT_ROOT / "data" / "processed" / "notebook_02"
NB12_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_12"
NB13_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_13"
NB14_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_14"
NB15_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_15"

NB15_CONFIG_DIR = NB15_ROOT / "config"
NB15_VALIDATION_DIR = NB15_ROOT / "validation"
NB15_MANIFEST_DIR = NB15_ROOT / "manifests"
NB15_METADATA_DIR = NB15_ROOT / "metadata"

DATASETS = {
    "adult_income": {"target": "income"},
    "bank_marketing": {"target": "y"},
    "diabetes_130us": {"target": "readmitted"},
}

MASTER_SEED = 2025
REPETITION_SEEDS = [3026, 3027, 3028, 3029, 3030]
N_REPETITIONS = len(REPETITION_SEEDS)

SPLIT_FRACTIONS = {
    "train": 0.70,
    "validation": 0.15,
    "test": 0.15,
}

PRIMARY_METRICS = ["accuracy", "macro_f1", "weighted_f1"]
SECONDARY_METRICS = ["precision", "recall", "roc_auc"]

EXPECTED_SPLITS = ("train", "validation", "test")
ROW_ID_COLUMN = "__original_row_id__"

assert PROJECT_ROOT.is_dir(), f"Project root not found: {PROJECT_ROOT}"
assert len(REPETITION_SEEDS) == len(set(REPETITION_SEEDS))
assert N_REPETITIONS == 5
assert np.isclose(sum(SPLIT_FRACTIONS.values()), 1.0)

# --------------------------------------------------------------------------------------------------
# 2. PATH REGISTRY
# --------------------------------------------------------------------------------------------------

REAL_SPLIT_PATHS = {
    dataset_id: {
        split: NB02_ROOT / "native" / dataset_id / f"{split}.csv"
        for split in EXPECTED_SPLITS
    }
    for dataset_id in DATASETS
}

SYNTHETIC_PATHS = {
    dataset_id: (
        NB13_ROOT
        / "synthetic_datasets"
        / f"{dataset_id}_sppgan_synthetic.csv"
    )
    for dataset_id in DATASETS
}

UPSTREAM_METADATA_PATHS = {
    "generation_manifest_csv":
        NB13_ROOT / "manifests" / "synthetic_generation_manifest.csv",
    "generation_manifest_json":
        NB13_ROOT / "manifests" / "synthetic_generation_manifest.json",
    "completion_summary_json":
        NB13_ROOT / "metadata" / "notebook_13_completion_summary.json",
    "generation_metadata_json":
        NB13_ROOT / "metadata" / "sppgan_synthetic_generation_metadata.json",
    "final_verification_csv":
        NB13_ROOT / "validation" / "final_verification.csv",
    "final_verification_json":
        NB13_ROOT / "validation" / "final_verification_summary.json",
}

# --------------------------------------------------------------------------------------------------
# 3. REPRODUCIBLE VALIDATION HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    """Return the SHA-256 fingerprint of a file."""
    digest = hashlib.sha256()

    with open(path, "rb") as file_obj:
        while True:
            chunk = file_obj.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def read_csv_checked(path):
    """Read a CSV and return its dataframe, status, and diagnostic."""
    path = Path(path)

    if not path.is_file():
        return None, "MISSING", f"File not found: {path}"

    try:
        dataframe = pd.read_csv(path, low_memory=False)
    except Exception as exc:
        return None, "READ_ERROR", f"{type(exc).__name__}: {exc}"

    if dataframe.empty:
        return dataframe, "EMPTY", "CSV contains no data rows."

    if not dataframe.columns.is_unique:
        return dataframe, "DUPLICATE_COLUMNS", "Duplicate column names detected."

    if any(
        not isinstance(column, str) or not column.strip()
        for column in dataframe.columns
    ):
        return dataframe, "INVALID_COLUMN_NAMES", (
            "Column names must be non-empty strings."
        )

    return dataframe, "PASS", "Readable, non-empty CSV; column names validated."


def compare_schema(reference_columns, candidate_columns):
    """Compare ordered schemas and identify missing or extra columns."""
    reference_columns = list(reference_columns)
    candidate_columns = list(candidate_columns)

    reference_set = set(reference_columns)
    candidate_set = set(candidate_columns)

    return {
        "exact_match": reference_columns == candidate_columns,
        "same_columns_different_order": (
            reference_set == candidate_set
            and reference_columns != candidate_columns
        ),
        "missing_columns": [
            column for column in reference_columns
            if column not in candidate_set
        ],
        "extra_columns": [
            column for column in candidate_columns
            if column not in reference_set
        ],
        "reference_column_count": len(reference_columns),
        "candidate_column_count": len(candidate_columns),
    }


def atomic_write_json(path, payload):
    """Write JSON through a temporary file to avoid partial outputs."""
    path = Path(path)
    temp_path = path.with_suffix(path.suffix + ".tmp")

    with open(temp_path, "w", encoding="utf-8") as file_obj:
        json.dump(
            payload,
            file_obj,
            indent=2,
            ensure_ascii=False,
            default=str,
        )

    temp_path.replace(path)


def atomic_write_csv(path, dataframe):
    """Write CSV through a temporary file to avoid partial outputs."""
    path = Path(path)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    dataframe.to_csv(temp_path, index=False)
    temp_path.replace(path)


# --------------------------------------------------------------------------------------------------
# 4. PRINT CONFIGURATION
# --------------------------------------------------------------------------------------------------

print("=" * 100)
print("SECTION 2 — CONFIGURATION & ARTIFACT DISCOVERY")
print("=" * 100)

for label, path in [
    ("MyDrive", Path("/content/drive/MyDrive")),
    ("Project", PROJECT_ROOT),
    ("Notebook 02", NB02_ROOT),
    ("Notebook 12", NB12_ROOT),
    ("Notebook 13", NB13_ROOT),
    ("Notebook 14", NB14_ROOT),
    ("Notebook 15", NB15_ROOT),
]:
    print(f"{label:<20}: {path} | Exists: {path.exists()}")

print("\nEXPERIMENT CONFIGURATION")
print("Datasets          :", list(DATASETS))
print("Targets           :", {
    dataset_id: config["target"]
    for dataset_id, config in DATASETS.items()
})
print("Master seed       :", MASTER_SEED)
print("Repetition seeds  :", REPETITION_SEEDS)
print("Split fractions   :", SPLIT_FRACTIONS)
print("Primary metrics   :", PRIMARY_METRICS)
print("Secondary metrics :", SECONDARY_METRICS)

# --------------------------------------------------------------------------------------------------
# 5. DISCOVER AND VALIDATE INPUT FILES
# --------------------------------------------------------------------------------------------------

loaded_frames = {}
input_validation_records = []
artifact_discovery_records = []

for dataset_id, dataset_config in DATASETS.items():
    target = dataset_config["target"]

    paths = {
        "real_train": REAL_SPLIT_PATHS[dataset_id]["train"],
        "real_validation": REAL_SPLIT_PATHS[dataset_id]["validation"],
        "real_test": REAL_SPLIT_PATHS[dataset_id]["test"],
        "synthetic_train": SYNTHETIC_PATHS[dataset_id],
    }

    for input_role, path in paths.items():
        dataframe, read_status, details = read_csv_checked(path)

        if dataframe is not None and read_status == "PASS":
            loaded_frames[(dataset_id, input_role)] = dataframe

        target_present = (
            dataframe is not None and target in dataframe.columns
        )

        if read_status == "PASS" and not target_present:
            read_status = "TARGET_MISSING"
            details = f"Required target column '{target}' is absent."

        exists = path.is_file()

        input_validation_records.append({
            "dataset_id": dataset_id,
            "input_role": input_role,
            "path": str(path),
            "read_status": read_status,
            "rows": len(dataframe) if dataframe is not None else None,
            "columns": len(dataframe.columns) if dataframe is not None else None,
            "target_column": target,
            "target_present": bool(target_present),
            "sha256": sha256_file(path) if exists else None,
            "details": details,
        })

        artifact_discovery_records.append({
            "dataset_id": dataset_id,
            "artifact_role": input_role,
            "path": str(path),
            "exists": exists,
            "bytes": path.stat().st_size if exists else None,
            "sha256": sha256_file(path) if exists else None,
        })

INPUT_VALIDATION_DF = pd.DataFrame(input_validation_records)
ARTIFACT_DISCOVERY_DF = pd.DataFrame(artifact_discovery_records)

# --------------------------------------------------------------------------------------------------
# 6. COMPARE REAL AND SYNTHETIC SCHEMAS
# --------------------------------------------------------------------------------------------------

schema_difference_records = []
dataset_readiness_records = []

for dataset_id, dataset_config in DATASETS.items():
    target = dataset_config["target"]

    frames = {
        role: loaded_frames.get((dataset_id, role))
        for role in (
            "real_train",
            "real_validation",
            "real_test",
            "synthetic_train",
        )
    }

    invalid_roles = [
        role
        for role, dataframe in frames.items()
        if dataframe is None or target not in dataframe.columns
    ]

    real_split_schema_match = False
    synthetic_schema_match = False
    target_values_valid = False
    row_id_values_valid = False
    schema_details = []

    if not invalid_roles:
        real_train = frames["real_train"]
        real_train_columns = list(real_train.columns)

        # Compare each real split and the synthetic dataset to real training.
        for role in ("real_validation", "real_test", "synthetic_train"):
            candidate = frames[role]
            comparison = compare_schema(
                real_train_columns,
                candidate.columns,
            )

            schema_difference_records.append({
                "dataset_id": dataset_id,
                "comparison": f"real_train_vs_{role}",
                "exact_schema_match": comparison["exact_match"],
                "same_columns_different_order":
                    comparison["same_columns_different_order"],
                "reference_columns": comparison["reference_column_count"],
                "candidate_columns": comparison["candidate_column_count"],
                "missing_columns": json.dumps(
                    comparison["missing_columns"]
                ),
                "extra_columns": json.dumps(
                    comparison["extra_columns"]
                ),
            })

            if not comparison["exact_match"]:
                schema_details.append(
                    f"{role}: missing={comparison['missing_columns']}; "
                    f"extra={comparison['extra_columns']}; "
                    f"same_columns_different_order="
                    f"{comparison['same_columns_different_order']}"
                )

        real_split_schema_match = all(
            list(frames[role].columns) == real_train_columns
            for role in ("real_validation", "real_test")
        )

        synthetic_comparison = compare_schema(
            real_train_columns,
            frames["synthetic_train"].columns,
        )
        synthetic_schema_match = synthetic_comparison["exact_match"]

        target_values_valid = all(
            frames[role][target].notna().all()
            for role in frames
        )

        # Diagnose the known row-ID difference without removing it.
        row_id_in_real = ROW_ID_COLUMN in real_train.columns
        row_id_in_synthetic = (
            ROW_ID_COLUMN in frames["synthetic_train"].columns
        )

        row_id_values_valid = (
            row_id_in_real
            and not row_id_in_synthetic
            and all(
                ROW_ID_COLUMN in frames[role].columns
                for role in ("real_validation", "real_test")
            )
        )

    if invalid_roles:
        schema_status = "INPUT_INVALID"
    elif not real_split_schema_match:
        schema_status = "REAL_SPLIT_SCHEMA_MISMATCH"
    elif not synthetic_schema_match and row_id_values_valid:
        schema_status = "ROW_ID_SCHEMA_MISMATCH"
    elif not synthetic_schema_match:
        schema_status = "SCHEMA_MISMATCH"
    elif not target_values_valid:
        schema_status = "TARGET_MISSING_VALUES"
    else:
        schema_status = "PASS"

    tstr_ready = (
        not invalid_roles
        and real_split_schema_match
        and synthetic_schema_match
        and target_values_valid
    )

    dataset_readiness_records.append({
        "dataset_id": dataset_id,
        "target_column": target,
        "unavailable_or_invalid_roles": json.dumps(invalid_roles),
        "real_split_schema_status": (
            "PASS" if real_split_schema_match else "MISMATCH_OR_NOT_CHECKED"
        ),
        "synthetic_schema_status": schema_status,
        "target_values_valid": target_values_valid,
        "row_id_difference_detected": row_id_values_valid,
        "tstr_ready": tstr_ready,
        "details": " | ".join(schema_details) if schema_details else (
            "All required inputs and schemas validated."
            if tstr_ready else
            "Required inputs are unavailable, invalid, or schema-incompatible."
        ),
    })

SCHEMA_DIFFERENCES_DF = pd.DataFrame(schema_difference_records)
DATASET_READINESS_DF = pd.DataFrame(dataset_readiness_records)

# --------------------------------------------------------------------------------------------------
# 7. VALIDATE UPSTREAM METADATA FILES
# --------------------------------------------------------------------------------------------------

metadata_validation_records = []

for artifact_name, path in UPSTREAM_METADATA_PATHS.items():
    exists = path.is_file()
    validation_status = "MISSING"
    details = "Artifact does not exist."

    if exists:
        try:
            if path.suffix.lower() == ".csv":
                metadata_df = pd.read_csv(path, low_memory=False)

                if metadata_df.empty:
                    validation_status = "EMPTY"
                    details = "CSV is readable but contains no rows."
                else:
                    validation_status = "PASS"
                    details = (
                        f"Readable CSV; rows={len(metadata_df)}, "
                        f"columns={len(metadata_df.columns)}."
                    )

            elif path.suffix.lower() == ".json":
                with open(path, "r", encoding="utf-8") as file_obj:
                    metadata_obj = json.load(file_obj)

                if isinstance(metadata_obj, (dict, list)):
                    validation_status = "PASS"
                    details = (
                        "Valid JSON syntax; "
                        f"root type={type(metadata_obj).__name__}."
                    )
                else:
                    validation_status = "INVALID_STRUCTURE"
                    details = "JSON root must be an object or list."

            else:
                validation_status = "UNSUPPORTED_FORMAT"
                details = f"Unsupported format: {path.suffix}"

        except Exception as exc:
            validation_status = "READ_ERROR"
            details = f"{type(exc).__name__}: {exc}"

    metadata_validation_records.append({
        "artifact_name": artifact_name,
        "path": str(path),
        "exists": exists,
        "validation_status": validation_status,
        "sha256": sha256_file(path) if exists else None,
        "details": details,
    })

UPSTREAM_METADATA_VALIDATION_DF = pd.DataFrame(
    metadata_validation_records
)

# --------------------------------------------------------------------------------------------------
# 8. CREATE REPRODUCIBILITY CONFIGURATION AND STATUS
# --------------------------------------------------------------------------------------------------

tstr_ready_count = int(DATASET_READINESS_DF["tstr_ready"].sum())

CONFIGURATION = {
    "notebook_id": "notebook_15",
    "notebook_title": "ML Utility / TSTR Evaluation",
    "notebook_scope": "TSTR and TRTR classification utility evaluation",
    "project_root": str(PROJECT_ROOT),
    "datasets": DATASETS,
    "master_seed": MASTER_SEED,
    "repetition_seeds": REPETITION_SEEDS,
    "n_repetitions": N_REPETITIONS,
    "split_fractions": SPLIT_FRACTIONS,
    "primary_metrics": PRIMARY_METRICS,
    "secondary_metrics": SECONDARY_METRICS,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}

STATUS_PAYLOAD = {
    **CONFIGURATION,
    "tstr_ready_datasets": tstr_ready_count,
    "total_datasets": len(DATASETS),
    "status": (
        "READY" if tstr_ready_count == len(DATASETS) else "BLOCKED"
    ),
    "training_performed": False,
    "upstream_notebooks_modified": False,
    "reason": (
        "All datasets passed input and schema validation."
        if tstr_ready_count == len(DATASETS)
        else "Resolve schema or input validation issues before TSTR."
    ),
}

# --------------------------------------------------------------------------------------------------
# 9. SAVE AUDIT ARTIFACTS
# --------------------------------------------------------------------------------------------------

for output_dir in (
    NB15_CONFIG_DIR,
    NB15_VALIDATION_DIR,
    NB15_MANIFEST_DIR,
    NB15_METADATA_DIR,
):
    output_dir.mkdir(parents=True, exist_ok=True)

atomic_write_json(
    NB15_CONFIG_DIR / "notebook_15_configuration.json",
    CONFIGURATION,
)

atomic_write_csv(
    NB15_VALIDATION_DIR / "input_schema_validation.csv",
    INPUT_VALIDATION_DF,
)

atomic_write_csv(
    NB15_VALIDATION_DIR / "schema_differences.csv",
    SCHEMA_DIFFERENCES_DF,
)

atomic_write_csv(
    NB15_VALIDATION_DIR / "dataset_readiness.csv",
    DATASET_READINESS_DF,
)

atomic_write_csv(
    NB15_VALIDATION_DIR / "upstream_metadata_validation.csv",
    UPSTREAM_METADATA_VALIDATION_DF,
)

atomic_write_csv(
    NB15_MANIFEST_DIR / "artifact_discovery.csv",
    ARTIFACT_DISCOVERY_DF,
)

atomic_write_json(
    NB15_VALIDATION_DIR / "upstream_artifact_status.json",
    STATUS_PAYLOAD,
)

# --------------------------------------------------------------------------------------------------
# 10. DISPLAY FINAL RESULTS
# --------------------------------------------------------------------------------------------------

print("\nINPUT VALIDATION")
print(INPUT_VALIDATION_DF[
    [
        "dataset_id",
        "input_role",
        "read_status",
        "rows",
        "columns",
        "target_column",
        "target_present",
        "details",
    ]
].to_string(index=False))

print("\nSCHEMA DIFFERENCES — EXACT COLUMN NAMES")
print(
    SCHEMA_DIFFERENCES_DF.to_string(index=False)
    if not SCHEMA_DIFFERENCES_DF.empty
    else "No schema comparisons could be completed."
)

print("\nDATASET READINESS")
print(DATASET_READINESS_DF.to_string(index=False))

print("\nUPSTREAM METADATA VALIDATION")
print(UPSTREAM_METADATA_VALIDATION_DF.to_string(index=False))

print("\n" + "=" * 100)
print("SECTION 2 — FINAL STATUS")
print("=" * 100)
print(f"Project root        : {PROJECT_ROOT}")
print(f"TSTR-ready datasets : {tstr_ready_count}/{len(DATASETS)}")
print("Upstream modified   : False")
print("Training performed  : False")
print(f"STATUS              : {STATUS_PAYLOAD['status']}")
print("Audit artifacts     : 7 saved")
print("Schema differences  :", NB15_VALIDATION_DIR / "schema_differences.csv")
print("Dataset readiness   :", NB15_VALIDATION_DIR / "dataset_readiness.csv")

if tstr_ready_count < len(DATASETS):
    print("Do not train classifiers or calculate TSTR metrics.")
    print(
        "Detected difference: '__original_row_id__' exists in real splits "
        "but not in synthetic datasets."
    )
    print(
        "This section deliberately does not remove the row-ID column. "
        "Confirm the intended feature schema in the upstream manifests "
        "before resolving this mismatch."
    )

print("=" * 100)

SECTION 2 — CONFIGURATION & ARTIFACT DISCOVERY
MyDrive             : /content/drive/MyDrive | Exists: True
Project             : /content/drive/MyDrive/SPP_GAN_Research | Exists: True
Notebook 02         : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02 | Exists: True
Notebook 12         : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_12 | Exists: True
Notebook 13         : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13 | Exists: True
Notebook 14         : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14 | Exists: True
Notebook 15         : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15 | Exists: True

EXPERIMENT CONFIGURATION
Datasets          : ['adult_income', 'bank_marketing', 'diabetes_130us']
Targets           : {'adult_income': 'income', 'bank_marketing': 'y', 'diabetes_130us': 'readmitted'}
Master seed       : 2025
Repetition seeds  : [3026, 3027, 3028, 3029, 3030]
Split f

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag


INPUT VALIDATION
    dataset_id      input_role read_status  rows  columns target_column  target_present                                          details
  adult_income      real_train        PASS 34189       16        income            True Readable, non-empty CSV; column names validated.
  adult_income real_validation        PASS  7326       16        income            True Readable, non-empty CSV; column names validated.
  adult_income       real_test        PASS  7327       16        income            True Readable, non-empty CSV; column names validated.
  adult_income synthetic_train        PASS 34189       15        income            True Readable, non-empty CSV; column names validated.
bank_marketing      real_train        PASS 31647       18             y            True Readable, non-empty CSV; column names validated.
bank_marketing real_validation        PASS  6782       18             y            True Readable, non-empty CSV; column names validated.
bank_marketing       re

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [67]:
# ==================================================================================================
# SECTION 3 — LOAD REAL DATA
# ==================================================================================================

from pathlib import Path
import hashlib
import pandas as pd

print("=" * 100)
print("SECTION 3 — LOAD REAL DATA")
print("=" * 100)

# --------------------------------------------------------------------------------
# 1. CONFIGURE PATHS AND INITIALIZE OUTPUTS
# --------------------------------------------------------------------------------

NATIVE_ROOT = NB02_ROOT / "native"
INPUT_AUDIT_DIR = NB15_ROOT / "inputs"
INPUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

REAL_TRAIN = {}
REAL_TEST = {}
REAL_DATA_AUDIT = []
REAL_SCHEMA_AUDIT = []

REQUIRED_SPLITS = ("train", "test")

# --------------------------------------------------------------------------------
# 2. VALIDATE CONFIGURATION
# --------------------------------------------------------------------------------

if not isinstance(DATASETS, dict) or not DATASETS:
    raise ValueError("DATASETS must be a non-empty dictionary.")

for dataset_id, metadata in DATASETS.items():
    if not isinstance(metadata, dict):
        raise TypeError(f"{dataset_id}: dataset metadata must be a dictionary.")

    target = metadata.get("target")

    if not isinstance(target, str) or not target.strip():
        raise ValueError(f"{dataset_id}: target must be a non-empty string.")

# --------------------------------------------------------------------------------
# 3. LOAD AND VALIDATE CANONICAL REAL SPLITS
# --------------------------------------------------------------------------------

for dataset_id, metadata in DATASETS.items():

    target = metadata["target"]
    dataset_root = NATIVE_ROOT / dataset_id

    split_paths = {
        "train": dataset_root / "train.csv",
        "test": dataset_root / "test.csv",
    }

    loaded_splits = {}

    for split_name, path in split_paths.items():

        audit = {
            "dataset_id": dataset_id,
            "split": split_name,
            "target": target,
            "path": str(path),
            "exists": path.is_file(),
            "rows": 0,
            "columns": 0,
            "target_missing": None,
            "duplicate_columns": None,
            "sha256": None,
            "status": "FAIL",
            "details": "",
        }

        try:
            if not path.is_file():
                raise FileNotFoundError(
                    f"Missing real {split_name} dataset: {path}"
                )

            # Read the original CSV without modifying its contents.
            df = pd.read_csv(path, low_memory=False)

            if df.empty:
                raise ValueError(
                    f"{dataset_id}/{split_name}: dataset is empty."
                )

            if len(df.columns) == 0:
                raise ValueError(
                    f"{dataset_id}/{split_name}: no columns found."
                )

            if df.columns.duplicated().any():
                duplicates = df.columns[
                    df.columns.duplicated()
                ].tolist()

                raise ValueError(
                    f"{dataset_id}/{split_name}: duplicate columns: "
                    f"{duplicates}"
                )

            invalid_names = [
                col for col in df.columns
                if not isinstance(col, str) or not col.strip()
            ]

            if invalid_names:
                raise ValueError(
                    f"{dataset_id}/{split_name}: invalid column names: "
                    f"{invalid_names}"
                )

            if target not in df.columns:
                raise ValueError(
                    f"{dataset_id}/{split_name}: target '{target}' is missing."
                )

            target_missing = int(df[target].isna().sum())

            if target_missing == len(df):
                raise ValueError(
                    f"{dataset_id}/{split_name}: target contains no valid labels."
                )

            # Hash the exact source file for reproducibility.
            file_hash = hashlib.sha256(path.read_bytes()).hexdigest()

            audit.update({
                "rows": len(df),
                "columns": len(df.columns),
                "target_missing": target_missing,
                "duplicate_columns": False,
                "sha256": file_hash,
                "status": "PASS",
                "details": (
                    "Readable, non-empty CSV; target and column names validated."
                ),
            })

            loaded_splits[split_name] = df

        except Exception as exc:
            audit["details"] = f"{type(exc).__name__}: {exc}"
            REAL_DATA_AUDIT.append(audit)

            # Persist failure evidence before stopping.
            pd.DataFrame(REAL_DATA_AUDIT).to_csv(
                INPUT_AUDIT_DIR / "real_data_audit.csv",
                index=False,
            )
            raise

        REAL_DATA_AUDIT.append(audit)

    # --------------------------------------------------------------------------------
    # 4. VALIDATE TRAIN/TEST SCHEMA CONSISTENCY
    # --------------------------------------------------------------------------------

    train_df = loaded_splits["train"]
    test_df = loaded_splits["test"]

    train_columns = list(train_df.columns)
    test_columns = list(test_df.columns)

    same_columns = set(train_columns) == set(test_columns)
    same_order = train_columns == test_columns

    missing_from_test = [
        col for col in train_columns if col not in test_columns
    ]

    extra_in_test = [
        col for col in test_columns if col not in train_columns
    ]

    schema_status = (
        "PASS"
        if same_columns and same_order
        else "COLUMN_ORDER_MISMATCH"
        if same_columns
        else "COLUMN_SET_MISMATCH"
    )

    REAL_SCHEMA_AUDIT.append({
        "dataset_id": dataset_id,
        "train_columns": len(train_columns),
        "test_columns": len(test_columns),
        "same_column_set": same_columns,
        "same_column_order": same_order,
        "missing_from_test": str(missing_from_test),
        "extra_in_test": str(extra_in_test),
        "schema_status": schema_status,
    })

    if schema_status != "PASS":

        # Save evidence before stopping; do not reorder or drop columns.
        pd.DataFrame(REAL_DATA_AUDIT).to_csv(
            INPUT_AUDIT_DIR / "real_data_audit.csv",
            index=False,
        )

        pd.DataFrame(REAL_SCHEMA_AUDIT).to_csv(
            INPUT_AUDIT_DIR / "real_schema_audit.csv",
            index=False,
        )

        raise ValueError(
            f"{dataset_id}: real train/test schema mismatch. "
            f"Status={schema_status}; "
            f"missing_from_test={missing_from_test}; "
            f"extra_in_test={extra_in_test}. "
            "Resolve the upstream schema issue before continuing."
        )

    # Store validated data only after both splits pass.
    REAL_TRAIN[dataset_id] = train_df
    REAL_TEST[dataset_id] = test_df

    print(
        f"{dataset_id}: "
        f"train={train_df.shape}, "
        f"test={test_df.shape}, "
        f"target={target}, "
        f"schema={schema_status}"
    )

# --------------------------------------------------------------------------------
# 5. SAVE AUDIT ARTIFACTS
# --------------------------------------------------------------------------------

REAL_DATA_AUDIT_DF = pd.DataFrame(REAL_DATA_AUDIT)
REAL_SCHEMA_AUDIT_DF = pd.DataFrame(REAL_SCHEMA_AUDIT)

real_audit_path = INPUT_AUDIT_DIR / "real_data_audit.csv"
schema_audit_path = INPUT_AUDIT_DIR / "real_schema_audit.csv"

REAL_DATA_AUDIT_DF.to_csv(real_audit_path, index=False)
REAL_SCHEMA_AUDIT_DF.to_csv(schema_audit_path, index=False)

# --------------------------------------------------------------------------------
# 6. FINAL COMPLETENESS CHECKS
# --------------------------------------------------------------------------------

expected_audit_rows = len(DATASETS) * len(REQUIRED_SPLITS)

if len(REAL_DATA_AUDIT_DF) != expected_audit_rows:
    raise RuntimeError(
        f"Expected {expected_audit_rows} input audit rows, "
        f"found {len(REAL_DATA_AUDIT_DF)}."
    )

if not REAL_DATA_AUDIT_DF["status"].eq("PASS").all():
    raise RuntimeError("One or more real data inputs failed validation.")

if len(REAL_SCHEMA_AUDIT_DF) != len(DATASETS):
    raise RuntimeError(
        "Schema audit does not contain exactly one row per dataset."
    )

if not REAL_SCHEMA_AUDIT_DF["schema_status"].eq("PASS").all():
    raise RuntimeError("One or more real train/test schemas failed validation.")

if set(REAL_TRAIN) != set(DATASETS) or set(REAL_TEST) != set(DATASETS):
    raise RuntimeError("Loaded dataset registry does not match DATASETS.")

# --------------------------------------------------------------------------------
# 7. REPORT FINAL STATUS
# --------------------------------------------------------------------------------

print("-" * 100)
print("REAL DATA INPUT AUDIT")
print(REAL_DATA_AUDIT_DF[
    ["dataset_id", "split", "rows", "columns", "target_missing", "status"]
].to_string(index=False))

print("\nREAL TRAIN/TEST SCHEMA AUDIT")
print(REAL_SCHEMA_AUDIT_DF.to_string(index=False))

print("\n" + "=" * 100)
print("SECTION 3 — FINAL STATUS")
print("=" * 100)
print(f"Datasets loaded        : {len(REAL_TRAIN)}/{len(DATASETS)}")
print(f"Input audit records    : {len(REAL_DATA_AUDIT_DF)}")
print(f"Schema audit records   : {len(REAL_SCHEMA_AUDIT_DF)}")
print("Input validation       : PASS")
print("Real train/test schema : PASS")
print(f"Input audit artifact   : {real_audit_path}")
print(f"Schema audit artifact  : {schema_audit_path}")
print("Upstream data modified : False")
print("Training performed     : False")
print("=" * 100)


SECTION 3 — LOAD REAL DATA
adult_income: train=(34189, 16), test=(7327, 16), target=income, schema=PASS
bank_marketing: train=(31647, 18), test=(6782, 18), target=y, schema=PASS


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


diabetes_130us: train=(71236, 49), test=(15265, 49), target=readmitted, schema=PASS
----------------------------------------------------------------------------------------------------
REAL DATA INPUT AUDIT
    dataset_id split  rows  columns  target_missing status
  adult_income train 34189       16               0   PASS
  adult_income  test  7327       16               0   PASS
bank_marketing train 31647       18               0   PASS
bank_marketing  test  6782       18               0   PASS
diabetes_130us train 71236       49               0   PASS
diabetes_130us  test 15265       49               0   PASS

REAL TRAIN/TEST SCHEMA AUDIT
    dataset_id  train_columns  test_columns  same_column_set  same_column_order missing_from_test extra_in_test schema_status
  adult_income             16            16             True               True                []            []          PASS
bank_marketing             18            18             True               True                [] 

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [68]:
# ==================================================================================================
# SECTION 4 — LOAD SYNTHETIC DATA
# ==================================================================================================

from pathlib import Path
import hashlib
import gc
import pandas as pd

print("=" * 100)
print("SECTION 4 — LOAD SYNTHETIC DATA")
print("=" * 100)

# --------------------------------------------------------------------------------
# 1. CONFIGURE PATHS AND INITIALIZE OUTPUTS
# --------------------------------------------------------------------------------

SYNTHETIC_ROOT = NB13_ROOT / "synthetic_datasets"
INPUT_AUDIT_DIR = NB15_ROOT / "inputs"
INPUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

REAL_FEATURES = {}
SYNTHETIC_DATA = {}
SYNTHETIC_DATA_AUDIT = []
SYNTHETIC_SCHEMA_AUDIT = []

# This is an explicitly configured exclusion list, not an inferred one.
if "ID_COLUMNS" not in globals():
    raise NameError(
        "ID_COLUMNS is not defined. Verify the canonical identifier-column "
        "configuration before loading synthetic data."
    )

if not isinstance(ID_COLUMNS, (list, tuple, set)):
    raise TypeError("ID_COLUMNS must be a list, tuple, or set of column names.")

ID_COLUMNS = set(ID_COLUMNS)

# --------------------------------------------------------------------------------
# 2. LOAD AND VALIDATE EACH SYNTHETIC DATASET
# --------------------------------------------------------------------------------

for dataset_id, metadata in DATASETS.items():

    target = metadata["target"]
    synthetic_path = SYNTHETIC_ROOT / f"{dataset_id}_sppgan_synthetic.csv"

    if dataset_id not in REAL_TRAIN or dataset_id not in REAL_TEST:
        raise RuntimeError(
            f"{dataset_id}: validated real training/test data are unavailable. "
            "Run Section 3 successfully before Section 4."
        )

    if not synthetic_path.is_file():
        raise FileNotFoundError(
            f"Missing Notebook 13 synthetic data for {dataset_id}: "
            f"{synthetic_path}"
        )

    real_train = REAL_TRAIN[dataset_id]
    real_test = REAL_TEST[dataset_id]

    # --------------------------------------------------------------------------
    # 3. VALIDATE REAL INPUT SCHEMA
    # --------------------------------------------------------------------------

    if target not in real_train.columns or target not in real_test.columns:
        raise ValueError(
            f"{dataset_id}: target '{target}' is missing from a real split."
        )

    if list(real_train.columns) != list(real_test.columns):
        raise ValueError(
            f"{dataset_id}: real train/test column order differs. "
            "Resolve the upstream schema before continuing."
        )

    if real_train.columns.duplicated().any():
        raise ValueError(f"{dataset_id}: duplicate columns in real training data.")

    # Exclude only configured identifiers and the target from predictors.
    real_feature_columns = [
        col for col in real_train.columns
        if col not in ID_COLUMNS and col != target
    ]

    if not real_feature_columns:
        raise ValueError(f"{dataset_id}: no real predictor features remain.")

    # --------------------------------------------------------------------------
    # 4. READ AND VALIDATE SYNTHETIC DATA
    # --------------------------------------------------------------------------

    synthetic_df = pd.read_csv(synthetic_path, low_memory=False)

    if synthetic_df.empty:
        raise ValueError(f"{dataset_id}: synthetic dataset is empty.")

    if synthetic_df.columns.duplicated().any():
        duplicates = synthetic_df.columns[
            synthetic_df.columns.duplicated()
        ].tolist()

        raise ValueError(
            f"{dataset_id}: duplicate synthetic columns: {duplicates}"
        )

    invalid_names = [
        col for col in synthetic_df.columns
        if not isinstance(col, str) or not col.strip()
    ]

    if invalid_names:
        raise ValueError(
            f"{dataset_id}: invalid synthetic column names: {invalid_names}"
        )

    if target not in synthetic_df.columns:
        raise ValueError(
            f"{dataset_id}: synthetic target '{target}' is missing."
        )

    if synthetic_df[target].isna().all():
        raise ValueError(
            f"{dataset_id}: synthetic target '{target}' contains no valid labels."
        )

    # --------------------------------------------------------------------------
    # 5. COMPARE FEATURE SCHEMAS WITHOUT HIDING MISMATCHES
    # --------------------------------------------------------------------------

    synthetic_feature_columns = [
        col for col in synthetic_df.columns
        if col not in ID_COLUMNS and col != target
    ]

    missing_features = [
        col for col in real_feature_columns
        if col not in synthetic_feature_columns
    ]

    extra_features = [
        col for col in synthetic_feature_columns
        if col not in real_feature_columns
    ]

    same_feature_set = (
        set(real_feature_columns) == set(synthetic_feature_columns)
    )

    same_feature_order = (
        real_feature_columns == synthetic_feature_columns
    )

    # Audit the full schema too, including identifiers and target.
    missing_full_columns = [
        col for col in real_train.columns
        if col not in synthetic_df.columns
    ]

    extra_full_columns = [
        col for col in synthetic_df.columns
        if col not in real_train.columns
    ]

    row_id_mismatch = (
        "__original_row_id__" in real_train.columns
        and "__original_row_id__" not in synthetic_df.columns
    )

    schema_status = (
        "PASS"
        if not missing_features and not extra_features
        else "FEATURE_SCHEMA_MISMATCH"
    )

    SYNTHETIC_SCHEMA_AUDIT.append({
        "dataset_id": dataset_id,
        "real_columns": len(real_train.columns),
        "synthetic_columns": len(synthetic_df.columns),
        "real_feature_count": len(real_feature_columns),
        "synthetic_feature_count": len(synthetic_feature_columns),
        "same_feature_set": same_feature_set,
        "same_feature_order": same_feature_order,
        "missing_features": str(missing_features),
        "extra_features": str(extra_features),
        "missing_full_columns": str(missing_full_columns),
        "extra_full_columns": str(extra_full_columns),
        "original_row_id_mismatch": row_id_mismatch,
        "schema_status": schema_status,
    })

    # Never silently ignore a genuine predictor-schema mismatch.
    if schema_status != "PASS":
        pd.DataFrame(SYNTHETIC_SCHEMA_AUDIT).to_csv(
            INPUT_AUDIT_DIR / "synthetic_schema_audit.csv",
            index=False,
        )
        raise ValueError(
            f"{dataset_id}: synthetic feature schema does not match real data. "
            f"Missing features={missing_features}; "
            f"extra features={extra_features}. "
            "Resolve the upstream feature schema before proceeding."
        )

    # --------------------------------------------------------------------------
    # 6. REORDER MATCHING FEATURES AND TARGET EXPLICITLY
    # --------------------------------------------------------------------------

    # Reordering is permitted only after feature-set equality is verified.
    # Keep the original source dataframe unchanged.
    aligned_columns = real_feature_columns + [target]
    aligned_synthetic = synthetic_df.loc[:, aligned_columns].copy()

    if list(aligned_synthetic.columns) != aligned_columns:
        raise RuntimeError(
            f"{dataset_id}: synthetic column alignment verification failed."
        )

    REAL_FEATURES[dataset_id] = real_feature_columns
    SYNTHETIC_DATA[dataset_id] = aligned_synthetic

    # --------------------------------------------------------------------------
    # 7. RECORD INPUT AUDIT
    # --------------------------------------------------------------------------

    file_hash = hashlib.sha256(synthetic_path.read_bytes()).hexdigest()

    SYNTHETIC_DATA_AUDIT.append({
        "dataset_id": dataset_id,
        "synthetic_rows": len(synthetic_df),
        "synthetic_columns": len(synthetic_df.columns),
        "target": target,
        "target_missing": int(synthetic_df[target].isna().sum()),
        "feature_count": len(real_feature_columns),
        "missing_features": len(missing_features),
        "extra_features": len(extra_features),
        "feature_set_match": same_feature_set,
        "feature_order_match_before_alignment": same_feature_order,
        "original_row_id_mismatch": row_id_mismatch,
        "synthetic_path": str(synthetic_path),
        "sha256": file_hash,
        "status": "PASS",
    })

    print(
        f"{dataset_id}: synthetic={synthetic_df.shape}, "
        f"aligned_features={len(real_feature_columns)}, "
        f"feature_schema={schema_status}, "
        f"original_row_id_mismatch={row_id_mismatch}"
    )

    del synthetic_df, aligned_synthetic
    gc.collect()

# --------------------------------------------------------------------------------
# 8. SAVE AUDIT ARTIFACTS
# --------------------------------------------------------------------------------

SYNTHETIC_DATA_AUDIT_DF = pd.DataFrame(SYNTHETIC_DATA_AUDIT)
SYNTHETIC_SCHEMA_AUDIT_DF = pd.DataFrame(SYNTHETIC_SCHEMA_AUDIT)

synthetic_audit_path = INPUT_AUDIT_DIR / "synthetic_data_audit.csv"
synthetic_schema_path = INPUT_AUDIT_DIR / "synthetic_schema_audit.csv"

SYNTHETIC_DATA_AUDIT_DF.to_csv(synthetic_audit_path, index=False)
SYNTHETIC_SCHEMA_AUDIT_DF.to_csv(synthetic_schema_path, index=False)

# --------------------------------------------------------------------------------
# 9. FINAL VALIDATION
# --------------------------------------------------------------------------------

if len(SYNTHETIC_DATA) != len(DATASETS):
    raise RuntimeError(
        f"Expected {len(DATASETS)} synthetic datasets, "
        f"loaded {len(SYNTHETIC_DATA)}."
    )

if not SYNTHETIC_DATA_AUDIT_DF["status"].eq("PASS").all():
    raise RuntimeError("One or more synthetic datasets failed validation.")

for dataset_id, metadata in DATASETS.items():
    target = metadata["target"]
    expected_columns = REAL_FEATURES[dataset_id] + [target]

    if list(SYNTHETIC_DATA[dataset_id].columns) != expected_columns:
        raise RuntimeError(
            f"{dataset_id}: final aligned synthetic schema verification failed."
        )

# --------------------------------------------------------------------------------
# 10. REPORT STATUS
# --------------------------------------------------------------------------------

print("-" * 100)
print("SYNTHETIC DATA AUDIT")
print(
    SYNTHETIC_DATA_AUDIT_DF[
        [
            "dataset_id",
            "synthetic_rows",
            "synthetic_columns",
            "feature_count",
            "feature_set_match",
            "original_row_id_mismatch",
            "status",
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 100)
print("SECTION 4 — FINAL STATUS")
print("=" * 100)
print(f"Synthetic datasets loaded : {len(SYNTHETIC_DATA)}/{len(DATASETS)}")
print(f"Audit records             : {len(SYNTHETIC_DATA_AUDIT_DF)}")
print(f"Feature schema audit      : {synthetic_schema_path}")
print(f"Input audit               : {synthetic_audit_path}")
print("Synthetic source modified : False")
print("Training performed        : False")
print("=" * 100)

print(
    "\nNote: feature schema validation does not resolve whether "
    "'__original_row_id__' is an intended predictor. "
    "TSTR remains blocked until its role is verified from upstream artifacts."
)


SECTION 4 — LOAD SYNTHETIC DATA


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


adult_income: synthetic=(34189, 15), aligned_features=14, feature_schema=PASS, original_row_id_mismatch=True
bank_marketing: synthetic=(31647, 17), aligned_features=16, feature_schema=PASS, original_row_id_mismatch=True


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


diabetes_130us: synthetic=(71236, 48), aligned_features=47, feature_schema=PASS, original_row_id_mismatch=True
----------------------------------------------------------------------------------------------------
SYNTHETIC DATA AUDIT
    dataset_id  synthetic_rows  synthetic_columns  feature_count  feature_set_match  original_row_id_mismatch status
  adult_income           34189                 15             14               True                      True   PASS
bank_marketing           31647                 17             16               True                      True   PASS
diabetes_130us           71236                 48             47               True                      True   PASS

SECTION 4 — FINAL STATUS
Synthetic datasets loaded : 3/3
Audit records             : 3
Feature schema audit      : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/inputs/synthetic_schema_audit.csv
Input audit               : /content/drive/MyDrive/SPP_GAN_Research/results/not

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [69]:
# ==================================================================================================
# SECTION 5 — VALIDATE PREDICTION TASK
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 100)
print("SECTION 5 — VALIDATE PREDICTION TASK")
print("=" * 100)

# --------------------------------------------------------------------------------
# 1. INITIALIZE OUTPUTS
# --------------------------------------------------------------------------------

VALIDATION_DIR = NB15_ROOT / "validation"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

TASK_VALIDATION = []
MODEL_DATA = {}

# --------------------------------------------------------------------------------
# 2. VALIDATE CLASSIFICATION TASKS AND PREPARE MODEL DATA
# --------------------------------------------------------------------------------

for dataset_id, metadata in DATASETS.items():

    target = metadata["target"]

    # Confirm all required inputs are available.
    required_objects = [
        (REAL_TRAIN, "REAL_TRAIN"),
        (REAL_TEST, "REAL_TEST"),
        (SYNTHETIC_DATA, "SYNTHETIC_DATA"),
        (REAL_FEATURES, "REAL_FEATURES"),
    ]

    for obj, object_name in required_objects:
        if dataset_id not in obj:
            raise RuntimeError(
                f"{dataset_id}: missing {object_name} entry. "
                "Run the preceding sections successfully."
            )

    train_source = REAL_TRAIN[dataset_id]
    test_source = REAL_TEST[dataset_id]
    synthetic_source = SYNTHETIC_DATA[dataset_id]
    feature_columns = list(REAL_FEATURES[dataset_id])

    if not feature_columns:
        raise ValueError(f"{dataset_id}: no predictor features configured.")

    if len(feature_columns) != len(set(feature_columns)):
        raise ValueError(f"{dataset_id}: duplicate predictor feature names.")

    # Validate target availability.
    for split_name, df in [
        ("real_train", train_source),
        ("real_test", test_source),
        ("synthetic_train", synthetic_source),
    ]:
        if target not in df.columns:
            raise ValueError(
                f"{dataset_id}/{split_name}: target '{target}' is missing."
            )

        missing_features = [
            col for col in feature_columns if col not in df.columns
        ]

        if missing_features:
            raise ValueError(
                f"{dataset_id}/{split_name}: missing predictor columns: "
                f"{missing_features}"
            )

        if df.columns.duplicated().any():
            raise ValueError(
                f"{dataset_id}/{split_name}: duplicate column names."
            )

    # Verify the canonical feature schema and its order.
    expected_columns = feature_columns + [target]

    if list(synthetic_source.columns) != expected_columns:
        raise ValueError(
            f"{dataset_id}: synthetic schema does not match the expected "
            "aligned feature schema. Do not silently reorder or drop columns."
        )

    if not set(expected_columns).issubset(set(train_source.columns)):
        raise ValueError(
            f"{dataset_id}: real training data do not contain the expected "
            "feature and target columns."
        )

    if not set(expected_columns).issubset(set(test_source.columns)):
        raise ValueError(
            f"{dataset_id}: real test data do not contain the expected "
            "feature and target columns."
        )

    # Keep only rows with observed target labels.
    # Feature missingness is checked separately; it is not silently removed.
    train_df = train_source.loc[train_source[target].notna()].copy()
    test_df = test_source.loc[test_source[target].notna()].copy()
    synthetic_df = synthetic_source.loc[
        synthetic_source[target].notna()
    ].copy()

    if train_df.empty or test_df.empty or synthetic_df.empty:
        raise ValueError(
            f"{dataset_id}: one or more datasets have no rows after "
            "filtering missing target labels."
        )

    # Convert labels to strings consistently for class-set comparison.
    # Retain original feature values for downstream preprocessing.
    y_real_train = train_df[target].astype(str)
    y_real_test = test_df[target].astype(str)
    y_synthetic_train = synthetic_df[target].astype(str)

    real_classes = set(y_real_train.unique())
    test_classes = set(y_real_test.unique())
    synthetic_classes = set(y_synthetic_train.unique())

    if len(real_classes) < 2:
        raise ValueError(
            f"{dataset_id}: real training target has fewer than two classes."
        )

    if not test_classes.issubset(real_classes):
        raise ValueError(
            f"{dataset_id}: test labels absent from real training: "
            f"{sorted(test_classes - real_classes)}"
        )

    missing_synthetic_classes = sorted(real_classes - synthetic_classes)

    if missing_synthetic_classes:
        raise ValueError(
            f"{dataset_id}: synthetic training is missing real target classes: "
            f"{missing_synthetic_classes}. "
            "Resolve synthetic target coverage before utility evaluation."
        )

    # Verify the target contains valid, non-empty labels.
    for split_name, labels in [
        ("real_train", y_real_train),
        ("real_test", y_real_test),
        ("synthetic_train", y_synthetic_train),
    ]:
        if labels.str.strip().eq("").any():
            raise ValueError(
                f"{dataset_id}/{split_name}: target contains empty labels."
            )

    # Validate predictor missingness without dropping rows or imputing here.
    feature_missing = {}

    for split_name, df in [
        ("real_train", train_df),
        ("real_test", test_df),
        ("synthetic_train", synthetic_df),
    ]:
        missing_count = int(df[feature_columns].isna().sum().sum())
        feature_missing[split_name] = missing_count

    # Store model inputs without using the test set for model selection.
    MODEL_DATA[dataset_id] = {
        "target": target,
        "features": feature_columns,
        "real_train": train_df[feature_columns].copy(),
        "y_real_train": y_real_train.copy(),
        "real_test": test_df[feature_columns].copy(),
        "y_real_test": y_real_test.copy(),
        "synthetic_train": synthetic_df[feature_columns].copy(),
        "y_synthetic_train": y_synthetic_train.copy(),
        "real_train_rows": len(train_df),
        "real_test_rows": len(test_df),
        "synthetic_train_rows": len(synthetic_df),
        "real_classes": sorted(real_classes),
        "synthetic_classes": sorted(synthetic_classes),
        "test_classes": sorted(test_classes),
        "feature_missing_counts": feature_missing,
    }

    TASK_VALIDATION.append({
        "dataset_id": dataset_id,
        "task_type": "classification",
        "target": target,
        "n_features": len(feature_columns),
        "n_classes_real_train": len(real_classes),
        "n_classes_synthetic": len(synthetic_classes),
        "n_classes_test": len(test_classes),
        "real_train_rows": len(train_df),
        "real_test_rows": len(test_df),
        "synthetic_train_rows": len(synthetic_df),
        "real_train_missing_feature_values": feature_missing["real_train"],
        "real_test_missing_feature_values": feature_missing["real_test"],
        "synthetic_missing_feature_values": feature_missing["synthetic_train"],
        "missing_synthetic_classes": str(missing_synthetic_classes),
        "test_classes_subset_of_train": test_classes.issubset(real_classes),
        "feature_schema_match": True,
        "status": "PASS",
    })

    print(
        f"{dataset_id}: "
        f"classes={len(real_classes)}, "
        f"features={len(feature_columns)}, "
        f"real_train={len(train_df)}, "
        f"real_test={len(test_df)}, "
        f"synthetic_train={len(synthetic_df)}, "
        f"feature_missing="
        f"{feature_missing['real_train']}/"
        f"{feature_missing['real_test']}/"
        f"{feature_missing['synthetic_train']}"
    )

# --------------------------------------------------------------------------------
# 3. SAVE VALIDATION ARTIFACT
# --------------------------------------------------------------------------------

TASK_VALIDATION_DF = pd.DataFrame(TASK_VALIDATION)

task_validation_path = VALIDATION_DIR / "task_validation.csv"
TASK_VALIDATION_DF.to_csv(task_validation_path, index=False)

# --------------------------------------------------------------------------------
# 4. FINAL COMPLETENESS CHECKS
# --------------------------------------------------------------------------------

if len(TASK_VALIDATION_DF) != len(DATASETS):
    raise RuntimeError(
        f"Expected {len(DATASETS)} task validation records, "
        f"found {len(TASK_VALIDATION_DF)}."
    )

if set(MODEL_DATA) != set(DATASETS):
    raise RuntimeError(
        "MODEL_DATA registry does not match the configured dataset registry."
    )

if not TASK_VALIDATION_DF["status"].eq("PASS").all():
    raise RuntimeError("One or more prediction tasks failed validation.")

print("-" * 100)
print("TASK VALIDATION SUMMARY")
print(
    TASK_VALIDATION_DF[
        [
            "dataset_id",
            "n_features",
            "n_classes_real_train",
            "n_classes_synthetic",
            "n_classes_test",
            "real_train_rows",
            "real_test_rows",
            "synthetic_train_rows",
            "status",
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 100)
print("SECTION 5 — FINAL STATUS")
print("=" * 100)
print(f"Tasks validated       : {len(TASK_VALIDATION_DF)}/{len(DATASETS)}")
print(f"Validation artifact   : {task_validation_path}")
print("Task validation       : PASS")
print("Test-set model tuning : Not performed")
print("Training performed    : False")
print("=" * 100)

print(
    "\nImportant: task validation does not resolve the upstream "
    "'__original_row_id__' schema question. Do not begin TSTR training "
    "until the intended feature schema is verified."
)


SECTION 5 — VALIDATE PREDICTION TASK


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


adult_income: classes=2, features=14, real_train=34189, real_test=7327, synthetic_train=34189, feature_missing=4599/891/0
bank_marketing: classes=2, features=16, real_train=31647, real_test=6782, synthetic_train=31647, feature_missing=0/0/0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


diabetes_130us: classes=3, features=47, real_train=71236, real_test=15265, synthetic_train=71236, feature_missing=261953/56123/0
----------------------------------------------------------------------------------------------------
TASK VALIDATION SUMMARY
    dataset_id  n_features  n_classes_real_train  n_classes_synthetic  n_classes_test  real_train_rows  real_test_rows  synthetic_train_rows status
  adult_income          14                     2                    2               2            34189            7327                 34189   PASS
bank_marketing          16                     2                    2               2            31647            6782                 31647   PASS
diabetes_130us          47                     3                    3               3            71236           15265                 71236   PASS

SECTION 5 — FINAL STATUS
Tasks validated       : 3/3
Validation artifact   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/valida

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [70]:
# ==================================================================================================
# SECTION 6 — DEFINE TSTR PROTOCOL
# ==================================================================================================

import json
from pathlib import Path

print("=" * 100)
print("SECTION 6 — DEFINE TSTR PROTOCOL")
print("=" * 100)

# --------------------------------------------------------------------------------
# 1. DEFINE THE EVALUATION PROTOCOL
# --------------------------------------------------------------------------------

TSTR_PROTOCOL = {
    "protocol_id": "TSTR",
    "protocol_version": "1.0",

    "objective": (
        "Evaluate the predictive utility of synthetic training data "
        "on a fixed held-out real test set."
    ),

    "training_source": (
        "Notebook 13 synthetic dataset generated for the corresponding dataset."
    ),

    "evaluation_source": (
        "Notebook 02 canonical held-out real test split."
    ),

    "target": "Dataset-specific target registered in DATASETS.",

    "predictor_features": (
        "Use the verified canonical predictor schema shared by real and "
        "synthetic training data. Do not silently drop or infer features."
    ),

    "preprocessing": {
        "rule": (
            "Fit all learned preprocessing steps on the corresponding "
            "training data only."
        ),
        "tstr": (
            "Fit preprocessing on synthetic training data, then transform "
            "synthetic training data and real test data."
        ),
        "trtr": (
            "Fit preprocessing on real training data, then transform "
            "real training data and real test data."
        ),
        "test_leakage": (
            "Never fit imputers, encoders, scalers, feature selectors, "
            "or other learned transformations on the real test set."
        ),
        "missing_values": (
            "Handle missing predictor values through a documented, "
            "training-fitted preprocessing pipeline. Do not silently "
            "drop rows or impute using the test set."
        ),
    },

    "test_set": (
        "Use the same fixed real test split for TSTR and TRTR comparisons."
    ),

    "test_usage": (
        "Final evaluation only; do not use test results for model selection, "
        "threshold tuning, or hyperparameter tuning."
    ),

    "synthetic_rows": (
        "Use all synthetic training rows with non-missing target labels, "
        "subject to documented input and schema validation."
    ),

    "target_validation": {
        "task_type": "classification",
        "require_at_least_two_real_training_classes": True,
        "require_synthetic_coverage_of_real_training_classes": True,
        "require_test_classes_to_be_subset_of_real_training_classes": True,
        "label_normalization": (
            "Apply a consistent, documented target-label representation "
            "across real training, synthetic training, and real test data."
        ),
    },

    "comparison_protocol": {
        "tstr": "Train on synthetic training data; evaluate on real test data.",
        "trtr": "Train on real training data; evaluate on the same real test data.",
        "classifier_configuration": (
            "Use the same predeclared classifier families and comparable "
            "hyperparameter-selection procedures for TSTR and TRTR."
        ),
        "random_seeds": list(REPETITION_SEEDS),
        "master_seed": MASTER_SEED,
    },

    "primary_metrics": list(PRIMARY_METRICS),
    "secondary_metrics": list(SECONDARY_METRICS),

    "readiness_requirements": [
        "All required real and synthetic input files pass validation.",
        "The intended predictor schema is verified against upstream artifacts.",
        "The __original_row_id__ role is resolved using upstream evidence.",
        "Training-fitted missing-value preprocessing is specified.",
        "Target labels and class coverage pass validation.",
        "Classifier configuration and metric definitions are frozen.",
    ],

    "execution_status": "PROTOCOL_DEFINED_TRAINING_BLOCKED",
}

# --------------------------------------------------------------------------------
# 2. VALIDATE REQUIRED CONFIGURATION REFERENCES
# --------------------------------------------------------------------------------

required_globals = [
    "NB15_ROOT",
    "DATASETS",
    "MASTER_SEED",
    "REPETITION_SEEDS",
    "PRIMARY_METRICS",
    "SECONDARY_METRICS",
]

missing_globals = [
    name for name in required_globals
    if name not in globals()
]

if missing_globals:
    raise NameError(
        f"Required configuration variables are missing: {missing_globals}"
    )

if not isinstance(DATASETS, dict) or not DATASETS:
    raise ValueError("DATASETS must be a non-empty dictionary.")

if not REPETITION_SEEDS:
    raise ValueError("REPETITION_SEEDS must not be empty.")

if not PRIMARY_METRICS:
    raise ValueError("PRIMARY_METRICS must not be empty.")

# --------------------------------------------------------------------------------
# 3. SAVE THE PROTOCOL
# --------------------------------------------------------------------------------

CONFIG_DIR = NB15_ROOT / "config"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

PROTOCOL_PATH = CONFIG_DIR / "tstr_protocol.json"
TEMP_PROTOCOL_PATH = CONFIG_DIR / "tstr_protocol.tmp"

with open(TEMP_PROTOCOL_PATH, "w", encoding="utf-8") as f:
    json.dump(TSTR_PROTOCOL, f, indent=2, ensure_ascii=False)
    f.write("\n")

TEMP_PROTOCOL_PATH.replace(PROTOCOL_PATH)

# --------------------------------------------------------------------------------
# 4. VERIFY THE SAVED ARTIFACT
# --------------------------------------------------------------------------------

with open(PROTOCOL_PATH, "r", encoding="utf-8") as f:
    SAVED_TSTR_PROTOCOL = json.load(f)

if SAVED_TSTR_PROTOCOL != TSTR_PROTOCOL:
    raise RuntimeError(
        "Saved TSTR protocol does not match the in-memory configuration."
    )

print(json.dumps(SAVED_TSTR_PROTOCOL, indent=2, ensure_ascii=False))

print("-" * 100)
print("SECTION 6 — FINAL STATUS")
print("-" * 100)
print(f"Protocol artifact : {PROTOCOL_PATH}")
print("Protocol saved    : True")
print("Protocol verified : True")
print("Training performed: False")
print("Execution status  : PROTOCOL_DEFINED_TRAINING_BLOCKED")
print("=" * 100)

print(
    "\nTSTR execution remains blocked until the upstream feature-schema "
    "question and missing-value preprocessing strategy are resolved."
)


SECTION 6 — DEFINE TSTR PROTOCOL
{
  "protocol_id": "TSTR",
  "protocol_version": "1.0",
  "objective": "Evaluate the predictive utility of synthetic training data on a fixed held-out real test set.",
  "training_source": "Notebook 13 synthetic dataset generated for the corresponding dataset.",
  "evaluation_source": "Notebook 02 canonical held-out real test split.",
  "target": "Dataset-specific target registered in DATASETS.",
  "predictor_features": "Use the verified canonical predictor schema shared by real and synthetic training data. Do not silently drop or infer features.",
  "preprocessing": {
    "rule": "Fit all learned preprocessing steps on the corresponding training data only.",
    "tstr": "Fit preprocessing on synthetic training data, then transform synthetic training data and real test data.",
    "trtr": "Fit preprocessing on real training data, then transform real training data and real test data.",
    "test_leakage": "Never fit imputers, encoders, scalers, feature s

In [71]:
# ==================================================================================================
# SECTION 7 — DEFINE REFERENCE TRTR PROTOCOL
# ==================================================================================================

import json
from pathlib import Path

print("=" * 100)
print("SECTION 7 — DEFINE REFERENCE TRTR PROTOCOL")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Validate Required Configuration
# --------------------------------------------------------------------------------------------------

required_globals = [
    "NB15_ROOT",
    "DATASETS",
    "MASTER_SEED",
    "REPETITION_SEEDS",
    "PRIMARY_METRICS",
    "SECONDARY_METRICS",
]

missing_globals = [
    name for name in required_globals
    if name not in globals()
]

if missing_globals:
    raise RuntimeError(
        f"Required configuration is missing: {missing_globals}. "
        "Run the preceding Notebook 15 configuration sections first."
    )

NB15_ROOT = Path(NB15_ROOT)
CONFIG_DIR = NB15_ROOT / "config"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------------------------------------
# 2. Define Reference TRTR Protocol
# --------------------------------------------------------------------------------------------------

TRTR_PROTOCOL = {
    "protocol_id": "TRTR",
    "protocol_version": "1.0",
    "objective": (
        "Evaluate reference predictive utility using real training data "
        "and the fixed held-out real test split."
    ),
    "training_source": (
        "Notebook 02 canonical real training split for the corresponding dataset."
    ),
    "evaluation_source": (
        "Notebook 02 canonical held-out real test split, identical to the split "
        "used for TSTR."
    ),
    "target": "Dataset-specific target registered in DATASETS.",
    "predictor_features": (
        "Use the verified canonical predictor schema. Exclude identifiers only "
        "when their non-predictor role is confirmed by upstream evidence."
    ),
    "preprocessing": {
        "fit_scope": "Real training data only.",
        "transform_scope": [
            "Real training data",
            "Fixed real test data",
        ],
        "missing_values": (
            "Use a documented preprocessing pipeline fitted on real training data "
            "only. Never fit imputers or other learned transformations on test data."
        ),
        "test_leakage": (
            "Do not use test data for fitting preprocessing, feature selection, "
            "hyperparameter tuning, threshold selection, or model selection."
        ),
    },
    "test_set": (
        "Use the same fixed Notebook 02 real test split for TSTR and TRTR."
    ),
    "test_usage": "Final evaluation only.",
    "comparison_protocol": {
        "tstr_comparator": (
            "TSTR trains on validated synthetic training data and evaluates "
            "on the same fixed real test split."
        ),
        "classifier_configuration": (
            "Use the same predeclared classifier families, metric definitions, "
            "and comparable hyperparameter-selection procedures as TSTR."
        ),
        "random_seeds": list(REPETITION_SEEDS),
        "master_seed": MASTER_SEED,
    },
    "primary_metrics": list(PRIMARY_METRICS),
    "secondary_metrics": list(SECONDARY_METRICS),
    "readiness_requirements": [
        "Notebook 02 real training and test files pass validation.",
        "The intended predictor schema is verified against upstream artifacts.",
        "The __original_row_id__ role is resolved using upstream evidence.",
        "Training-fitted missing-value preprocessing is specified.",
        "Target labels and class coverage pass validation.",
        "Classifier configuration and metric definitions are frozen.",
        "The held-out test set is reserved for final evaluation.",
    ],
    "execution_status": "PROTOCOL_DEFINED_TRAINING_BLOCKED",
}

# --------------------------------------------------------------------------------------------------
# 3. Save Protocol Atomically
# --------------------------------------------------------------------------------------------------

protocol_path = CONFIG_DIR / "trtr_protocol.json"
temporary_path = CONFIG_DIR / "trtr_protocol.tmp"

with open(temporary_path, "w", encoding="utf-8") as f:
    json.dump(TRTR_PROTOCOL, f, indent=2, ensure_ascii=False)
    f.write("\n")

temporary_path.replace(protocol_path)

# --------------------------------------------------------------------------------------------------
# 4. Reload and Verify Saved Artifact
# --------------------------------------------------------------------------------------------------

with open(protocol_path, "r", encoding="utf-8") as f:
    SAVED_TRTR_PROTOCOL = json.load(f)

protocol_verified = SAVED_TRTR_PROTOCOL == TRTR_PROTOCOL

if not protocol_verified:
    raise RuntimeError(
        "TRTR protocol verification failed. "
        "The saved JSON does not match the in-memory protocol."
    )

# --------------------------------------------------------------------------------------------------
# 5. Report Status
# --------------------------------------------------------------------------------------------------

print("-" * 100)
print("TRTR PROTOCOL")
print("-" * 100)
print(json.dumps(SAVED_TRTR_PROTOCOL, indent=2, ensure_ascii=False))

print("-" * 100)
print("SECTION 7 — FINAL STATUS")
print("-" * 100)
print(f"Protocol artifact : {protocol_path}")
print(f"Protocol saved    : {protocol_path.exists()}")
print(f"Protocol verified : {protocol_verified}")
print("Training performed: False")
print(f"Execution status  : {TRTR_PROTOCOL['execution_status']}")
print("=" * 100)


SECTION 7 — DEFINE REFERENCE TRTR PROTOCOL
----------------------------------------------------------------------------------------------------
TRTR PROTOCOL
----------------------------------------------------------------------------------------------------
{
  "protocol_id": "TRTR",
  "protocol_version": "1.0",
  "objective": "Evaluate reference predictive utility using real training data and the fixed held-out real test split.",
  "training_source": "Notebook 02 canonical real training split for the corresponding dataset.",
  "evaluation_source": "Notebook 02 canonical held-out real test split, identical to the split used for TSTR.",
  "target": "Dataset-specific target registered in DATASETS.",
  "predictor_features": "Use the verified canonical predictor schema. Exclude identifiers only when their non-predictor role is confirmed by upstream evidence.",
  "preprocessing": {
    "fit_scope": "Real training data only.",
    "transform_scope": [
      "Real training data",
      "Fixe

In [72]:
# ==================================================================================================
# SECTION 8 — CONFIGURE PREDICTIVE MODELS
# SPP-GAN Research | Notebook 15 — ML Utility / TSTR Evaluation
# ==================================================================================================

import gc
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# --------------------------------------------------------------------------------------------------
# 1. Validate Configuration
# --------------------------------------------------------------------------------------------------

print("=" * 100)
print("SECTION 8 — CONFIGURE PREDICTIVE MODELS")
print("=" * 100)

required_globals = [
    "NB15_ROOT",
    "MASTER_SEED",
    "REPETITION_SEEDS",
    "DATASETS",
]

missing_globals = [
    name for name in required_globals
    if name not in globals()
]

if missing_globals:
    raise RuntimeError(
        f"Missing required configuration variables: {missing_globals}. "
        "Run the preceding Notebook 15 configuration sections first."
    )

NB15_ROOT = Path(NB15_ROOT)
CONFIG_DIR = NB15_ROOT / "config"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

if not isinstance(MASTER_SEED, (int, np.integer)):
    raise TypeError("MASTER_SEED must be an integer.")

if not REPETITION_SEEDS:
    raise ValueError("REPETITION_SEEDS must contain at least one seed.")

if not all(isinstance(seed, (int, np.integer)) for seed in REPETITION_SEEDS):
    raise TypeError("All repetition seeds must be integers.")

# --------------------------------------------------------------------------------------------------
# 2. Detect Optional XGBoost Availability
# --------------------------------------------------------------------------------------------------

try:
    import xgboost as xgb
    from xgboost import XGBClassifier

    XGBOOST_AVAILABLE = True
    XGBOOST_VERSION = getattr(xgb, "__version__", "unknown")

except ImportError:
    XGBClassifier = None
    XGBOOST_AVAILABLE = False
    XGBOOST_VERSION = None

# --------------------------------------------------------------------------------------------------
# 3. Define the Authoritative Model Registry
# --------------------------------------------------------------------------------------------------
# Registry parameters are the source of truth for build_classifier().
# Keep model configurations fixed before the final evaluation.
# No model is fitted in this section.

MODEL_REGISTRY = {
    "logistic_regression": {
        "class": "LogisticRegression",
        "enabled": True,
        "parameters": {
            "max_iter": 500,
            "class_weight": "balanced",
            "solver": "lbfgs",
        },
        "runtime_parameters": {
            "random_state_from_repetition_seed": True,
        },
    },
    "random_forest": {
        "class": "RandomForestClassifier",
        "enabled": True,
        "parameters": {
            "n_estimators": 150,
            "max_depth": 18,
            "min_samples_leaf": 2,
            "class_weight": "balanced",
            "n_jobs": 2,
        },
        "runtime_parameters": {
            "random_state_from_repetition_seed": True,
        },
    },
    "xgboost": {
        "class": "XGBClassifier",
        "enabled": bool(XGBOOST_AVAILABLE),
        "parameters": {
            "n_estimators": 150,
            "max_depth": 6,
            "learning_rate": 0.05,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "tree_method": "hist",
            "n_jobs": 2,
        },
        "runtime_parameters": {
            "random_state_from_repetition_seed": True,
            "objective_binary": "binary:logistic",
            "objective_multiclass": "multi:softprob",
            "eval_metric_binary": "logloss",
            "eval_metric_multiclass": "mlogloss",
        },
    },
}

# Validate required registry fields and model parameters.
for model_id, info in MODEL_REGISTRY.items():
    if not {"class", "enabled", "parameters"} <= set(info):
        raise ValueError(f"Incomplete registry entry: {model_id}")

    if not isinstance(info["enabled"], bool):
        raise TypeError(f"Model enabled flag must be boolean: {model_id}")

    if not isinstance(info["parameters"], dict):
        raise TypeError(f"Model parameters must be a dictionary: {model_id}")

if MODEL_REGISTRY["xgboost"]["enabled"] and not XGBOOST_AVAILABLE:
    raise RuntimeError("XGBoost is enabled in the registry but is unavailable.")

ACTIVE_MODELS = [
    model_id
    for model_id, info in MODEL_REGISTRY.items()
    if info["enabled"]
]

if not ACTIVE_MODELS:
    raise RuntimeError("No predictive models are available.")

# --------------------------------------------------------------------------------------------------
# 4. Persist Model Registry and Runtime Metadata
# --------------------------------------------------------------------------------------------------

MODEL_REGISTRY_PATH = CONFIG_DIR / "predictive_model_registry.json"
MODEL_REGISTRY_TEMP_PATH = CONFIG_DIR / "predictive_model_registry.tmp"

MODEL_CONFIGURATION = {
    "configuration_version": "1.0",
    "project": "SPP-GAN",
    "notebook": "15",
    "section": 8,
    "master_seed": int(MASTER_SEED),
    "repetition_seeds": [int(seed) for seed in REPETITION_SEEDS],
    "software": {
        "scikit_learn": sklearn.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "xgboost_available": bool(XGBOOST_AVAILABLE),
        "xgboost_version": XGBOOST_VERSION,
    },
    "resource_limits": {
        "random_forest_n_jobs": 2,
        "xgboost_n_jobs": 2,
        "parallel_model_training": False,
    },
    "preprocessing_policy": {
        "fit_preprocessor_separately_for_each_evaluation": True,
        "tstr_fit_data": "Synthetic training data only",
        "trtr_fit_data": "Real training data only",
        "test_transform_only": True,
        "numeric_imputation": "median",
        "categorical_imputation": "most_frequent",
        "categorical_encoding": "one-hot",
        "numeric_scaling": "standard_scaler",
        "unknown_categories": "ignore",
        "feature_schema_must_be_verified_before_training": True,
        "original_row_id_must_be_resolved_before_training": True,
    },
    "models": MODEL_REGISTRY,
    "active_models": ACTIVE_MODELS,
    "training_performed": False,
    "execution_status": "CONFIGURED_TRAINING_BLOCKED",
}

with open(MODEL_REGISTRY_TEMP_PATH, "w", encoding="utf-8") as f:
    json.dump(MODEL_CONFIGURATION, f, indent=2, ensure_ascii=False)
    f.write("\n")

MODEL_REGISTRY_TEMP_PATH.replace(MODEL_REGISTRY_PATH)

with open(MODEL_REGISTRY_PATH, "r", encoding="utf-8") as f:
    SAVED_MODEL_CONFIGURATION = json.load(f)

if SAVED_MODEL_CONFIGURATION != MODEL_CONFIGURATION:
    raise RuntimeError("Saved model registry verification failed.")

# --------------------------------------------------------------------------------------------------
# 5. Build a Training-Fitted Preprocessor
# --------------------------------------------------------------------------------------------------
# This function only constructs a transformer.
# Call fit_transform() exclusively on the corresponding training data.
# For TSTR, training data must be synthetic.
# For TRTR, training data must be real.
# Transform the held-out real test set without fitting on it.
#
# IMPORTANT:
# - Do not pass the target column as a predictor.
# - Do not silently discard columns to force schema compatibility.
# - Resolve identifier roles and align validated feature schemas upstream.
# - This function does not resolve the upstream __original_row_id__ question.

def build_preprocessor(X_train):
    if not isinstance(X_train, pd.DataFrame):
        raise TypeError("X_train must be a pandas DataFrame.")

    if X_train.empty:
        raise ValueError("X_train contains no rows.")

    if X_train.shape[1] == 0:
        raise ValueError("X_train contains no predictor columns.")

    if not X_train.columns.is_unique:
        raise ValueError("Duplicate predictor column names are not allowed.")

    if X_train.columns.isna().any():
        raise ValueError("Predictor column names cannot be missing.")

    # Detect columns containing only missing values.
    all_missing = X_train.columns[X_train.isna().all()].tolist()

    if all_missing:
        raise ValueError(
            "All-missing predictor columns require an explicit, documented "
            f"upstream policy before model configuration: {all_missing[:20]}"
        )

    # Treat Boolean and numeric predictors as numeric.
    numeric_columns = X_train.select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()

    categorical_columns = [
        column
        for column in X_train.columns
        if column not in numeric_columns
    ]

    if not numeric_columns and not categorical_columns:
        raise ValueError("No supported numeric or categorical predictors found.")

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    # Support both recent and older scikit-learn versions.
    try:
        one_hot_encoder = OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=2,
            sparse_output=True,
            dtype=np.float32,
        )
    except TypeError:
        one_hot_encoder = OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=2,
            sparse=True,
            dtype=np.float32,
        )

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", one_hot_encoder),
    ])

    transformer = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_columns),
            ("categorical", categorical_pipeline, categorical_columns),
        ],
        remainder="drop",
        sparse_threshold=1.0,
    )

    return transformer

# --------------------------------------------------------------------------------------------------
# 6. Build Classifiers From the Registry
# --------------------------------------------------------------------------------------------------
# The registry is authoritative for shared hyperparameters.
# Random state and XGBoost objective are set at construction time.
# No model is fitted here.

def build_classifier(model_id, seed, n_classes):
    if model_id not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model ID: {model_id}")

    model_config = MODEL_REGISTRY[model_id]

    if not model_config["enabled"]:
        raise ValueError(f"Model is disabled or unavailable: {model_id}")

    if not isinstance(seed, (int, np.integer)):
        raise TypeError("seed must be an integer.")

    if not isinstance(n_classes, (int, np.integer)) or n_classes < 2:
        raise ValueError("n_classes must be an integer of at least 2.")

    params = dict(model_config["parameters"])
    seed = int(seed)
    n_classes = int(n_classes)

    if model_id == "logistic_regression":
        params["random_state"] = seed
        return LogisticRegression(**params)

    if model_id == "random_forest":
        params["random_state"] = seed
        return RandomForestClassifier(**params)

    if model_id == "xgboost":
        if not XGBOOST_AVAILABLE:
            raise RuntimeError("XGBoost is not installed.")

        runtime_params = model_config["runtime_parameters"]
        params["random_state"] = seed
        params["verbosity"] = 0

        if n_classes == 2:
            params["objective"] = runtime_params["objective_binary"]
            params["eval_metric"] = runtime_params["eval_metric_binary"]
        else:
            params["objective"] = runtime_params["objective_multiclass"]
            params["num_class"] = n_classes
            params["eval_metric"] = runtime_params["eval_metric_multiclass"]

        return XGBClassifier(**params)

    raise ValueError(f"Unsupported model ID: {model_id}")

# --------------------------------------------------------------------------------------------------
# 7. Validate Model Construction Without Fitting
# --------------------------------------------------------------------------------------------------

MODEL_BUILD_VALIDATION = []

for model_id in ACTIVE_MODELS:
    for n_classes in (2, 3):
        try:
            model = build_classifier(
                model_id=model_id,
                seed=int(REPETITION_SEEDS[0]),
                n_classes=n_classes,
            )

            MODEL_BUILD_VALIDATION.append({
                "model_id": model_id,
                "n_classes_tested": n_classes,
                "status": "PASS",
                "class_name": model.__class__.__name__,
            })

            del model

        except Exception as exc:
            MODEL_BUILD_VALIDATION.append({
                "model_id": model_id,
                "n_classes_tested": n_classes,
                "status": "FAIL",
                "error": str(exc),
            })

MODEL_BUILD_VALIDATION_DF = pd.DataFrame(MODEL_BUILD_VALIDATION)

validation_path = CONFIG_DIR / "predictive_model_build_validation.csv"
MODEL_BUILD_VALIDATION_DF.to_csv(validation_path, index=False)

if not MODEL_BUILD_VALIDATION_DF["status"].eq("PASS").all():
    raise RuntimeError(
        "One or more model-construction checks failed. "
        f"Review: {validation_path}"
    )

# --------------------------------------------------------------------------------------------------
# 8. Report Configuration Status
# --------------------------------------------------------------------------------------------------

print("-" * 100)
print("MODEL REGISTRY")
print("-" * 100)
print("Active models:", ACTIVE_MODELS)
print("XGBoost available:", XGBOOST_AVAILABLE)
print("XGBoost version:", XGBOOST_VERSION)
print("scikit-learn version:", sklearn.__version__)
print("Master seed:", MASTER_SEED)
print("Repetition seeds:", list(REPETITION_SEEDS))
print("Model construction checks:", len(MODEL_BUILD_VALIDATION_DF))
print("All model construction checks passed:", True)

print("-" * 100)
print("SAVED ARTIFACTS")
print("-" * 100)
print("Registry:", MODEL_REGISTRY_PATH)
print("Registry verified:", SAVED_MODEL_CONFIGURATION == MODEL_CONFIGURATION)
print("Build validation:", validation_path)

print("-" * 100)
print("SECTION 8 — FINAL STATUS")
print("-" * 100)
print("Model configuration saved: True")
print("Model construction validated: True")
print("Model fitting performed: False")
print("Training performed: False")
print("Execution status: CONFIGURED_TRAINING_BLOCKED")
print("=" * 100)

gc.collect()


SECTION 8 — CONFIGURE PREDICTIVE MODELS
----------------------------------------------------------------------------------------------------
MODEL REGISTRY
----------------------------------------------------------------------------------------------------
Active models: ['logistic_regression', 'random_forest', 'xgboost']
XGBoost available: True
XGBoost version: 3.4.1
scikit-learn version: 1.6.1
Master seed: 2025
Repetition seeds: [3026, 3027, 3028, 3029, 3030]
Model construction checks: 6
All model construction checks passed: True
----------------------------------------------------------------------------------------------------
SAVED ARTIFACTS
----------------------------------------------------------------------------------------------------
Registry: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/config/predictive_model_registry.json
Registry verified: True
Build validation: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/config/predict

95

In [74]:

# ==================================================================================================
# NOTEBOOK 15 — SECTION 9: TRAIN ON SYNTHETIC DATA (TSTR)
# SPP-GAN Research | RAM-conscious | Google Colab compatible
#
# Protocol:
#   Training data : SPP-GAN synthetic training dataset only
#   Evaluation    : Reserved real test dataset only
#   Preprocessing : Fitted on synthetic training predictors only
#   Models        : Logistic Regression, Random Forest
#   Outputs       : Models, predictions, metrics, validation report and manifest
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import time
import warnings

import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)
import joblib

warnings.filterwarnings("default")

# --------------------------------------------------------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Research")

DATASET_TARGETS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

REAL_DATA_ROOT = (
    PROJECT_ROOT / "data" / "processed" / "notebook_02" / "native"
)

SYNTHETIC_DATA_ROOT = (
    PROJECT_ROOT / "results" / "notebooks" / "notebook_13" / "synthetic_datasets"
)

NOTEBOOK_OUTPUT_ROOT = (
    PROJECT_ROOT / "results" / "notebooks" / "notebook_15"
)

TSTR_OUTPUT_DIR = NOTEBOOK_OUTPUT_ROOT / "tstr"
MODEL_OUTPUT_DIR = NOTEBOOK_OUTPUT_ROOT / "models"
VALIDATION_OUTPUT_DIR = NOTEBOOK_OUTPUT_ROOT / "validation"

for directory in (
    TSTR_OUTPUT_DIR,
    MODEL_OUTPUT_DIR,
    VALIDATION_OUTPUT_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

MASTER_SEED = 2025

# Identifier and metadata columns must never be used as predictive features.
IDENTIFIER_CANDIDATES = {
    "__original_row_id__",
    "original_row_id",
    "row_id",
    "record_id",
    "recordid",
    "id",
}

# Frozen classifier configuration for this TSTR experiment.
FROZEN_MODEL_CONFIG = {
    "logistic_regression": {
        "class": "LogisticRegression",
        "parameters": {
            "max_iter": 500,
            "class_weight": "balanced",
            "solver": "lbfgs",
        },
    },
    "random_forest": {
        "class": "RandomForestClassifier",
        "parameters": {
            "n_estimators": 150,
            "max_depth": 18,
            "min_samples_leaf": 2,
            "class_weight": "balanced",
            "n_jobs": 2,
        },
    },
}

# Keep resource use bounded.
MAX_FEATURE_COUNT = 5000
MAX_ROWS_WARNING_THRESHOLD = 250000

print("=" * 100)
print("NOTEBOOK 15 — SECTION 9: TRAIN ON SYNTHETIC DATA (TSTR)")
print("=" * 100)
print(f"Project root : {PROJECT_ROOT}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"Master seed  : {MASTER_SEED}")
print(f"Datasets     : {len(DATASET_TARGETS)}")
print(f"Models       : {len(FROZEN_MODEL_CONFIG)}")

# --------------------------------------------------------------------------------------------------
# 2. VERIFY PROJECT AND INPUT FILES
# --------------------------------------------------------------------------------------------------

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root does not exist: {PROJECT_ROOT}\n"
        "Mount Google Drive and verify the canonical project path."
    )

input_paths = {}

for dataset_id, target_col in DATASET_TARGETS.items():
    real_train_path = REAL_DATA_ROOT / dataset_id / "train.csv"
    real_test_path = REAL_DATA_ROOT / dataset_id / "test.csv"
    synthetic_path = (
        SYNTHETIC_DATA_ROOT / f"{dataset_id}_sppgan_synthetic.csv"
    )

    input_paths[dataset_id] = {
        "real_train": real_train_path,
        "real_test": real_test_path,
        "synthetic_train": synthetic_path,
        "target": target_col,
    }

    for data_role, path in input_paths[dataset_id].items():
        if data_role == "target":
            continue
        if not path.is_file():
            raise FileNotFoundError(
                f"Required {data_role} file is missing for {dataset_id}:\n"
                f"{path}"
            )

    print(f"\n✓ Input files found: {dataset_id}")

# --------------------------------------------------------------------------------------------------
# 3. HELPER FUNCTIONS
# --------------------------------------------------------------------------------------------------

def canonical_column_name(column):
    """Normalize column names for identifier detection."""
    return str(column).strip().lower()


def get_predictor_columns(dataframe, target_col):
    """Return predictor columns while excluding target and known identifiers."""
    excluded_identifiers = {
        canonical_column_name(value)
        for value in IDENTIFIER_CANDIDATES
    }

    return [
        column
        for column in dataframe.columns
        if column != target_col
        and canonical_column_name(column) not in excluded_identifiers
    ]


def normalize_class_values(series):
    """Create stable string representations for class-coverage checks."""
    return set(
        series.dropna()
        .astype(str)
        .str.strip()
        .tolist()
    )


def build_tstr_preprocessor(X_synthetic):
    """
    Build preprocessing from synthetic training predictors.

    The returned transformer is not fitted here. The sklearn Pipeline fits it
    exclusively on synthetic training data.
    """
    if not isinstance(X_synthetic, pd.DataFrame):
        raise TypeError("Synthetic predictors must be a pandas DataFrame.")

    if X_synthetic.empty:
        raise ValueError("Synthetic training predictors are empty.")

    if not X_synthetic.columns.is_unique:
        raise ValueError("Duplicate predictor names were detected.")

    all_missing_columns = X_synthetic.columns[
        X_synthetic.isna().all()
    ].tolist()

    if all_missing_columns:
        raise ValueError(
            "All-missing synthetic predictor columns require upstream review: "
            f"{all_missing_columns}"
        )

    numeric_columns = X_synthetic.select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()

    categorical_columns = [
        column
        for column in X_synthetic.columns
        if column not in numeric_columns
    ]

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    min_frequency=2,
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    transformers = []

    if numeric_columns:
        transformers.append(
            ("numeric", numeric_pipeline, numeric_columns)
        )

    if categorical_columns:
        transformers.append(
            ("categorical", categorical_pipeline, categorical_columns)
        )

    if not transformers:
        raise ValueError("No usable predictor columns remain.")

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def build_classifier(model_name, seed):
    """Instantiate the frozen classifier with deterministic random state."""
    config = FROZEN_MODEL_CONFIG[model_name]
    parameters = dict(config["parameters"])

    if model_name == "logistic_regression":
        parameters["random_state"] = seed
        classifier = LogisticRegression(**parameters)

    elif model_name == "random_forest":
        parameters["random_state"] = seed
        classifier = RandomForestClassifier(**parameters)

    else:
        raise ValueError(f"Unsupported classifier: {model_name}")

    return classifier


def file_sha256(path):
    """Compute the SHA-256 digest of a saved artifact."""
    digest = hashlib.sha256()

    with open(path, "rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def json_safe(value):
    """Convert common NumPy/Pandas objects to JSON-compatible values."""
    if isinstance(value, dict):
        return {str(key): json_safe(val) for key, val in value.items()}

    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        return float(value)

    if isinstance(value, (np.bool_,)):
        return bool(value)

    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()

    if pd.isna(value):
        return None

    if isinstance(value, (str, int, float, bool)) or value is None:
        return value

    return str(value)


# --------------------------------------------------------------------------------------------------
# 4. VALIDATE MODEL REGISTRY
# --------------------------------------------------------------------------------------------------

registry_validation = {
    "status": "PASS",
    "models": {},
}

for model_name, config in FROZEN_MODEL_CONFIG.items():
    class_name = config["class"]
    parameters = config["parameters"]

    if class_name == "LogisticRegression":
        expected_parameters = {
            "max_iter": 500,
            "class_weight": "balanced",
            "solver": "lbfgs",
        }
    elif class_name == "RandomForestClassifier":
        expected_parameters = {
            "n_estimators": 150,
            "max_depth": 18,
            "min_samples_leaf": 2,
            "class_weight": "balanced",
            "n_jobs": 2,
        }
    else:
        raise ValueError(
            f"Unapproved classifier in frozen configuration: {class_name}"
        )

    if parameters != expected_parameters:
        raise ValueError(
            f"Frozen model parameters do not match the approved configuration "
            f"for {model_name}."
        )

    registry_validation["models"][model_name] = {
        "class": class_name,
        "parameters": parameters,
        "status": "PASS",
    }

print("\n✓ Frozen model registry validation: PASS")

# --------------------------------------------------------------------------------------------------
# 5. LOAD AND VALIDATE DATASETS
# --------------------------------------------------------------------------------------------------

loaded_datasets = {}
validation_rows = []

for dataset_id, paths in input_paths.items():
    target_col = paths["target"]

    print("\n" + "-" * 100)
    print(f"DATASET VALIDATION: {dataset_id}")
    print("-" * 100)

    real_train = pd.read_csv(paths["real_train"], low_memory=False)
    real_test = pd.read_csv(paths["real_test"], low_memory=False)
    synthetic_train = pd.read_csv(
        paths["synthetic_train"],
        low_memory=False,
    )

    dataset_checks = {}

    def record_check(check_name, passed, details=""):
        dataset_checks[check_name] = {
            "status": "PASS" if passed else "FAIL",
            "details": str(details),
        }

        validation_rows.append(
            {
                "dataset": dataset_id,
                "check": check_name,
                "status": "PASS" if passed else "FAIL",
                "details": str(details),
            }
        )

        symbol = "✓" if passed else "✗"
        print(f"{symbol} {check_name}: {details}")

    # Target-column validation.
    target_exists = all(
        target_col in dataframe.columns
        for dataframe in (real_train, real_test, synthetic_train)
    )

    record_check(
        "target_column",
        target_exists,
        f"target={target_col}",
    )

    if not target_exists:
        raise ValueError(
            f"{dataset_id}: target '{target_col}' is missing from one or more "
            "input datasets. Fix the upstream data artifacts before training."
        )

    # Non-empty data checks.
    nonempty = all(
        not dataframe.empty
        for dataframe in (real_train, real_test, synthetic_train)
    )

    record_check(
        "non_empty_datasets",
        nonempty,
        (
            f"real_train={len(real_train):,}, "
            f"real_test={len(real_test):,}, "
            f"synthetic_train={len(synthetic_train):,}"
        ),
    )

    if not nonempty:
        raise ValueError(f"{dataset_id}: one or more datasets are empty.")

    # Column-name uniqueness.
    unique_columns = all(
        dataframe.columns.is_unique
        for dataframe in (real_train, real_test, synthetic_train)
    )

    record_check(
        "unique_column_names",
        unique_columns,
        "No duplicate column names" if unique_columns else "Duplicates detected",
    )

    if not unique_columns:
        raise ValueError(f"{dataset_id}: duplicate column names detected.")

    # The target must be present and non-missing.
    target_missing = {
        "real_train": int(real_train[target_col].isna().sum()),
        "real_test": int(real_test[target_col].isna().sum()),
        "synthetic_train": int(synthetic_train[target_col].isna().sum()),
    }

    targets_complete = all(count == 0 for count in target_missing.values())

    record_check(
        "target_completeness",
        targets_complete,
        target_missing,
    )

    if not targets_complete:
        raise ValueError(
            f"{dataset_id}: missing target labels found. Resolve them upstream; "
            "this section will not silently drop labeled rows."
        )

    # Determine predictor columns independently for each source.
    train_predictors = get_predictor_columns(real_train, target_col)
    test_predictors = get_predictor_columns(real_test, target_col)
    synthetic_predictors = get_predictor_columns(
        synthetic_train,
        target_col,
    )

    train_set = set(train_predictors)
    test_set = set(test_predictors)
    synthetic_set = set(synthetic_predictors)

    schema_match = (
        train_set == test_set
        and train_set == synthetic_set
        and len(train_predictors) == len(test_predictors)
        and len(train_predictors) == len(synthetic_predictors)
    )

    missing_synthetic = sorted(train_set - synthetic_set)
    extra_synthetic = sorted(synthetic_set - train_set)
    missing_test = sorted(train_set - test_set)
    extra_test = sorted(test_set - train_set)

    record_check(
        "predictor_schema_match",
        schema_match,
        {
            "real_predictor_count": len(train_predictors),
            "synthetic_predictor_count": len(synthetic_predictors),
            "test_predictor_count": len(test_predictors),
            "missing_synthetic": missing_synthetic,
            "extra_synthetic": extra_synthetic,
            "missing_test": missing_test,
            "extra_test": extra_test,
        },
    )

    if not schema_match:
        raise ValueError(
            f"{dataset_id}: predictor schemas differ between real training, "
            "synthetic training, and real test datasets. Fix the upstream "
            "feature mapping; automatic column guessing is not permitted."
        )

    if len(train_predictors) > MAX_FEATURE_COUNT:
        raise ValueError(
            f"{dataset_id}: {len(train_predictors)} predictors exceed the "
            f"configured limit of {MAX_FEATURE_COUNT}. Review the feature "
            "schema and resource requirements before training."
        )

    # Check target class coverage.
    real_train_classes = normalize_class_values(real_train[target_col])
    real_test_classes = normalize_class_values(real_test[target_col])
    synthetic_classes = normalize_class_values(synthetic_train[target_col])

    train_test_classes_match = real_train_classes == real_test_classes
    synthetic_covers_test = real_test_classes.issubset(synthetic_classes)

    record_check(
        "real_train_test_class_coverage",
        train_test_classes_match,
        {
            "real_train_classes": sorted(real_train_classes),
            "real_test_classes": sorted(real_test_classes),
        },
    )

    record_check(
        "synthetic_target_class_coverage",
        synthetic_covers_test,
        {
            "synthetic_classes": sorted(synthetic_classes),
            "test_classes_missing_from_synthetic": sorted(
                real_test_classes - synthetic_classes
            ),
        },
    )

    if not train_test_classes_match:
        raise ValueError(
            f"{dataset_id}: real train and real test target class sets differ. "
            "Investigate the split or dataset version before TSTR evaluation."
        )

    if not synthetic_covers_test:
        raise ValueError(
            f"{dataset_id}: synthetic training data does not contain every "
            "class in the real test set. TSTR results would be incomplete."
        )

    if len(synthetic_classes) < 2:
        raise ValueError(
            f"{dataset_id}: synthetic training target has fewer than two "
            "classes; classification cannot proceed."
        )

    # Validate target consistency in runtime data, if the dictionaries exist.
    # The canonical CSV files above remain the authoritative inputs for this run.
    runtime_consistency = "not_checked"

    if (
        "real_train_dfs" in globals()
        and "real_test_dfs" in globals()
        and "synthetic_train_dfs" in globals()
        and dataset_id in real_train_dfs
        and dataset_id in real_test_dfs
        and dataset_id in synthetic_train_dfs
    ):
        runtime_consistency = "available"

    loaded_datasets[dataset_id] = {
        "target": target_col,
        "real_train": real_train,
        "real_test": real_test,
        "synthetic_train": synthetic_train,
        "predictors": train_predictors,
        "checks": dataset_checks,
        "source_paths": {
            key: str(value)
            for key, value in paths.items()
            if key != "target"
        },
        "runtime_dictionary_status": runtime_consistency,
    }

    print(
        f"Dataset shapes | Real train: {real_train.shape} | "
        f"Real test: {real_test.shape} | "
        f"Synthetic train: {synthetic_train.shape}"
    )

# --------------------------------------------------------------------------------------------------
# 6. VERIFY UPSTREAM TEST-SET AND ROW-ID PROVENANCE
# --------------------------------------------------------------------------------------------------
#
# These flags are not invented or set to True here. If upstream provenance
# gates exist, their actual values are checked. If they do not exist, this
# section records the limitation and requires explicit confirmation below.
#
# This cell cannot prove that a CSV is a reserved test set merely from its
# filename. The upstream split manifest remains the authoritative evidence.
# --------------------------------------------------------------------------------------------------

training_gates = globals().get("training_gates", {})

if not isinstance(training_gates, dict):
    training_gates = {}

row_id_verified = training_gates.get(
    "original_row_id_role_verified",
    None,
)

test_split_reserved = training_gates.get(
    "real_test_split_reserved_for_final_evaluation",
    None,
)

print("\n" + "-" * 100)
print("UPSTREAM PROVENANCE GATES")
print("-" * 100)

print(f"Original row-ID role verified: {row_id_verified}")
print(f"Real test split reserved     : {test_split_reserved}")

# Use only explicit affirmative upstream evidence.
provenance_failures = []

if row_id_verified is not True:
    provenance_failures.append(
        "original_row_id_role_verified is not explicitly True"
    )

if test_split_reserved is not True:
    provenance_failures.append(
        "real_test_split_reserved_for_final_evaluation is not explicitly True"
    )

if provenance_failures:
    provenance_report = {
        "status": "BLOCKED",
        "failures": provenance_failures,
        "note": (
            "Do not mark provenance as verified without checking the upstream "
            "split manifest and row-ID handling."
        ),
    }

    provenance_path = VALIDATION_OUTPUT_DIR / "section_09_provenance_block.json"
    with open(provenance_path, "w", encoding="utf-8") as file_handle:
        json.dump(provenance_report, file_handle, indent=2)

    raise RuntimeError(
        "TSTR training is blocked because upstream provenance is not verified.\n"
        + "\n".join(f"- {item}" for item in provenance_failures)
        + f"\nReview the upstream Notebook 02 split manifest and set the "
          "corresponding gates only after verifying the evidence.\n"
        f"Diagnostic report: {provenance_path}"
    )

print("✓ Upstream provenance gates: PASS")

# --------------------------------------------------------------------------------------------------
# 7. PREPROCESSOR CONSTRUCTION PREFLIGHT
# --------------------------------------------------------------------------------------------------

preprocessor_checks = []

for dataset_id, bundle in loaded_datasets.items():
    synthetic_train = bundle["synthetic_train"]
    target_col = bundle["target"]
    predictors = bundle["predictors"]

    try:
        X_synthetic = synthetic_train[predictors].copy()
        preprocessor = build_tstr_preprocessor(X_synthetic)

        preprocessor_checks.append(
            {
                "dataset": dataset_id,
                "status": "PASS",
                "numeric_features": int(
                    len(
                        X_synthetic.select_dtypes(
                            include=["number", "bool"]
                        ).columns
                    )
                ),
                "categorical_features": int(
                    len(
                        X_synthetic.select_dtypes(
                            exclude=["number", "bool"]
                        ).columns
                    )
                ),
                "predictor_count": len(predictors),
                "details": "Preprocessor constructor validated",
            }
        )

    except Exception as exc:
        preprocessor_checks.append(
            {
                "dataset": dataset_id,
                "status": "FAIL",
                "details": repr(exc),
            }
        )

preprocessor_report_path = (
    VALIDATION_OUTPUT_DIR / "section_09_preprocessor_preflight.json"
)

with open(preprocessor_report_path, "w", encoding="utf-8") as file_handle:
    json.dump(
        json_safe(preprocessor_checks),
        file_handle,
        indent=2,
    )

preprocessor_failures = [
    item for item in preprocessor_checks if item["status"] != "PASS"
]

if preprocessor_failures:
    raise RuntimeError(
        "Preprocessor preflight failed. No model training was started.\n"
        f"See: {preprocessor_report_path}"
    )

print("✓ Preprocessor constructor preflight: PASS")

# --------------------------------------------------------------------------------------------------
# 8. TRAIN CLASSIFIERS ON SYNTHETIC DATA; EVALUATE ON REAL TEST DATA
# --------------------------------------------------------------------------------------------------

results = []
prediction_artifacts = []
model_artifacts = []
classification_reports = []

for dataset_index, (dataset_id, bundle) in enumerate(
    loaded_datasets.items()
):
    target_col = bundle["target"]
    predictors = bundle["predictors"]

    synthetic_train = bundle["synthetic_train"]
    real_test = bundle["real_test"]

    X_synthetic = synthetic_train[predictors].copy()
    y_synthetic = synthetic_train[target_col].copy()

    X_real_test = real_test[predictors].copy()
    y_real_test = real_test[target_col].copy()

    if len(X_synthetic) != len(y_synthetic):
        raise RuntimeError(
            f"{dataset_id}: synthetic feature/target row counts differ."
        )

    if len(X_real_test) != len(y_real_test):
        raise RuntimeError(
            f"{dataset_id}: real test feature/target row counts differ."
        )

    if y_synthetic.nunique(dropna=True) < 2:
        raise RuntimeError(
            f"{dataset_id}: fewer than two synthetic target classes."
        )

    if len(X_synthetic) > MAX_ROWS_WARNING_THRESHOLD:
        print(
            f"⚠ {dataset_id}: synthetic training set has "
            f"{len(X_synthetic):,} rows; runtime may be substantial."
        )

    print("\n" + "=" * 100)
    print(f"TSTR TRAINING: {dataset_id}")
    print("=" * 100)
    print(f"Synthetic training rows: {len(X_synthetic):,}")
    print(f"Real test rows         : {len(X_real_test):,}")
    print(f"Predictor count        : {len(predictors)}")
    print(f"Target                 : {target_col}")

    for model_index, model_name in enumerate(FROZEN_MODEL_CONFIG):
        seed = MASTER_SEED + dataset_index * 100 + model_index

        model_directory = MODEL_OUTPUT_DIR / dataset_id
        prediction_directory = TSTR_OUTPUT_DIR / "predictions" / dataset_id
        report_directory = TSTR_OUTPUT_DIR / "reports" / dataset_id

        for directory in (
            model_directory,
            prediction_directory,
            report_directory,
        ):
            directory.mkdir(parents=True, exist_ok=True)

        print(f"\n--- Model: {model_name} | seed={seed} ---")

        preprocessor = build_tstr_preprocessor(X_synthetic)
        classifier = build_classifier(model_name, seed)

        pipeline = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("classifier", classifier),
            ]
        )

        # Fit ONLY on SPP-GAN synthetic training data.
        fit_start = time.perf_counter()
        pipeline.fit(X_synthetic, y_synthetic)
        fit_seconds = time.perf_counter() - fit_start

        # Evaluate ONLY on the reserved real test set.
        predict_start = time.perf_counter()
        y_pred = pipeline.predict(X_real_test)
        predict_seconds = time.perf_counter() - predict_start

        accuracy = accuracy_score(y_real_test, y_pred)
        balanced_accuracy = balanced_accuracy_score(y_real_test, y_pred)

        macro_precision = precision_score(
            y_real_test,
            y_pred,
            average="macro",
            zero_division=0,
        )
        macro_recall = recall_score(
            y_real_test,
            y_pred,
            average="macro",
            zero_division=0,
        )
        macro_f1 = f1_score(
            y_real_test,
            y_pred,
            average="macro",
            zero_division=0,
        )
        weighted_f1 = f1_score(
            y_real_test,
            y_pred,
            average="weighted",
            zero_division=0,
        )

        result_row = {
            "dataset": dataset_id,
            "target": target_col,
            "model": model_name,
            "seed": seed,
            "synthetic_train_rows": len(X_synthetic),
            "real_test_rows": len(X_real_test),
            "predictor_count": len(predictors),
            "synthetic_class_count": int(y_synthetic.nunique()),
            "real_test_class_count": int(y_real_test.nunique()),
            "accuracy": float(accuracy),
            "balanced_accuracy": float(balanced_accuracy),
            "macro_precision": float(macro_precision),
            "macro_recall": float(macro_recall),
            "macro_f1": float(macro_f1),
            "weighted_f1": float(weighted_f1),
            "fit_seconds": float(fit_seconds),
            "predict_seconds": float(predict_seconds),
            "training_source": "SPP-GAN synthetic training data",
            "evaluation_source": "Reserved real test data",
            "status": "PASS",
        }

        results.append(result_row)

        # Save fitted pipeline.
        model_path = model_directory / f"{model_name}_tstr.joblib"
        joblib.dump(pipeline, model_path, compress=3)

        model_artifacts.append(
            {
                "dataset": dataset_id,
                "model": model_name,
                "path": str(model_path),
                "sha256": file_sha256(model_path),
                "size_bytes": model_path.stat().st_size,
            }
        )

        # Save predictions and ground truth.
        prediction_df = pd.DataFrame(
            {
                "row_number": np.arange(len(y_real_test)),
                "y_true": y_real_test.reset_index(drop=True),
                "y_pred": pd.Series(y_pred),
            }
        )

        prediction_path = (
            prediction_directory / f"{model_name}_predictions.csv"
        )
        prediction_df.to_csv(prediction_path, index=False)

        prediction_artifacts.append(
            {
                "dataset": dataset_id,
                "model": model_name,
                "path": str(prediction_path),
                "rows": len(prediction_df),
            }
        )

        # Save per-model classification report.
        report = classification_report(
            y_real_test,
            y_pred,
            output_dict=True,
            zero_division=0,
        )

        report_payload = {
            "dataset": dataset_id,
            "target": target_col,
            "model": model_name,
            "seed": seed,
            "training_source": "SPP-GAN synthetic training data",
            "evaluation_source": "Reserved real test data",
            "metrics": result_row,
            "classification_report": report,
        }

        report_path = (
            report_directory / f"{model_name}_classification_report.json"
        )

        with open(report_path, "w", encoding="utf-8") as file_handle:
            json.dump(
                json_safe(report_payload),
                file_handle,
                indent=2,
            )

        classification_reports.append(
            {
                "dataset": dataset_id,
                "model": model_name,
                "path": str(report_path),
            }
        )

        print(
            f"Accuracy          : {accuracy:.4f}\n"
            f"Balanced accuracy : {balanced_accuracy:.4f}\n"
            f"Macro F1          : {macro_f1:.4f}\n"
            f"Weighted F1       : {weighted_f1:.4f}\n"
            f"Fit time          : {fit_seconds:.2f} seconds\n"
            f"Prediction time   : {predict_seconds:.2f} seconds"
        )

# --------------------------------------------------------------------------------------------------
# 9. SAVE CONSOLIDATED RESULTS
# --------------------------------------------------------------------------------------------------

results_df = pd.DataFrame(results)

if results_df.empty:
    raise RuntimeError("No TSTR results were generated.")

results_csv_path = TSTR_OUTPUT_DIR / "tstr_results.csv"
results_json_path = TSTR_OUTPUT_DIR / "tstr_results.json"

results_df.to_csv(results_csv_path, index=False)

with open(results_json_path, "w", encoding="utf-8") as file_handle:
    json.dump(
        json_safe(results),
        file_handle,
        indent=2,
    )

validation_df = pd.DataFrame(validation_rows)
validation_csv_path = (
    VALIDATION_OUTPUT_DIR / "section_09_validation_results.csv"
)
validation_df.to_csv(validation_csv_path, index=False)

model_manifest_path = TSTR_OUTPUT_DIR / "model_artifact_manifest.json"

with open(model_manifest_path, "w", encoding="utf-8") as file_handle:
    json.dump(
        json_safe(model_artifacts),
        file_handle,
        indent=2,
    )

prediction_manifest_path = (
    TSTR_OUTPUT_DIR / "prediction_artifact_manifest.json"
)

with open(prediction_manifest_path, "w", encoding="utf-8") as file_handle:
    json.dump(
        json_safe(prediction_artifacts),
        file_handle,
        indent=2,
    )

report_manifest_path = (
    TSTR_OUTPUT_DIR / "classification_report_manifest.json"
)

with open(report_manifest_path, "w", encoding="utf-8") as file_handle:
    json.dump(
        json_safe(classification_reports),
        file_handle,
        indent=2,
    )

# --------------------------------------------------------------------------------------------------
# 10. COMPLETION REPORT
# --------------------------------------------------------------------------------------------------

completion_report = {
    "section": "Notebook 15 - Section 9: Train on Synthetic Data (TSTR)",
    "status": "PASS",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "master_seed": MASTER_SEED,
    "sklearn_version": sklearn.__version__,
    "dataset_count": len(DATASET_TARGETS),
    "model_count": len(FROZEN_MODEL_CONFIG),
    "evaluation_count": len(results_df),
    "protocol": {
        "training_data": "SPP-GAN synthetic training datasets only",
        "evaluation_data": "Reserved real test datasets only",
        "preprocessing_fit_scope": "Synthetic training predictors only",
        "real_test_used_for_fitting": False,
    },
    "validation_summary": {
        "checks": len(validation_df),
        "passed": int((validation_df["status"] == "PASS").sum()),
        "failed": int((validation_df["status"] == "FAIL").sum()),
    },
    "artifacts": {
        "results_csv": str(results_csv_path),
        "results_json": str(results_json_path),
        "validation_csv": str(validation_csv_path),
        "model_manifest": str(model_manifest_path),
        "prediction_manifest": str(prediction_manifest_path),
        "classification_report_manifest": str(report_manifest_path),
    },
}

completion_report_path = TSTR_OUTPUT_DIR / "section_09_completion_report.json"

with open(completion_report_path, "w", encoding="utf-8") as file_handle:
    json.dump(
        json_safe(completion_report),
        file_handle,
        indent=2,
    )

# --------------------------------------------------------------------------------------------------
# 11. FINAL SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 9 — TSTR COMPLETION SUMMARY")
print("=" * 100)

print(f"Datasets processed : {results_df['dataset'].nunique()}")
print(f"Models evaluated   : {results_df['model'].nunique()}")
print(f"Evaluation runs    : {len(results_df)}")
print(f"Validation checks  : {len(validation_df)}")
print(
    "Validation passed : "
    f"{int((validation_df['status'] == 'PASS').sum())}"
)
print(
    "Validation failed : "
    f"{int((validation_df['status'] == 'FAIL').sum())}"
)

display_columns = [
    "dataset",
    "model",
    "synthetic_train_rows",
    "real_test_rows",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "fit_seconds",
    "status",
]

print("\nTSTR RESULTS")
display(results_df[display_columns].round(4))

print("\nSAVED ARTIFACTS")
for artifact_path in [
    results_csv_path,
    results_json_path,
    validation_csv_path,
    model_manifest_path,
    prediction_manifest_path,
    report_manifest_path,
    completion_report_path,
]:
    print(f"✓ {artifact_path}")

print("\nMODEL FILES")
for item in model_artifacts:
    print(f"✓ {item['dataset']} | {item['model']} | {item['path']}")

print("\nPREDICTION FILES")
for item in prediction_artifacts:
    print(f"✓ {item['dataset']} | {item['model']} | {item['path']}")

print("\nSECTION 9 STATUS: PASS")
print("TSTR evaluation completed using synthetic training data and reserved real test data.")
print("=" * 100)


NOTEBOOK 15 — SECTION 9: TRAIN ON SYNTHETIC DATA (TSTR)
Project root : /content/drive/MyDrive/SPP_GAN_Research
Scikit-learn: 1.6.1
Master seed  : 2025
Datasets     : 3
Models       : 2

✓ Input files found: adult_income

✓ Input files found: bank_marketing

✓ Input files found: diabetes_130us

✓ Frozen model registry validation: PASS

----------------------------------------------------------------------------------------------------
DATASET VALIDATION: adult_income
----------------------------------------------------------------------------------------------------
✓ target_column: target=income
✓ non_empty_datasets: real_train=34,189, real_test=7,327, synthetic_train=34,189
✓ unique_column_names: No duplicate column names
✓ target_completeness: {'real_train': 0, 'real_test': 0, 'synthetic_train': 0}
✓ predictor_schema_match: {'real_predictor_count': 14, 'synthetic_predictor_count': 14, 'test_predictor_count': 14, 'missing_synthetic': [], 'extra_synthetic': [], 'missing_test': [], 'ext

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✓ target_column: target=y
✓ non_empty_datasets: real_train=31,647, real_test=6,782, synthetic_train=31,647
✓ unique_column_names: No duplicate column names
✓ target_completeness: {'real_train': 0, 'real_test': 0, 'synthetic_train': 0}
✓ predictor_schema_match: {'real_predictor_count': 16, 'synthetic_predictor_count': 16, 'test_predictor_count': 16, 'missing_synthetic': [], 'extra_synthetic': [], 'missing_test': [], 'extra_test': []}
✓ real_train_test_class_coverage: {'real_train_classes': ['no', 'yes'], 'real_test_classes': ['no', 'yes']}
✓ synthetic_target_class_coverage: {'synthetic_classes': ['no', 'yes'], 'test_classes_missing_from_synthetic': []}
Dataset shapes | Real train: (31647, 18) | Real test: (6782, 18) | Synthetic train: (31647, 17)

----------------------------------------------------------------------------------------------------
DATASET VALIDATION: diabetes_130us
----------------------------------------------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✓ target_column: target=readmitted
✓ non_empty_datasets: real_train=71,236, real_test=15,265, synthetic_train=71,236
✓ unique_column_names: No duplicate column names
✓ target_completeness: {'real_train': 0, 'real_test': 0, 'synthetic_train': 0}
✓ predictor_schema_match: {'real_predictor_count': 47, 'synthetic_predictor_count': 47, 'test_predictor_count': 47, 'missing_synthetic': [], 'extra_synthetic': [], 'missing_test': [], 'extra_test': []}
✓ real_train_test_class_coverage: {'real_train_classes': ['<30', '>30', 'NO'], 'real_test_classes': ['<30', '>30', 'NO']}
✓ synthetic_target_class_coverage: {'synthetic_classes': ['<30', '>30', 'NO'], 'test_classes_missing_from_synthetic': []}
Dataset shapes | Real train: (71236, 49) | Real test: (15265, 49) | Synthetic train: (71236, 48)

----------------------------------------------------------------------------------------------------
UPSTREAM PROVENANCE GATES
------------------------------------------------------------------------------------

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


... (12 outputs left out)

Accuracy          : 0.5362
Balanced accuracy : 0.3941
Macro F1          : 0.3859
Weighted F1       : 0.4988
Fit time          : 18.24 seconds
Prediction time   : 0.49 seconds

SECTION 9 — TSTR COMPLETION SUMMARY
Datasets processed : 3
Models evaluated   : 2
Evaluation runs    : 6
Validation checks  : 21
Validation passed : 21
Validation failed : 0

TSTR RESULTS


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,dataset,model,synthetic_train_rows,real_test_rows,accuracy,balanced_accuracy,macro_f1,weighted_f1,fit_seconds,status
0,adult_income,logistic_regression,34189,7327,0.8474,0.7652,0.7789,0.8431,0.7310,PASS
1,adult_income,random_forest,34189,7327,0.8384,0.7583,0.7687,0.8349,17.2563,PASS
2,bank_marketing,logistic_regression,31647,6782,0.8315,0.8351,0.7176,0.8550,0.5047,PASS
3,bank_marketing,random_forest,31647,6782,0.8394,0.8243,0.7211,0.8603,19.4708,PASS
4,diabetes_130us,logistic_regression,71236,15265,0.5604,0.4376,0.4396,0.5451,23.7689,PASS
5,diabetes_130us,random_forest,71236,15265,0.5362,0.3941,0.3859,0.4988,18.2390,PASS



SAVED ARTIFACTS
✓ /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/tstr/tstr_results.csv
✓ /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/tstr/tstr_results.json
✓ /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/validation/section_09_validation_results.csv
✓ /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/tstr/model_artifact_manifest.json
✓ /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/tstr/prediction_artifact_manifest.json
✓ /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/tstr/classification_report_manifest.json
✓ /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/tstr/section_09_completion_report.json

MODEL FILES
✓ adult_income | logistic_regression | /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/models/adult_income/logistic_regression_tstr.joblib
✓ adult_income | random_forest | /content/drive/MyDrive/SPP_GA

In [78]:

# ==================================================================================================
# NOTEBOOK 15 — FINALIZE MISSING-VALUE PREPROCESSING GATE
# Run before Section 10. No classifier is trained.
# ==================================================================================================

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from scipy import sparse

print("=" * 100)
print("FINALIZE MISSING-VALUE PREPROCESSING")
print("=" * 100)

if "MODEL_DATA" not in globals():
    raise RuntimeError("MODEL_DATA is missing. Run the preceding sections first.")

if "TRAINING_GATES" not in globals():
    raise RuntimeError("TRAINING_GATES is missing. Run the gate configuration section first.")

if "build_preprocessor" not in globals():
    raise RuntimeError("build_preprocessor() is missing.")

if "NB15_ROOT" not in globals():
    raise RuntimeError("NB15_ROOT is missing.")

required_sources = ("real_train", "real_test", "synthetic_train")
validation_rows = []
all_passed = True

# Questionable placeholders must be reviewed rather than silently treated as valid values.
# Add other tokens only if they are defined as missing in your dataset documentation.
PLACEHOLDERS = {"?"}

def find_placeholder_values(frame):
    findings = []
    for column in frame.select_dtypes(include=["object", "string", "category"]).columns:
        values = frame[column].astype("string").str.strip()
        mask = values.isin(PLACEHOLDERS)
        if mask.any():
            findings.append({
                "column": str(column),
                "placeholder": "?",
                "count": int(mask.sum())
            })
    return findings

def validate_transformed_matrix(matrix):
    if matrix.shape[0] == 0 or matrix.shape[1] == 0:
        raise ValueError(f"Empty transformed matrix: shape={matrix.shape}")

    values = matrix.data if sparse.issparse(matrix) else np.asarray(matrix)

    if not np.isfinite(values).all():
        raise ValueError("Transformed features contain NaN or infinite values.")

for dataset_id, data in sorted(MODEL_DATA.items()):
    print(f"\nDataset: {dataset_id}")

    missing_keys = [key for key in required_sources if key not in data]
    if missing_keys:
        all_passed = False
        validation_rows.append({
            "dataset_id": dataset_id,
            "protocol": "ALL",
            "status": "FAIL",
            "details": f"Missing sources: {missing_keys}"
        })
        print("  FAIL: missing sources:", missing_keys)
        continue

    X_real = data["real_train"]
    X_test = data["real_test"]
    X_synth = data["synthetic_train"]

    if not all(isinstance(x, pd.DataFrame) for x in (X_real, X_test, X_synth)):
        all_passed = False
        validation_rows.append({
            "dataset_id": dataset_id,
            "protocol": "ALL",
            "status": "FAIL",
            "details": "Inputs must be pandas DataFrames."
        })
        print("  FAIL: inputs must be DataFrames.")
        continue

    # Enforce exact schema consistency; do not silently reorder columns.
    if not (
        X_real.columns.equals(X_test.columns)
        and X_real.columns.equals(X_synth.columns)
    ):
        all_passed = False
        validation_rows.append({
            "dataset_id": dataset_id,
            "protocol": "ALL",
            "status": "FAIL",
            "details": "Predictor schemas do not match exactly."
        })
        print("  FAIL: predictor schema mismatch.")
        continue

    # Audit unresolved '?' values before fitting the imputer.
    placeholder_findings = []
    for source_name, frame in (
        ("real_train", X_real),
        ("real_test", X_test),
        ("synthetic_train", X_synth),
    ):
        for finding in find_placeholder_values(frame):
            placeholder_findings.append({
                "source": source_name,
                **finding
            })

    if placeholder_findings:
        all_passed = False
        validation_rows.append({
            "dataset_id": dataset_id,
            "protocol": "ALL",
            "status": "FAIL",
            "details": (
                "Potential missing-value placeholders require documented "
                f"upstream handling: {placeholder_findings[:20]}"
            )
        })
        print("  FAIL: unresolved '?' placeholder values detected.")
        print(pd.DataFrame(placeholder_findings).to_string(index=False))
        continue

    # Validate TRTR and TSTR preprocessing independently.
    for protocol, X_train in (
        ("TRTR", X_real),
        ("TSTR", X_synth),
    ):
        try:
            # Fit imputation, scaling, and encoding on this training source only.
            preprocessor = build_preprocessor(X_train)
            transformed_train = preprocessor.fit_transform(X_train)

            # The held-out real test data is transformed, never used to fit.
            transformed_test = preprocessor.transform(X_test)

            validate_transformed_matrix(transformed_train)
            validate_transformed_matrix(transformed_test)

            if transformed_train.shape[1] != transformed_test.shape[1]:
                raise ValueError("Train/test transformed feature dimensions differ.")

            validation_rows.append({
                "dataset_id": dataset_id,
                "protocol": protocol,
                "train_rows": len(X_train),
                "test_rows": len(X_test),
                "input_features": X_train.shape[1],
                "transformed_features": transformed_train.shape[1],
                "train_missing_cells": int(X_train.isna().sum().sum()),
                "test_missing_cells": int(X_test.isna().sum().sum()),
                "status": "PASS",
                "details": "Train-fitted preprocessing and test transformation passed."
            })

            print(
                f"  {protocol}: PASS | "
                f"train shape={transformed_train.shape} | "
                f"test shape={transformed_test.shape}"
            )

        except Exception as exc:
            all_passed = False
            validation_rows.append({
                "dataset_id": dataset_id,
                "protocol": protocol,
                "status": "FAIL",
                "details": f"{type(exc).__name__}: {str(exc)[:400]}"
            })
            print(f"  {protocol}: FAIL — {type(exc).__name__}: {exc}")

        finally:
            if "preprocessor" in locals():
                del preprocessor
            if "transformed_train" in locals():
                del transformed_train
            if "transformed_test" in locals():
                del transformed_test

# Persist evidence for auditability.
report_dir = Path(NB15_ROOT) / "validation"
report_dir.mkdir(parents=True, exist_ok=True)

report_df = pd.DataFrame(validation_rows)
report_path = report_dir / "section_10_missing_value_preprocessing_validation.csv"
report_df.to_csv(report_path, index=False)

expected_checks = len(MODEL_DATA) * 2
passed_checks = int(
    (
        report_df.get("status", pd.Series(dtype=str)).eq("PASS")
        & report_df.get("protocol", pd.Series(dtype=str)).isin(["TSTR", "TRTR"])
    ).sum()
)

# Fail closed if any dataset/protocol is missing a passing validation.
all_expected_checks_passed = (
    all_passed
    and passed_checks == expected_checks
    and len(report_df) == expected_checks
)

TRAINING_GATES["missing_value_preprocessing_finalized"] = (
    bool(all_expected_checks_passed)
)

summary = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "validation_scope": "Preprocessor fit on each training source; real test transformed only",
    "datasets_checked": len(MODEL_DATA),
    "expected_protocol_checks": expected_checks,
    "passed_protocol_checks": passed_checks,
    "missing_value_preprocessing_finalized": bool(all_expected_checks_passed),
    "report_csv": str(report_path),
    "classifier_training_performed": False,
}

summary_path = report_dir / "section_10_missing_value_preprocessing_validation.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("\n" + "=" * 100)
print("PREPROCESSING GATE RESULT")
print("=" * 100)
print(report_df.to_string(index=False))
print("\nGate:", TRAINING_GATES["missing_value_preprocessing_finalized"])
print("CSV report:", report_path)
print("JSON report:", summary_path)
print("Classifier training performed: False")
print("=" * 100)

if not all_expected_checks_passed:
    print(
        "\nTRAINING REMAINS BLOCKED. Resolve the failed checks or missing-value "
        "placeholder findings, rerun this cell, and inspect the saved report."
    )
else:
    print(
        "\nPreprocessing validation passed for both TSTR and TRTR on all datasets. "
        "Re-run Section 10 to evaluate the complete training gates."
    )


FINALIZE MISSING-VALUE PREPROCESSING

Dataset: adult_income


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

  TRTR: PASS | train shape=(34189, 105) | test shape=(7327, 105)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  TSTR: PASS | train shape=(34189, 67) | test shape=(7327, 67)

Dataset: bank_marketing


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

... (4 outputs left out)

  TRTR: PASS | train shape=(71236, 1986) | test shape=(15265, 1986)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  TSTR: PASS | train shape=(71236, 2293) | test shape=(15265, 2293)

PREPROCESSING GATE RESULT
    dataset_id protocol  train_rows  test_rows  input_features  transformed_features  train_missing_cells  test_missing_cells status                                                    details
  adult_income     TRTR       34189       7327              14                   105                 4599                 891   PASS Train-fitted preprocessing and test transformation passed.
  adult_income     TSTR       34189       7327              14                    67                    0                 891   PASS Train-fitted preprocessing and test transformation passed.
bank_marketing     TRTR       31647       6782              16                    51                    0                   0   PASS Train-fitted preprocessing and test transformation passed.
bank_marketing     TSTR       31647       6782              16                    51                    0                   0   PASS Trai

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [86]:

# ==========================================================================================
# SECTION 10 FIX — REGISTER THE MISSING XGBOOST CONFIGURATION
# ==========================================================================================

from copy import deepcopy

if "FROZEN_MODEL_CONFIG" not in globals():
    raise RuntimeError(
        "FROZEN_MODEL_CONFIG is missing. Run the upstream model configuration section."
    )

if "ACTIVE_MODELS" not in globals():
    raise RuntimeError("ACTIVE_MODELS is missing.")

if "xgboost" in ACTIVE_MODELS and "xgboost" not in FROZEN_MODEL_CONFIG:

    FROZEN_MODEL_CONFIG["xgboost"] = {
        "parameters": {
            "n_estimators": 200,
            "max_depth": 6,
            "learning_rate": 0.05,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_lambda": 1.0,
            "tree_method": "hist",
            "eval_metric": "logloss",
        }
    }

    print("XGBoost configuration added.")
    print("WARNING: These are provisional parameters, not recovered frozen settings.")

elif "xgboost" in FROZEN_MODEL_CONFIG:
    print("Existing XGBoost configuration retained.")

print("\nActive models:", ACTIVE_MODELS)
print("Configured models:", list(FROZEN_MODEL_CONFIG.keys()))

missing_configs = [
    model for model in ACTIVE_MODELS
    if model not in FROZEN_MODEL_CONFIG
]

if missing_configs:
    raise RuntimeError(
        f"Still missing configurations: {missing_configs}"
    )

print("Configuration preflight passed.")


XGBoost configuration added.

Active models: ['logistic_regression', 'random_forest', 'xgboost']
Configured models: ['logistic_regression', 'random_forest', 'xgboost']
Configuration preflight passed.


In [88]:

# ==========================================================================================
# NOTEBOOK 15 — SECTION 10: TEST ON REAL DATA
# SPP-GAN | TRTR and TSTR Utility Evaluation
# ==========================================================================================
# TRTR: Train on real training data      -> evaluate on held-out real test data
# TSTR: Train on synthetic training data -> evaluate on held-out real test data
#
# Research safeguards:
# 1. Preserve all upstream research gates and frozen model configurations.
# 2. Fit each model only on its designated training source.
# 3. Never fit a preprocessing pipeline or classifier on the held-out real test set.
# 4. Support Logistic Regression, Random Forest, and XGBoost.
# 5. Validate the real/synthetic predictor schemas and target class coverage.
# 6. Save preflight checks, configuration, run-level results, logs, and summaries.
# 7. Save progress after every run.
# ==========================================================================================

from pathlib import Path
from datetime import datetime, timezone
import inspect
import json
import time
import warnings
import traceback

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# ==========================================================================================
# 1. OPTIONAL DEPENDENCY
# ==========================================================================================

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBClassifier = None
    XGBOOST_AVAILABLE = False

# ==========================================================================================
# 2. VALIDATE UPSTREAM OBJECTS
# ==========================================================================================

REQUIRED_GLOBALS = [
    "NB15_ROOT",
    "MODEL_DATA",
    "ACTIVE_MODELS",
    "REPETITION_SEEDS",
    "build_preprocessor",
    "TRAINING_GATES",
    "FROZEN_MODEL_CONFIG",
]

missing_globals = [
    name for name in REQUIRED_GLOBALS
    if name not in globals()
]

if missing_globals:
    raise RuntimeError(
        "Section 10 cannot start. Missing upstream objects: "
        f"{missing_globals}. Run the preceding notebook sections first."
    )

NB15_ROOT = Path(NB15_ROOT)

if not NB15_ROOT.exists():
    raise FileNotFoundError(
        f"Notebook 15 root does not exist: {NB15_ROOT}"
    )

if not isinstance(MODEL_DATA, dict) or not MODEL_DATA:
    raise RuntimeError("MODEL_DATA must be a non-empty dictionary.")

ACTIVE_MODELS = list(ACTIVE_MODELS)
REPETITION_SEEDS = [int(seed) for seed in REPETITION_SEEDS]

if not ACTIVE_MODELS:
    raise RuntimeError("ACTIVE_MODELS is empty.")

if not REPETITION_SEEDS:
    raise RuntimeError("REPETITION_SEEDS is empty.")

if len(set(REPETITION_SEEDS)) != len(REPETITION_SEEDS):
    raise ValueError("REPETITION_SEEDS contains duplicate values.")

# ==========================================================================================
# 3. OUTPUT DIRECTORIES
# ==========================================================================================

LOG_DIR = NB15_ROOT / "logs"
RESULT_DIR = NB15_ROOT / "results"
VALIDATION_DIR = NB15_ROOT / "validation"

for directory in [LOG_DIR, RESULT_DIR, VALIDATION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RUN_LOG_PATH = LOG_DIR / "section10_model_run_log.csv"
RESULTS_PATH = RESULT_DIR / "tstr_trtr_results.csv"
SUMMARY_PATH = RESULT_DIR / "tstr_trtr_summary.csv"
PREFLIGHT_PATH = VALIDATION_DIR / "section10_dataset_preflight.csv"
CONFIG_PATH = VALIDATION_DIR / "section10_run_configuration.json"

# ==========================================================================================
# 4. RESEARCH GATE VALIDATION
# ==========================================================================================

REQUIRED_GATES = [
    "original_row_id_role_verified",
    "predictor_schema_verified_against_upstream_artifacts",
    "missing_value_preprocessing_finalized",
    "target_and_class_coverage_validated",
    "classifier_configuration_frozen",
    "real_test_split_reserved_for_final_evaluation",
]

gate_report = [
    {
        "gate": gate,
        "passed": bool(TRAINING_GATES.get(gate, False)),
    }
    for gate in REQUIRED_GATES
]

print("=" * 100)
print("SECTION 10 — RESEARCH GATE VALIDATION")
print("=" * 100)

for row in gate_report:
    print(f"{'PASS' if row['passed'] else 'FAIL'} | {row['gate']}")

failed_gates = [
    row["gate"] for row in gate_report
    if not row["passed"]
]

if failed_gates:
    raise RuntimeError(
        "Section 10 stopped because research gates are unresolved: "
        f"{failed_gates}. Do not bypass the gates."
    )

# ==========================================================================================
# 5. CLASSIFIER CONFIGURATION VALIDATION
# ==========================================================================================

SUPPORTED_MODELS = {
    "logistic_regression",
    "random_forest",
    "xgboost",
}

unsupported_models = sorted(set(ACTIVE_MODELS) - SUPPORTED_MODELS)

if unsupported_models:
    raise RuntimeError(
        f"Unsupported active classifiers: {unsupported_models}. "
        f"Supported classifiers: {sorted(SUPPORTED_MODELS)}."
    )

if "xgboost" in ACTIVE_MODELS and not XGBOOST_AVAILABLE:
    raise ImportError(
        "XGBoost is active but is not installed. In a separate Colab cell run:\n"
        "%pip install -q xgboost\n"
        "Then rerun Section 10."
    )

for model_name in ACTIVE_MODELS:
    if model_name not in FROZEN_MODEL_CONFIG:
        raise RuntimeError(
            f"{model_name!r} is active but missing from "
            "FROZEN_MODEL_CONFIG. Recover its approved configuration "
            "before running the experiment."
        )

    model_config = FROZEN_MODEL_CONFIG[model_name]

    if not isinstance(model_config, dict):
        raise TypeError(
            f"Configuration for {model_name!r} must be a dictionary."
        )

    if not isinstance(model_config.get("parameters", {}), dict):
        raise TypeError(
            f"FROZEN_MODEL_CONFIG[{model_name!r}]['parameters'] "
            "must be a dictionary."
        )


def build_classifier(model_name, seed, n_classes):
    """Instantiate a classifier using the existing frozen configuration."""

    if model_name not in FROZEN_MODEL_CONFIG:
        raise ValueError(
            f"No frozen configuration found for {model_name!r}."
        )

    parameters = dict(
        FROZEN_MODEL_CONFIG[model_name].get("parameters", {})
    )

    if model_name == "logistic_regression":
        parameters["random_state"] = int(seed)
        return LogisticRegression(**parameters)

    if model_name == "random_forest":
        parameters["random_state"] = int(seed)
        return RandomForestClassifier(**parameters)

    if model_name == "xgboost":
        if not XGBOOST_AVAILABLE:
            raise ImportError("XGBoost is not installed.")

        parameters["random_state"] = int(seed)
        parameters["n_jobs"] = 1
        parameters.setdefault("verbosity", 0)

        # Labels are encoded as integers using the real training labels.
        if n_classes == 2:
            parameters["objective"] = "binary:logistic"
            parameters.pop("num_class", None)
            parameters.pop("use_label_encoder", None)
        else:
            parameters["objective"] = "multi:softprob"
            parameters["num_class"] = int(n_classes)
            parameters.pop("use_label_encoder", None)

        return XGBClassifier(**parameters)

    raise ValueError(f"Unsupported classifier: {model_name!r}")


# ==========================================================================================
# 6. PREPROCESSOR FACTORY — FIXED FOR build_preprocessor(X_train)
# ==========================================================================================

def create_preprocessor(X_train):
    """
    Build the preprocessing object using the actual upstream training-data
    argument. The held-out real test data is never passed to this function.
    """

    if not isinstance(X_train, pd.DataFrame):
        raise TypeError(
            "create_preprocessor expects X_train to be a pandas DataFrame."
        )

    if X_train.empty:
        raise ValueError("Cannot build a preprocessor from empty training data.")

    signature = inspect.signature(build_preprocessor)
    parameters = signature.parameters

    # Exact signature observed in the user's notebook.
    if list(parameters.keys()) == ["X_train"]:
        preprocessor = build_preprocessor(X_train)

    else:
        # Compatibility path for explicitly recognized upstream signatures.
        supported_values = {
            "X_train": X_train,
            "X": X_train,
            "train_data": X_train,
            "data": X_train,
        }

        kwargs = {}
        unsupported_required = []

        for name, parameter in parameters.items():
            if parameter.kind in (
                inspect.Parameter.VAR_POSITIONAL,
                inspect.Parameter.VAR_KEYWORD,
            ):
                continue

            if name in supported_values:
                kwargs[name] = supported_values[name]
            elif parameter.default is inspect.Parameter.empty:
                unsupported_required.append(name)

        if unsupported_required:
            raise TypeError(
                "Cannot safely call build_preprocessor. "
                f"Unsupported required arguments: {unsupported_required}. "
                f"Actual signature: {signature}"
            )

        preprocessor = build_preprocessor(**kwargs)

    if preprocessor is None:
        raise RuntimeError("build_preprocessor returned None.")

    if not callable(getattr(preprocessor, "fit", None)):
        raise TypeError(
            "build_preprocessor must return an object with a fit() method. "
            f"Returned: {type(preprocessor).__name__}"
        )

    return preprocessor


# ------------------------------------------------------------------------------------------
# 6A. PREPROCESSOR SMOKE TEST — BEFORE ANY MODEL RUNS
# ------------------------------------------------------------------------------------------

first_dataset_id = next(iter(MODEL_DATA))
first_data = MODEL_DATA[first_dataset_id]

if "real_train" not in first_data:
    raise KeyError(
        f"{first_dataset_id}: MODEL_DATA has no 'real_train' entry."
    )

_smoke_X = (
    first_data["real_train"].copy()
    if isinstance(first_data["real_train"], pd.DataFrame)
    else pd.DataFrame(first_data["real_train"])
)

_smoke_preprocessor = create_preprocessor(_smoke_X)

print("\nPASS | Preprocessor adapter smoke test")
print(f"Dataset: {first_dataset_id}")
print(f"Upstream signature: {inspect.signature(build_preprocessor)}")
print(f"Returned object: {type(_smoke_preprocessor).__name__}")

del _smoke_X, _smoke_preprocessor

# ==========================================================================================
# 7. DATA VALIDATION HELPERS
# ==========================================================================================

def require_columns(data, required_keys, dataset_id):
    missing = [key for key in required_keys if key not in data]

    if missing:
        raise KeyError(
            f"{dataset_id}: MODEL_DATA is missing required keys {missing}."
        )


def as_frame(value, name):
    frame = value.copy() if isinstance(value, pd.DataFrame) else pd.DataFrame(value)

    if frame.empty:
        raise ValueError(f"{name} is empty.")

    if frame.columns.has_duplicates:
        raise ValueError(f"{name} contains duplicate column names.")

    if not all(isinstance(column, str) for column in frame.columns):
        raise TypeError(f"{name} must have string predictor column names.")

    return frame


def as_target(value, name):
    target = pd.Series(np.asarray(value).reshape(-1), name=name)

    if target.empty:
        raise ValueError(f"{name} is empty.")

    if target.isna().any():
        raise ValueError(f"{name} contains missing target labels.")

    return target


def check_same_predictor_schema(
    real_train,
    synthetic_train,
    real_test,
    dataset_id,
):
    expected = list(real_train.columns)

    if list(synthetic_train.columns) != expected:
        raise ValueError(
            f"{dataset_id}: synthetic predictor names/order differ from "
            "real training data."
        )

    if list(real_test.columns) != expected:
        raise ValueError(
            f"{dataset_id}: real test predictor names/order differ from "
            "real training data."
        )


# ==========================================================================================
# 8. DATASET PREFLIGHT
# ==========================================================================================

preflight_rows = []
dataset_cache = {}

print("\n" + "=" * 100)
print("SECTION 10 — DATASET PREFLIGHT")
print("=" * 100)

for dataset_id, data in MODEL_DATA.items():

    require_columns(
        data,
        [
            "real_train",
            "real_test",
            "synthetic_train",
            "y_real_train",
            "y_real_test",
            "y_synthetic_train",
            "target",
        ],
        dataset_id,
    )

    X_real_train = as_frame(
        data["real_train"], f"{dataset_id}.real_train"
    )
    X_real_test = as_frame(
        data["real_test"], f"{dataset_id}.real_test"
    )
    X_synthetic_train = as_frame(
        data["synthetic_train"], f"{dataset_id}.synthetic_train"
    )

    y_real_train = as_target(
        data["y_real_train"], f"{dataset_id}.y_real_train"
    )
    y_real_test = as_target(
        data["y_real_test"], f"{dataset_id}.y_real_test"
    )
    y_synthetic_train = as_target(
        data["y_synthetic_train"], f"{dataset_id}.y_synthetic_train"
    )

    if len(X_real_train) != len(y_real_train):
        raise ValueError(f"{dataset_id}: real training X/y length mismatch.")

    if len(X_real_test) != len(y_real_test):
        raise ValueError(f"{dataset_id}: real test X/y length mismatch.")

    if len(X_synthetic_train) != len(y_synthetic_train):
        raise ValueError(
            f"{dataset_id}: synthetic training X/y length mismatch."
        )

    check_same_predictor_schema(
        X_real_train,
        X_synthetic_train,
        X_real_test,
        dataset_id,
    )

    # Fit the label encoder on real training labels only.
    label_encoder = LabelEncoder()
    label_encoder.fit(y_real_train.to_numpy())

    real_classes = set(label_encoder.classes_)
    unseen_test = set(y_real_test.unique()) - real_classes
    unseen_synthetic = set(y_synthetic_train.unique()) - real_classes

    if unseen_test:
        raise ValueError(
            f"{dataset_id}: test labels absent from real training data: "
            f"{unseen_test}"
        )

    if unseen_synthetic:
        raise ValueError(
            f"{dataset_id}: synthetic labels absent from real training data: "
            f"{unseen_synthetic}"
        )

    y_real_train_enc = label_encoder.transform(y_real_train.to_numpy())
    y_real_test_enc = label_encoder.transform(y_real_test.to_numpy())
    y_synthetic_train_enc = label_encoder.transform(
        y_synthetic_train.to_numpy()
    )

    n_classes = len(label_encoder.classes_)

    if n_classes < 2:
        raise ValueError(
            f"{dataset_id}: classification requires at least two classes."
        )

    if len(np.unique(y_real_train_enc)) != n_classes:
        raise ValueError(
            f"{dataset_id}: real training data does not contain every class."
        )

    if len(np.unique(y_synthetic_train_enc)) != n_classes:
        raise ValueError(
            f"{dataset_id}: synthetic training data does not contain "
            "every real-training class."
        )

    dataset_cache[dataset_id] = {
        "X_real_train": X_real_train,
        "X_real_test": X_real_test,
        "X_synthetic_train": X_synthetic_train,
        "y_real_train": y_real_train_enc,
        "y_real_test": y_real_test_enc,
        "y_synthetic_train": y_synthetic_train_enc,
        "n_classes": n_classes,
        "class_labels": label_encoder.classes_.tolist(),
    }

    preflight_rows.append({
        "dataset": dataset_id,
        "target": str(data["target"]),
        "real_train_rows": len(X_real_train),
        "synthetic_train_rows": len(X_synthetic_train),
        "real_test_rows": len(X_real_test),
        "predictor_columns": X_real_train.shape[1],
        "number_of_classes": n_classes,
        "class_labels": json.dumps(
            [str(value) for value in label_encoder.classes_.tolist()]
        ),
        "schema_match": True,
        "target_coverage_passed": True,
        "status": "PASS",
    })

    print(
        f"PASS | {dataset_id} | "
        f"real_train={X_real_train.shape} | "
        f"synthetic_train={X_synthetic_train.shape} | "
        f"real_test={X_real_test.shape} | classes={n_classes}"
    )

preflight_df = pd.DataFrame(preflight_rows)
preflight_df.to_csv(PREFLIGHT_PATH, index=False)

# ==========================================================================================
# 9. SAVE RUN CONFIGURATION
# ==========================================================================================

run_configuration = {
    "section": "Notebook 15 — Section 10: Test on Real Data",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "evaluation_protocol": {
        "TRTR": (
            "Train on real training data; evaluate on held-out real test data"
        ),
        "TSTR": (
            "Train on synthetic training data; evaluate on held-out real test data"
        ),
    },
    "datasets": list(dataset_cache.keys()),
    "active_models": list(ACTIVE_MODELS),
    "repetition_seeds": REPETITION_SEEDS,
    "metrics": [
        "accuracy",
        "macro_f1",
        "weighted_f1",
        "macro_precision",
        "macro_recall",
        "roc_auc_ovr",
    ],
    "classifier_config": {
        model_name: {
            "parameters": FROZEN_MODEL_CONFIG[model_name].get(
                "parameters", {}
            )
        }
        for model_name in ACTIVE_MODELS
    },
    "held_out_real_test_used_for_training": False,
    "preprocessor_signature": str(inspect.signature(build_preprocessor)),
}

with open(CONFIG_PATH, "w", encoding="utf-8") as file:
    json.dump(run_configuration, file, indent=2, default=str)

print("\nPreflight saved:", PREFLIGHT_PATH)
print("Run configuration saved:", CONFIG_PATH)

# ==========================================================================================
# 10. METRIC CALCULATION
# ==========================================================================================

def calculate_metrics(y_true, y_pred, y_score, n_classes):
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(
            f1_score(y_true, y_pred, average="macro", zero_division=0)
        ),
        "weighted_f1": float(
            f1_score(y_true, y_pred, average="weighted", zero_division=0)
        ),
        "macro_precision": float(
            precision_score(
                y_true, y_pred, average="macro", zero_division=0
            )
        ),
        "macro_recall": float(
            recall_score(
                y_true, y_pred, average="macro", zero_division=0
            )
        ),
        "roc_auc_ovr": np.nan,
    }

    try:
        if y_score is not None and len(np.unique(y_true)) == n_classes:
            if n_classes == 2:
                metrics["roc_auc_ovr"] = float(
                    roc_auc_score(y_true, y_score[:, 1])
                )
            else:
                metrics["roc_auc_ovr"] = float(
                    roc_auc_score(
                        y_true,
                        y_score,
                        labels=np.arange(n_classes),
                        multi_class="ovr",
                        average="macro",
                    )
                )
    except (ValueError, IndexError):
        metrics["roc_auc_ovr"] = np.nan

    return metrics


# ==========================================================================================
# 11. RUN TRTR AND TSTR
# ==========================================================================================

print("\n" + "=" * 100)
print("SECTION 10 — RUN TRTR / TSTR EVALUATIONS")
print("=" * 100)

result_rows = []
run_log_rows = []

evaluation_sources = {
    "TRTR": "real",
    "TSTR": "synthetic",
}

expected_runs = (
    len(dataset_cache)
    * len(ACTIVE_MODELS)
    * len(REPETITION_SEEDS)
    * len(evaluation_sources)
)

print("Datasets:", len(dataset_cache))
print("Models:", ACTIVE_MODELS)
print("Seeds:", REPETITION_SEEDS)
print("Expected runs:", expected_runs)

for dataset_id, cached in dataset_cache.items():

    X_real_train = cached["X_real_train"]
    X_real_test = cached["X_real_test"]
    X_synthetic_train = cached["X_synthetic_train"]

    y_real_train = cached["y_real_train"]
    y_real_test = cached["y_real_test"]
    y_synthetic_train = cached["y_synthetic_train"]

    n_classes = cached["n_classes"]

    for model_id in ACTIVE_MODELS:
        for seed in REPETITION_SEEDS:
            for evaluation_mode, source_name in evaluation_sources.items():

                run_start = time.perf_counter()
                run_status = "FAILED"
                error_message = ""
                metrics = {}

                X_train = (
                    X_real_train
                    if source_name == "real"
                    else X_synthetic_train
                )
                y_train = (
                    y_real_train
                    if source_name == "real"
                    else y_synthetic_train
                )

                try:
                    # Crucial fix:
                    # build_preprocessor has the signature build_preprocessor(X_train).
                    # Pass only this run's training predictors.
                    preprocessor = create_preprocessor(X_train)

                    classifier = build_classifier(
                        model_name=model_id,
                        seed=seed,
                        n_classes=n_classes,
                    )

                    model = Pipeline([
                        ("preprocessor", preprocessor),
                        ("classifier", classifier),
                    ])

                    # Fit exclusively on the designated training source.
                    model.fit(X_train, y_train)

                    # Held-out real test data is used only for evaluation.
                    y_pred = model.predict(X_real_test)

                    try:
                        y_score = model.predict_proba(X_real_test)
                    except (
                        AttributeError,
                        NotImplementedError,
                        ValueError,
                    ):
                        y_score = None

                    metrics = calculate_metrics(
                        y_true=y_real_test,
                        y_pred=y_pred,
                        y_score=y_score,
                        n_classes=n_classes,
                    )

                    run_status = "SUCCESS"

                    result_rows.append({
                        "dataset": dataset_id,
                        "model": model_id,
                        "evaluation": evaluation_mode,
                        "training_source": source_name,
                        "seed": seed,
                        "n_classes": n_classes,
                        "n_train": len(X_train),
                        "n_test": len(X_real_test),
                        "n_predictors_raw": X_train.shape[1],
                        **metrics,
                        "status": run_status,
                        "error": "",
                    })

                except Exception as exc:
                    error_message = f"{type(exc).__name__}: {exc}"

                    result_rows.append({
                        "dataset": dataset_id,
                        "model": model_id,
                        "evaluation": evaluation_mode,
                        "training_source": source_name,
                        "seed": seed,
                        "n_classes": n_classes,
                        "n_train": len(X_train),
                        "n_test": len(X_real_test),
                        "n_predictors_raw": X_train.shape[1],
                        "accuracy": np.nan,
                        "macro_f1": np.nan,
                        "weighted_f1": np.nan,
                        "macro_precision": np.nan,
                        "macro_recall": np.nan,
                        "roc_auc_ovr": np.nan,
                        "status": run_status,
                        "error": error_message,
                    })

                    print(
                        f"FAILED | {dataset_id} | {model_id} | "
                        f"{evaluation_mode} | seed={seed} | {error_message}"
                    )

                elapsed = time.perf_counter() - run_start

                run_log_rows.append({
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                    "dataset": dataset_id,
                    "model": model_id,
                    "evaluation": evaluation_mode,
                    "training_source": source_name,
                    "seed": seed,
                    "status": run_status,
                    "runtime_seconds": float(elapsed),
                    "error": error_message,
                })

                # Persist after every run.
                pd.DataFrame(result_rows).to_csv(
                    RESULTS_PATH, index=False
                )
                pd.DataFrame(run_log_rows).to_csv(
                    RUN_LOG_PATH, index=False
                )

                print(
                    f"{run_status:7s} | {dataset_id:18s} | "
                    f"{model_id:20s} | {evaluation_mode} | "
                    f"seed={seed} | {elapsed:.2f}s"
                )

# ==========================================================================================
# 12. FINAL RESULTS AND SUMMARY
# ==========================================================================================

results_df = pd.DataFrame(result_rows)
run_log_df = pd.DataFrame(run_log_rows)

if results_df.empty:
    raise RuntimeError("Section 10 produced no result rows.")

successful_df = results_df.loc[
    results_df["status"].eq("SUCCESS")
].copy()

metric_columns = [
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "macro_precision",
    "macro_recall",
    "roc_auc_ovr",
]

if not successful_df.empty:
    summary_df = (
        successful_df
        .groupby(
            ["dataset", "model", "evaluation", "training_source"],
            dropna=False,
        )[metric_columns]
        .agg(["mean", "std", "count"])
    )

    summary_df.columns = [
        f"{metric}_{statistic}"
        for metric, statistic in summary_df.columns
    ]
    summary_df = summary_df.reset_index()
else:
    summary_df = pd.DataFrame(
        columns=[
            "dataset", "model", "evaluation", "training_source"
        ]
    )

summary_df.to_csv(SUMMARY_PATH, index=False)
results_df.to_csv(RESULTS_PATH, index=False)
run_log_df.to_csv(RUN_LOG_PATH, index=False)

# ==========================================================================================
# 13. FINAL STATUS
# ==========================================================================================

successful_runs = int(results_df["status"].eq("SUCCESS").sum())
failed_runs = int(results_df["status"].eq("FAILED").sum())
actual_runs = len(results_df)

print("\n" + "=" * 100)
print("SECTION 10 — FINAL STATUS")
print("=" * 100)

print(f"Expected runs       : {expected_runs}")
print(f"Completed runs      : {actual_runs}")
print(f"Successful runs     : {successful_runs}")
print(f"Failed runs         : {failed_runs}")
print(
    f"Success rate        : "
    f"{100 * successful_runs / actual_runs:.2f}%"
    if actual_runs
    else "Success rate        : N/A"
)

print("\nSaved artifacts:")
print("Run log       :", RUN_LOG_PATH)
print("Run results   :", RESULTS_PATH)
print("Summary       :", SUMMARY_PATH)
print("Dataset check :", PREFLIGHT_PATH)
print("Configuration :", CONFIG_PATH)

if not successful_df.empty:
    display_columns = [
        "dataset",
        "model",
        "evaluation",
        "accuracy",
        "macro_f1",
        "weighted_f1",
        "roc_auc_ovr",
        "status",
    ]

    print("\nRun-level results:")
    display(results_df[display_columns])

    print("\nMean metrics by dataset, model, and evaluation:")
    display(summary_df)

if failed_runs:
    print(
        "\nWARNING: Some runs failed. Inspect the run log and error "
        "messages before interpreting results."
    )
else:
    print(
        "\nAll Section 10 runs completed successfully. "
        "Verify the saved configuration against the approved protocol."
    )

print("=" * 100)


SECTION 10 — RESEARCH GATE VALIDATION
PASS | original_row_id_role_verified
PASS | predictor_schema_verified_against_upstream_artifacts
PASS | missing_value_preprocessing_finalized
PASS | target_and_class_coverage_validated
PASS | classifier_configuration_frozen
PASS | real_test_split_reserved_for_final_evaluation

PASS | Preprocessor adapter smoke test
Dataset: adult_income
Upstream signature: (X_train)
Returned object: ColumnTransformer

SECTION 10 — DATASET PREFLIGHT
PASS | adult_income | real_train=(34189, 14) | synthetic_train=(34189, 14) | real_test=(7327, 14) | classes=2
PASS | bank_marketing | real_train=(31647, 16) | synthetic_train=(31647, 16) | real_test=(6782, 16) | classes=2


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


PASS | diabetes_130us | real_train=(71236, 47) | synthetic_train=(71236, 47) | real_test=(15265, 47) | classes=3

Preflight saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/validation/section10_dataset_preflight.csv
Run configuration saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/validation/section10_run_configuration.json

SECTION 10 — RUN TRTR / TSTR EVALUATIONS
Datasets: 3
Models: ['logistic_regression', 'random_forest', 'xgboost']
Seeds: [3026, 3027, 3028, 3029, 3030]
Expected runs: 90


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

SUCCESS | adult_income       | logistic_regression  | TRTR | seed=3026 | 1.91s


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


... (179 outputs left out)


Mean metrics by dataset, model, and evaluation:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,dataset,model,evaluation,training_source,accuracy_mean,accuracy_std,accuracy_count,macro_f1_mean,macro_f1_std,macro_f1_count,...,weighted_f1_count,macro_precision_mean,macro_precision_std,macro_precision_count,macro_recall_mean,macro_recall_std,macro_recall_count,roc_auc_ovr_mean,roc_auc_ovr_std,roc_auc_ovr_count
0,adult_income,logistic_regression,TRTR,real,0.798417,0.000000,5,0.761460,0.000000,5,...,5,0.746537,0.000000,5,0.814719,0.000000,5,0.901956,0.000000,5
1,adult_income,logistic_regression,TSTR,synthetic,0.847414,0.000000,5,0.778885,0.000000,5,...,5,0.797202,0.000000,5,0.765193,0.000000,5,0.901804,0.000000,5
2,adult_income,random_forest,TRTR,real,0.811710,0.001433,5,0.775817,0.001414,5,...,5,0.759164,0.001214,5,0.828305,0.001088,5,0.912664,0.000288,5
3,adult_income,random_forest,TSTR,synthetic,0.837751,0.000881,5,0.766984,0.001346,5,...,5,0.781139,0.001368,5,0.755948,0.001531,5,0.889304,0.000659,5
4,adult_income,xgboost,TRTR,real,0.871926,0.001200,5,0.813034,0.001592,5,...,5,0.836908,0.002128,5,0.795616,0.001298,5,0.926169,0.000216,5
5,adult_income,xgboost,TSTR,synthetic,0.846158,0.000311,5,0.768780,0.000448,5,...,5,0.802700,0.000631,5,0.747866,0.000477,5,0.897875,0.000326,5
6,bank_marketing,logistic_regression,TRTR,real,0.847685,0.000000,5,0.730419,0.000000,5,...,5,0.695705,0.000000,5,0.828969,0.000000,5,0.909904,0.000000,5
7,bank_marketing,logistic_regression,TSTR,synthetic,0.831466,0.000000,5,0.717553,0.000000,5,...,5,0.685528,0.000000,5,0.835102,0.000000,5,0.909546,0.000000,5
8,bank_marketing,random_forest,TRTR,real,0.897169,0.000798,5,0.779999,0.001510,5,...,5,0.753370,0.001470,5,0.818476,0.002369,5,0.930573,0.000740,5
9,bank_marketing,random_forest,TSTR,synthetic,0.837128,0.000648,5,0.718312,0.001672,5,...,5,0.685766,0.001356,5,0.822116,0.003366,5,0.896115,0.000783,5



All Section 10 runs completed successfully. Verify the saved configuration against the approved protocol.


In [91]:

# ==================================================================================================
# SECTION 11 — ACCURACY
# Source of truth: persisted Section 10 TRTR/TSTR results
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 100)
print("SECTION 11 — ACCURACY")
print("=" * 100)

# 1. Define paths
NB15_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15"
)

input_path = NB15_ROOT / "results" / "tstr_trtr_results.csv"
output_dir = NB15_ROOT / "results"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "accuracy_results.csv"

# 2. Verify persisted Section 10 results
if not input_path.exists():
    raise FileNotFoundError(
        f"Section 10 results not found: {input_path}. "
        "Check the Notebook 15 root and do not rerun training automatically."
    )

source_df = pd.read_csv(input_path)

if source_df.empty:
    raise ValueError(
        "The persisted Section 10 results file is empty. "
        "Accuracy reporting cannot proceed."
    )

required_source_columns = [
    "dataset",
    "model",
    "evaluation",
    "seed",
    "accuracy",
    "status",
]

missing = [
    column for column in required_source_columns
    if column not in source_df.columns
]

if missing:
    raise ValueError(
        f"Section 10 results are missing required columns: {missing}"
    )

# 3. Keep successful evaluations only
successful = source_df.loc[
    source_df["status"].astype(str).str.strip().str.upper().eq("SUCCESS")
].copy()

if successful.empty:
    raise ValueError(
        "No successful evaluations were found in the persisted results."
    )

# 4. Normalize to the Section 11 schema
accuracy_results = successful.rename(
    columns={
        "dataset": "dataset_id",
        "model": "model_id",
        "evaluation": "protocol",
    }
)[
    [
        "dataset_id",
        "protocol",
        "model_id",
        "seed",
        "accuracy",
    ]
].copy()

accuracy_results["accuracy"] = pd.to_numeric(
    accuracy_results["accuracy"],
    errors="coerce",
)

if accuracy_results["accuracy"].isna().any():
    raise ValueError("Successful evaluations contain missing accuracy values.")

if not np.isfinite(accuracy_results["accuracy"]).all():
    raise ValueError("Accuracy contains non-finite values.")

if not accuracy_results["accuracy"].between(0, 1).all():
    raise ValueError("Accuracy values must be between 0 and 1.")

# Derive repetition index from the frozen five-seed protocol.
seed_order = [3026, 3027, 3028, 3029, 3030]
seed_to_repetition = {
    seed: index for index, seed in enumerate(seed_order, start=1)
}

accuracy_results["seed"] = pd.to_numeric(
    accuracy_results["seed"], errors="raise"
).astype(int)

unexpected_seeds = sorted(
    set(accuracy_results["seed"]) - set(seed_to_repetition)
)

if unexpected_seeds:
    raise ValueError(
        f"Unexpected seeds found; verify the approved protocol: {unexpected_seeds}"
    )

accuracy_results["repetition"] = accuracy_results["seed"].map(
    seed_to_repetition
)

accuracy_results = accuracy_results[
    [
        "dataset_id",
        "protocol",
        "model_id",
        "repetition",
        "seed",
        "accuracy",
    ]
].sort_values(
    ["dataset_id", "model_id", "protocol", "repetition"]
).reset_index(drop=True)

# 5. Validate expected evaluation coverage
expected_rows = 3 * 3 * 5 * 2  # datasets × models × seeds × protocols

if len(accuracy_results) != expected_rows:
    raise ValueError(
        f"Expected {expected_rows} successful evaluation rows, "
        f"but found {len(accuracy_results)}. "
        "Inspect Section 10 results before treating this report as complete."
    )

duplicate_keys = accuracy_results.duplicated(
    subset=["dataset_id", "model_id", "protocol", "seed"],
    keep=False,
)

if duplicate_keys.any():
    raise ValueError(
        "Duplicate dataset/model/protocol/seed combinations were found."
    )

# 6. Save the accuracy report
accuracy_results.to_csv(output_path, index=False)

print("STATUS: PASS")
print(f"Successful accuracy records: {len(accuracy_results)} / {expected_rows}")
print(f"Datasets: {accuracy_results['dataset_id'].nunique()}")
print(f"Models: {accuracy_results['model_id'].nunique()}")
print(f"Protocols: {sorted(accuracy_results['protocol'].unique().tolist())}")
print(f"Saved to: {output_path}")

print("\nAccuracy results by dataset, model, and protocol:")
display(
    accuracy_results.groupby(
        ["dataset_id", "model_id", "protocol"],
        as_index=False
    ).agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        repetitions=("accuracy", "count"),
    ).round(4)
)

print("=" * 100)


SECTION 11 — ACCURACY
STATUS: PASS
Successful accuracy records: 90 / 90
Datasets: 3
Models: 3
Protocols: ['TRTR', 'TSTR']
Saved to: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/results/accuracy_results.csv

Accuracy results by dataset, model, and protocol:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,dataset_id,model_id,protocol,accuracy_mean,accuracy_std,repetitions
0,adult_income,logistic_regression,TRTR,0.7984,0.0000,5
1,adult_income,logistic_regression,TSTR,0.8474,0.0000,5
2,adult_income,random_forest,TRTR,0.8117,0.0014,5
3,adult_income,random_forest,TSTR,0.8378,0.0009,5
4,adult_income,xgboost,TRTR,0.8719,0.0012,5
5,adult_income,xgboost,TSTR,0.8462,0.0003,5
6,bank_marketing,logistic_regression,TRTR,0.8477,0.0000,5
7,bank_marketing,logistic_regression,TSTR,0.8315,0.0000,5
8,bank_marketing,random_forest,TRTR,0.8972,0.0008,5
9,bank_marketing,random_forest,TSTR,0.8371,0.0006,5


In [93]:

# ==================================================================================================
# SECTION 12 — MACRO-F1
# ==================================================================================================

from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 12 — MACRO-F1")
print("=" * 100)

# -----------------------------------------------------------------------------------------------
# 1. Load authoritative Section 10 results
# -----------------------------------------------------------------------------------------------

RESULTS_PATH = NB15_ROOT / "results" / "tstr_trtr_results.csv"
OUTPUT_PATH = NB15_ROOT / "results" / "macro_f1_results.csv"

if not RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"Section 10 results not found: {RESULTS_PATH}"
    )

section10_df = pd.read_csv(RESULTS_PATH)

required_columns = {
    "dataset", "evaluation", "model", "seed",
    "macro_f1", "status"
}

missing_columns = required_columns - set(section10_df.columns)

if missing_columns:
    raise ValueError(
        f"Section 10 results are missing required columns: "
        f"{sorted(missing_columns)}"
    )

# -----------------------------------------------------------------------------------------------
# 2. Keep successful runs and normalize the schema
# -----------------------------------------------------------------------------------------------

successful_df = section10_df[
    section10_df["status"].astype(str).str.upper().eq("SUCCESS")
].copy()

if successful_df.empty:
    raise RuntimeError(
        "No successful runs found in the persisted Section 10 results."
    )

macro_f1_results = successful_df.rename(
    columns={
        "dataset": "dataset_id",
        "evaluation": "protocol",
        "model": "model_id"
    }
)[
    [
        "dataset_id",
        "protocol",
        "model_id",
        "seed",
        "macro_f1"
    ]
].copy()

# Map the five configured seeds to repetition numbers.
seed_to_repetition = {
    3026: 1,
    3027: 2,
    3028: 3,
    3029: 4,
    3030: 5
}

macro_f1_results["seed"] = pd.to_numeric(
    macro_f1_results["seed"], errors="coerce"
)

macro_f1_results["repetition"] = (
    macro_f1_results["seed"].map(seed_to_repetition)
)

macro_f1_results["macro_f1"] = pd.to_numeric(
    macro_f1_results["macro_f1"], errors="coerce"
)

# Standardize protocol labels.
macro_f1_results["protocol"] = (
    macro_f1_results["protocol"]
    .astype(str)
    .str.upper()
)

# -----------------------------------------------------------------------------------------------
# 3. Validate results
# -----------------------------------------------------------------------------------------------

if macro_f1_results["repetition"].isna().any():
    invalid_seeds = sorted(
        macro_f1_results.loc[
            macro_f1_results["repetition"].isna(), "seed"
        ].dropna().unique().tolist()
    )
    raise ValueError(f"Unexpected or unmapped seeds: {invalid_seeds}")

if macro_f1_results["macro_f1"].isna().any():
    raise ValueError("Macro-F1 contains missing or non-numeric values.")

if not np.isfinite(macro_f1_results["macro_f1"]).all():
    raise ValueError("Macro-F1 contains non-finite values.")

if not macro_f1_results["macro_f1"].between(0, 1).all():
    raise ValueError("Macro-F1 values must be between 0 and 1.")

if not macro_f1_results["protocol"].isin(["TRTR", "TSTR"]).all():
    raise ValueError("Unexpected protocol labels detected.")

key_columns = [
    "dataset_id", "protocol", "model_id", "seed"
]

if macro_f1_results.duplicated(key_columns).any():
    raise ValueError(
        "Duplicate dataset/protocol/model/seed records detected."
    )

# Sort for consistent output.
macro_f1_results = macro_f1_results.sort_values(
    ["dataset_id", "model_id", "protocol", "repetition"]
).reset_index(drop=True)

# -----------------------------------------------------------------------------------------------
# 4. Save validated Macro-F1 results
# -----------------------------------------------------------------------------------------------

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

macro_f1_results = macro_f1_results[
    [
        "dataset_id",
        "protocol",
        "model_id",
        "repetition",
        "seed",
        "macro_f1"
    ]
]

macro_f1_results.to_csv(OUTPUT_PATH, index=False)

# -----------------------------------------------------------------------------------------------
# 5. Report status and summary
# -----------------------------------------------------------------------------------------------

print("\nSTATUS: PASS")
print(f"Successful Macro-F1 records: {len(macro_f1_results)}")
print(f"Datasets: {macro_f1_results['dataset_id'].nunique()}")
print(f"Models: {macro_f1_results['model_id'].nunique()}")
print(f"Protocols: {sorted(macro_f1_results['protocol'].unique())}")
print(f"Saved to: {OUTPUT_PATH}")

print("\nFIRST 10 RECORDS")
print(macro_f1_results.head(10).to_string(index=False))

macro_f1_summary = (
    macro_f1_results
    .groupby(["dataset_id", "model_id", "protocol"], as_index=False)
    .agg(
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        repetitions=("macro_f1", "count")
    )
)

summary_path = NB15_ROOT / "results" / "macro_f1_summary.csv"
macro_f1_summary.to_csv(summary_path, index=False)

print("\nMACRO-F1 SUMMARY")
print(macro_f1_summary.to_string(index=False))
print(f"\nSummary saved to: {summary_path}")

expected_rows = 90
if len(macro_f1_results) != expected_rows:
    print(
        f"\nWARNING: Expected {expected_rows} records, "
        f"but found {len(macro_f1_results)}. "
        "Inspect Section 10 results before interpreting the summary."
    )


SECTION 12 — MACRO-F1

STATUS: PASS
Successful Macro-F1 records: 90
Datasets: 3
Models: 3
Protocols: ['TRTR', 'TSTR']
Saved to: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/results/macro_f1_results.csv

FIRST 10 RECORDS
  dataset_id protocol            model_id  repetition  seed  macro_f1
adult_income     TRTR logistic_regression           1  3026  0.761460
adult_income     TRTR logistic_regression           2  3027  0.761460
adult_income     TRTR logistic_regression           3  3028  0.761460
adult_income     TRTR logistic_regression           4  3029  0.761460
adult_income     TRTR logistic_regression           5  3030  0.761460
adult_income     TSTR logistic_regression           1  3026  0.778885
adult_income     TSTR logistic_regression           2  3027  0.778885
adult_income     TSTR logistic_regression           3  3028  0.778885
adult_income     TSTR logistic_regression           4  3029  0.778885
adult_income     TSTR logistic_regression           5  

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [95]:

# ==================================================================================================
# SECTION 13 — WEIGHTED-F1
# ==================================================================================================

import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 13 — WEIGHTED-F1")
print("=" * 100)

# -----------------------------------------------------------------------------------------------
# 1. Load authoritative Section 10 results
# -----------------------------------------------------------------------------------------------

RESULTS_PATH = NB15_ROOT / "results" / "tstr_trtr_results.csv"
OUTPUT_PATH = NB15_ROOT / "results" / "weighted_f1_results.csv"
SUMMARY_PATH = NB15_ROOT / "results" / "weighted_f1_summary.csv"

if not RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"Section 10 results not found: {RESULTS_PATH}"
    )

section10_df = pd.read_csv(RESULTS_PATH)

required_columns = {
    "dataset", "evaluation", "model", "seed", "weighted_f1", "status"
}

missing_columns = required_columns - set(section10_df.columns)
if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

# -----------------------------------------------------------------------------------------------
# 2. Keep successful runs and normalize column names
# -----------------------------------------------------------------------------------------------

successful_df = section10_df[
    section10_df["status"].astype(str).str.upper().eq("SUCCESS")
].copy()

if successful_df.empty:
    raise RuntimeError("No successful Section 10 runs were found.")

weighted_f1_results = successful_df.rename(
    columns={
        "dataset": "dataset_id",
        "evaluation": "protocol",
        "model": "model_id"
    }
)[
    ["dataset_id", "protocol", "model_id", "seed", "weighted_f1"]
].copy()

weighted_f1_results["seed"] = pd.to_numeric(
    weighted_f1_results["seed"], errors="coerce"
)
weighted_f1_results["weighted_f1"] = pd.to_numeric(
    weighted_f1_results["weighted_f1"], errors="coerce"
)

seed_to_repetition = {
    3026: 1,
    3027: 2,
    3028: 3,
    3029: 4,
    3030: 5
}

weighted_f1_results["repetition"] = (
    weighted_f1_results["seed"].map(seed_to_repetition)
)
weighted_f1_results["protocol"] = (
    weighted_f1_results["protocol"].astype(str).str.upper()
)

# Arrange columns consistently with Sections 11 and 12.
weighted_f1_results = weighted_f1_results[
    [
        "dataset_id", "protocol", "model_id",
        "repetition", "seed", "weighted_f1"
    ]
]

# -----------------------------------------------------------------------------------------------
# 3. Validate results
# -----------------------------------------------------------------------------------------------

if weighted_f1_results["repetition"].isna().any():
    raise ValueError("One or more seeds could not be mapped to repetitions.")

if weighted_f1_results["weighted_f1"].isna().any():
    raise ValueError("Weighted-F1 contains missing or non-numeric values.")

if not np.isfinite(weighted_f1_results["weighted_f1"]).all():
    raise ValueError("Weighted-F1 contains non-finite values.")

if not weighted_f1_results["weighted_f1"].between(0, 1).all():
    raise ValueError("Weighted-F1 values must be between 0 and 1.")

if not weighted_f1_results["protocol"].isin(["TRTR", "TSTR"]).all():
    raise ValueError("Unexpected protocol labels detected.")

key_columns = ["dataset_id", "protocol", "model_id", "seed"]
if weighted_f1_results.duplicated(key_columns).any():
    raise ValueError("Duplicate dataset/model/protocol/seed records detected.")

weighted_f1_results = weighted_f1_results.sort_values(
    ["dataset_id", "model_id", "protocol", "repetition"]
).reset_index(drop=True)

# -----------------------------------------------------------------------------------------------
# 4. Save run-level results
# -----------------------------------------------------------------------------------------------

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
weighted_f1_results.to_csv(OUTPUT_PATH, index=False)

# -----------------------------------------------------------------------------------------------
# 5. Generate and save summary
# -----------------------------------------------------------------------------------------------

weighted_f1_summary = (
    weighted_f1_results
    .groupby(["dataset_id", "model_id", "protocol"], as_index=False)
    .agg(
        weighted_f1_mean=("weighted_f1", "mean"),
        weighted_f1_std=("weighted_f1", "std"),
        repetitions=("weighted_f1", "count")
    )
)

weighted_f1_summary.to_csv(SUMMARY_PATH, index=False)

# -----------------------------------------------------------------------------------------------
# 6. Report status
# -----------------------------------------------------------------------------------------------

print("\nSTATUS: PASS")
print(f"Successful Weighted-F1 records: {len(weighted_f1_results)}")
print(f"Datasets: {weighted_f1_results['dataset_id'].nunique()}")
print(f"Models: {weighted_f1_results['model_id'].nunique()}")
print(f"Protocols: {sorted(weighted_f1_results['protocol'].unique())}")
print(f"Saved to: {OUTPUT_PATH}")

print("\nFIRST 10 RECORDS")
print(weighted_f1_results.head(10).to_string(index=False))

print("\nWEIGHTED-F1 SUMMARY")
print(weighted_f1_summary.to_string(index=False))
print(f"\nSummary saved to: {SUMMARY_PATH}")

if len(weighted_f1_results) != 90:
    print(
        "\nWARNING: Expected 90 records. Inspect Section 10 results "
        "before interpreting the summary."
    )


SECTION 13 — WEIGHTED-F1

STATUS: PASS
Successful Weighted-F1 records: 90
Datasets: 3
Models: 3
Protocols: ['TRTR', 'TSTR']
Saved to: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/results/weighted_f1_results.csv

FIRST 10 RECORDS
  dataset_id protocol            model_id  repetition  seed  weighted_f1
adult_income     TRTR logistic_regression           1  3026     0.810424
adult_income     TRTR logistic_regression           2  3027     0.810424
adult_income     TRTR logistic_regression           3  3028     0.810424
adult_income     TRTR logistic_regression           4  3029     0.810424
adult_income     TRTR logistic_regression           5  3030     0.810424
adult_income     TSTR logistic_regression           1  3026     0.843079
adult_income     TSTR logistic_regression           2  3027     0.843079
adult_income     TSTR logistic_regression           3  3028     0.843079
adult_income     TSTR logistic_regression           4  3029     0.843079
adult_income    

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [97]:

# ==================================================================================================
# SECTION 14 — ADDITIONAL METRICS
# ==================================================================================================

import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 14 — ADDITIONAL METRICS")
print("=" * 100)

# -----------------------------------------------------------------------------------------------
# 1. Define paths and load authoritative Section 10 results
# -----------------------------------------------------------------------------------------------

RESULTS_PATH = NB15_ROOT / "results" / "tstr_trtr_results.csv"
OUTPUT_PATH = NB15_ROOT / "results" / "additional_metrics_results.csv"
COVERAGE_PATH = NB15_ROOT / "validation" / "additional_metric_coverage.csv"

if not RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"Section 10 results not found: {RESULTS_PATH}"
    )

section10_df = pd.read_csv(RESULTS_PATH)

# -----------------------------------------------------------------------------------------------
# 2. Validate required source columns
# -----------------------------------------------------------------------------------------------

REQUIRED_COLUMNS = {
    "dataset", "evaluation", "model", "seed", "status",
    "macro_precision", "macro_recall", "roc_auc_ovr"
}

missing_columns = REQUIRED_COLUMNS - set(section10_df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required Section 10 columns: {sorted(missing_columns)}"
    )

# -----------------------------------------------------------------------------------------------
# 3. Select successful runs and normalize schema
# -----------------------------------------------------------------------------------------------

successful_df = section10_df[
    section10_df["status"].astype(str).str.upper().eq("SUCCESS")
].copy()

if successful_df.empty:
    raise RuntimeError(
        "No successful runs found in the persisted Section 10 results."
    )

additional_results = successful_df.rename(
    columns={
        "dataset": "dataset_id",
        "evaluation": "protocol",
        "model": "model_id",
        "macro_precision": "precision_macro",
        "macro_recall": "recall_macro"
    })[
        [
            "dataset_id", "protocol", "model_id", "seed",
            "precision_macro", "recall_macro", "roc_auc_ovr"
        ]
    ].copy()

additional_results["protocol"] = (
    additional_results["protocol"].astype(str).str.upper()
)

additional_results["seed"] = pd.to_numeric(
    additional_results["seed"], errors="coerce"
)

seed_to_repetition = {
    3026: 1,
    3027: 2,
    3028: 3,
    3029: 4,
    3030: 5
}

additional_results["repetition"] = (
    additional_results["seed"].map(seed_to_repetition)
)

if additional_results["repetition"].isna().any():
    raise ValueError("One or more seeds could not be mapped to repetitions.")

if not additional_results["protocol"].isin(["TRTR", "TSTR"]).all():
    raise ValueError("Unexpected protocol labels detected.")

# Keep the same column order used by previous sections.
ADDITIONAL_METRICS = [
    "precision_macro",
    "recall_macro",
    "roc_auc_ovr"
]

additional_results = additional_results[
    [
        "dataset_id", "protocol", "model_id",
        "repetition", "seed"
    ] + ADDITIONAL_METRICS
]

# -----------------------------------------------------------------------------------------------
# 4. Validate defined metric values
# -----------------------------------------------------------------------------------------------

for metric in ADDITIONAL_METRICS:
    additional_results[metric] = pd.to_numeric(
        additional_results[metric], errors="coerce"
    )

    finite_values = additional_results[metric].dropna()

    if not np.isfinite(finite_values).all():
        raise ValueError(f"{metric} contains infinite values.")

    if not finite_values.between(0, 1).all():
        raise ValueError(
            f"{metric} contains defined values outside [0, 1]."
        )

key_columns = ["dataset_id", "protocol", "model_id", "seed"]

if additional_results.duplicated(key_columns).any():
    raise ValueError(
        "Duplicate dataset/protocol/model/seed records detected."
    )

additional_results = additional_results.sort_values(
    ["dataset_id", "model_id", "protocol", "repetition"]
).reset_index(drop=True)

# -----------------------------------------------------------------------------------------------
# 5. Calculate metric coverage
# -----------------------------------------------------------------------------------------------

applicable = len(additional_results)
additional_coverage = []

for metric in ADDITIONAL_METRICS:
    valid_mask = (
        additional_results[metric].notna()
        & np.isfinite(additional_results[metric])
    )

    valid = int(valid_mask.sum())

    additional_coverage.append({
        "metric": metric,
        "successful_model_runs": applicable,
        "valid_values": valid,
        "missing_or_undefined": applicable - valid,
        "coverage_rate": valid / applicable if applicable else np.nan
    })

ADDITIONAL_METRIC_COVERAGE_DF = pd.DataFrame(additional_coverage)

# -----------------------------------------------------------------------------------------------
# 6. Save results and coverage report
# -----------------------------------------------------------------------------------------------

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
COVERAGE_PATH.parent.mkdir(parents=True, exist_ok=True)

additional_results.to_csv(OUTPUT_PATH, index=False)

ADDITIONAL_METRIC_COVERAGE_DF.to_csv(
    COVERAGE_PATH, index=False
)

# -----------------------------------------------------------------------------------------------
# 7. Report status
# -----------------------------------------------------------------------------------------------

print("\nSTATUS: PASS")
print(f"Successful model runs: {applicable}")
print(f"Datasets: {additional_results['dataset_id'].nunique()}")
print(f"Models: {additional_results['model_id'].nunique()}")
print(f"Protocols: {sorted(additional_results['protocol'].unique())}")

print("\nADDITIONAL METRIC COVERAGE")
print(ADDITIONAL_METRIC_COVERAGE_DF.to_string(index=False))

print("\nFIRST 10 RECORDS")
print(additional_results.head(10).to_string(index=False))

print(f"\nResults saved to: {OUTPUT_PATH}")
print(f"Coverage report saved to: {COVERAGE_PATH}")

if applicable != 90:
    print(
        f"\nWARNING: Expected 90 successful runs but found {applicable}. "
        "Inspect Section 10 results before publication."
    )


SECTION 14 — ADDITIONAL METRICS

STATUS: PASS
Successful model runs: 90
Datasets: 3
Models: 3
Protocols: ['TRTR', 'TSTR']

ADDITIONAL METRIC COVERAGE
         metric  successful_model_runs  valid_values  missing_or_undefined  coverage_rate
precision_macro                     90            90                     0            1.0
   recall_macro                     90            90                     0            1.0
    roc_auc_ovr                     90            90                     0            1.0

FIRST 10 RECORDS
  dataset_id protocol            model_id  repetition  seed  precision_macro  recall_macro  roc_auc_ovr
adult_income     TRTR logistic_regression           1  3026         0.746537      0.814719     0.901956
adult_income     TRTR logistic_regression           2  3027         0.746537      0.814719     0.901956
adult_income     TRTR logistic_regression           3  3028         0.746537      0.814719     0.901956
adult_income     TRTR logistic_regression           4  3

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [98]:
# ==================================================================================================
# SECTION 15 — RUNTIME MEASUREMENT
# ==================================================================================================

RUNTIME_DF = RESULTS_DF[[
    "dataset_id",
    "protocol",
    "model_id",
    "repetition",
    "seed",
    "train_rows",
    "test_rows",
    "runtime_seconds",
    "status",
]].copy()

RUNTIME_DF.to_csv(
    NB15_ROOT / "results" / "runtime_results.csv", index=False
)

RUNTIME_SUMMARY_DF = (
    RUNTIME_DF.groupby(
        ["dataset_id", "protocol", "model_id"], dropna=False
    )
    .agg(
        runs=("runtime_seconds", "count"),
        mean_runtime_seconds=("runtime_seconds", "mean"),
        median_runtime_seconds=("runtime_seconds", "median"),
        std_runtime_seconds=("runtime_seconds", "std"),
        total_runtime_seconds=("runtime_seconds", "sum"),
    )
    .reset_index()
)

RUNTIME_SUMMARY_DF.to_csv(
    NB15_ROOT / "aggregations" / "runtime_summary.csv", index=False
)

print(RUNTIME_SUMMARY_DF.to_string(index=False))

    dataset_id protocol            model_id  runs  mean_runtime_seconds  median_runtime_seconds  std_runtime_seconds  total_runtime_seconds
  adult_income     TRTR logistic_regression     5              0.013071                0.012811             0.000693               0.065357
  adult_income     TRTR       random_forest     5              0.014617                0.014410             0.000686               0.073087
  adult_income     TRTR             xgboost     5              0.013700                0.013716             0.000606               0.068499
  adult_income     TSTR logistic_regression     5              0.014247                0.014334             0.000935               0.071237
  adult_income     TSTR       random_forest     5              0.014432                0.014273             0.000636               0.072159
  adult_income     TSTR             xgboost     5              0.015397                0.014321             0.001797               0.076983
bank_marketing     T

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [101]:

# ==================================================================================================
# SECTION 16 — REPETITION AGGREGATION
# ==================================================================================================

from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 16 — REPETITION AGGREGATION")
print("=" * 100)

# --------------------------------------------------------------------------------
# 1. Paths and experiment configuration
# --------------------------------------------------------------------------------

NB15_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15"
)

SOURCE_PATH = NB15_ROOT / "results" / "tstr_trtr_results.csv"
RUNTIME_PATH = NB15_ROOT / "results" / "runtime_results.csv"
OUTPUT_DIR = NB15_ROOT / "aggregations"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_REPETITIONS = 5

SEED_TO_REPETITION = {
    3026: 1,
    3027: 2,
    3028: 3,
    3029: 4,
    3030: 5,
}

PRIMARY_METRICS = ["accuracy", "macro_f1", "weighted_f1"]
ADDITIONAL_METRICS = [
    "precision_macro",
    "recall_macro",
    "roc_auc_ovr",
]
METRICS = PRIMARY_METRICS + ADDITIONAL_METRICS

# --------------------------------------------------------------------------------
# 2. Load authoritative predictive results
# --------------------------------------------------------------------------------

if not SOURCE_PATH.exists():
    raise FileNotFoundError(
        f"Section 10 results file not found: {SOURCE_PATH}"
    )

source_df = pd.read_csv(SOURCE_PATH)

required_columns = {
    "dataset", "evaluation", "model", "seed", "status",
    "accuracy", "macro_f1", "weighted_f1",
    "macro_precision", "macro_recall", "roc_auc_ovr",
}

missing_columns = required_columns - set(source_df.columns)
if missing_columns:
    raise ValueError(
        f"Predictive results are missing required columns: "
        f"{sorted(missing_columns)}"
    )

source_df = source_df.loc[
    source_df["status"].astype(str).str.upper().eq("SUCCESS")
].copy()

source_df = source_df.rename(columns={
    "dataset": "dataset_id",
    "evaluation": "protocol",
    "model": "model_id",
    "macro_precision": "precision_macro",
    "macro_recall": "recall_macro",
})

source_df["seed"] = pd.to_numeric(
    source_df["seed"], errors="raise"
).astype(int)

source_df["repetition"] = source_df["seed"].map(SEED_TO_REPETITION)

if source_df["repetition"].isna().any():
    unknown_seeds = sorted(
        source_df.loc[source_df["repetition"].isna(), "seed"].unique()
    )
    raise ValueError(f"Unexpected experiment seeds: {unknown_seeds}")

source_df["repetition"] = source_df["repetition"].astype(int)
source_df["protocol"] = source_df["protocol"].astype(str).str.upper()

for metric in METRICS:
    source_df[metric] = pd.to_numeric(
        source_df[metric], errors="coerce"
    )
    if np.isinf(source_df[metric].dropna()).any():
        raise ValueError(f"Infinite values detected in {metric}.")

for metric in METRICS:
    invalid = source_df[metric].notna() & (
        (source_df[metric] < 0) | (source_df[metric] > 1)
    )
    if invalid.any():
        raise ValueError(
            f"Out-of-range values detected in {metric}."
        )

KEYS = ["dataset_id", "protocol", "model_id", "seed"]

if source_df.duplicated(KEYS).any():
    raise ValueError(
        "Duplicate dataset/protocol/model/seed records found."
    )

# --------------------------------------------------------------------------------
# 3. Load runtime separately when the Section 15 artifact exists
# --------------------------------------------------------------------------------

source_df["runtime_seconds"] = np.nan
runtime_available = False

if RUNTIME_PATH.exists():
    runtime_df = pd.read_csv(RUNTIME_PATH)

    # Accommodate common Section 15 column names.
    runtime_aliases = {
        "dataset": "dataset_id",
        "model": "model_id",
        "evaluation": "protocol",
        "runtime": "runtime_seconds",
        "mean_runtime_seconds": "runtime_seconds",
    }
    runtime_df = runtime_df.rename(columns={
        old: new for old, new in runtime_aliases.items()
        if old in runtime_df.columns
        and new not in runtime_df.columns
    })

    runtime_keys = {"dataset_id", "protocol", "model_id"}
    if runtime_keys.issubset(runtime_df.columns) and (
        "runtime_seconds" in runtime_df.columns
    ):
        runtime_df["protocol"] = (
            runtime_df["protocol"].astype(str).str.upper()
        )
        runtime_df["runtime_seconds"] = pd.to_numeric(
            runtime_df["runtime_seconds"], errors="coerce"
        )

        if (
            runtime_df["runtime_seconds"].dropna().lt(0).any()
            or np.isinf(
                runtime_df["runtime_seconds"].dropna()
            ).any()
        ):
            raise ValueError("Invalid runtime values in Section 15 artifact.")

        # Runtime artifact may contain one mean per group rather than
        # one value per seed. Preserve it as a group-level descriptive value.
        runtime_group = (
            runtime_df.groupby(
                ["dataset_id", "protocol", "model_id"],
                as_index=False
            )["runtime_seconds"]
            .mean()
        )

        source_df = source_df.drop(columns=["runtime_seconds"]).merge(
            runtime_group,
            on=["dataset_id", "protocol", "model_id"],
            how="left",
            validate="many_to_one",
        )
        runtime_available = source_df["runtime_seconds"].notna().any()

        print(
            "Runtime source:",
            RUNTIME_PATH,
            "| usable runtime values:",
            int(source_df["runtime_seconds"].notna().sum()),
        )
    else:
        print(
            "WARNING: Section 15 file exists, but its schema does not "
            "contain dataset_id, protocol, model_id, and runtime_seconds."
        )
else:
    print(
        "WARNING: Section 15 runtime_results.csv not found. "
        "Predictive metrics will be aggregated without runtime."
    )

# --------------------------------------------------------------------------------
# 4. Aggregate repetitions
# --------------------------------------------------------------------------------

aggregation_spec = {}

for metric in METRICS + ["runtime_seconds"]:
    aggregation_spec[f"{metric}_mean"] = (metric, "mean")
    aggregation_spec[f"{metric}_std"] = (metric, "std")
    aggregation_spec[f"{metric}_valid_repetitions"] = (metric, "count")

group_columns = ["dataset_id", "protocol", "model_id"]

REPETITION_AGG_DF = (
    source_df.groupby(group_columns, dropna=False)
    .agg(**aggregation_spec)
    .reset_index()
)

REPETITION_AGG_DF["expected_repetitions"] = N_REPETITIONS
REPETITION_AGG_DF["repetition_coverage"] = (
    REPETITION_AGG_DF["accuracy_valid_repetitions"] / N_REPETITIONS
)
REPETITION_AGG_DF["synthetic_generation_repetitions"] = 1
REPETITION_AGG_DF["repetition_interpretation"] = (
    "Predictive-model seed repetitions on one canonical synthetic dataset"
)

# --------------------------------------------------------------------------------
# 5. Validation
# --------------------------------------------------------------------------------

expected_groups = (
    source_df[group_columns].drop_duplicates().shape[0]
)

if len(REPETITION_AGG_DF) != expected_groups:
    raise ValueError("Unexpected number of aggregation groups.")

if not REPETITION_AGG_DF["accuracy_valid_repetitions"].eq(
    N_REPETITIONS
).all():
    raise ValueError(
        "At least one group does not contain five valid accuracy values."
    )

if len(source_df) != 90 or expected_groups != 18:
    print(
        f"WARNING: Expected 90 successful records and 18 groups; "
        f"found {len(source_df)} records and {expected_groups} groups."
    )

# --------------------------------------------------------------------------------
# 6. Save output
# --------------------------------------------------------------------------------

OUTPUT_PATH = OUTPUT_DIR / "repetition_aggregation.csv"
REPETITION_AGG_DF.to_csv(OUTPUT_PATH, index=False)

print("-" * 100)
print("SECTION 16 VALIDATION SUMMARY")
print("-" * 100)
print("Successful predictive records :", len(source_df))
print("Aggregation groups            :", len(REPETITION_AGG_DF))
print("Expected repetitions/group    :", N_REPETITIONS)
print("Runtime artifact available    :", runtime_available)
print("Saved                         :", OUTPUT_PATH)
print("Status                        : PASS")


SECTION 16 — REPETITION AGGREGATION
Runtime source: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/results/runtime_results.csv | usable runtime values: 90
----------------------------------------------------------------------------------------------------
SECTION 16 VALIDATION SUMMARY
----------------------------------------------------------------------------------------------------
Successful predictive records : 90
Aggregation groups            : 18
Expected repetitions/group    : 5
Runtime artifact available    : True
Saved                         : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/aggregations/repetition_aggregation.csv
Status                        : PASS


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [102]:
# ==================================================================================================
# SECTION 17 — MODEL COMPARISON
# ==================================================================================================

comparison_rows = []

for dataset_id in DATASETS:
    for model_id in ACTIVE_MODELS:
        group = REPETITION_AGG_DF[
            (REPETITION_AGG_DF["dataset_id"] == dataset_id)
            & (REPETITION_AGG_DF["model_id"] == model_id)
        ]

        tstr = group[group["protocol"] == "TSTR"]
        trtr = group[group["protocol"] == "TRTR"]

        if tstr.empty or trtr.empty:
            comparison_rows.append({
                "dataset_id": dataset_id,
                "model_id": model_id,
                "comparison_status": "INCOMPLETE",
            })
            continue

        tstr_row = tstr.iloc[0]
        trtr_row = trtr.iloc[0]

        row = {
            "dataset_id": dataset_id,
            "model_id": model_id,
            "comparison_status": "PASS",
        }

        preservation_scores = []

        for metric in PRIMARY_METRICS:
            tstr_value = tstr_row.get(f"{metric}_mean", np.nan)
            trtr_value = trtr_row.get(f"{metric}_mean", np.nan)

            row[f"tstr_{metric}"] = tstr_value
            row[f"trtr_{metric}"] = trtr_value
            row[f"{metric}_absolute_difference"] = (
                tstr_value - trtr_value
                if pd.notna(tstr_value) and pd.notna(trtr_value)
                else np.nan
            )
            row[f"{metric}_degradation_percent"] = (
                (trtr_value - tstr_value) / trtr_value * 100
                if pd.notna(tstr_value)
                and pd.notna(trtr_value)
                and trtr_value != 0
                else np.nan
            )
            row[f"{metric}_preservation_percent"] = (
                tstr_value / trtr_value * 100
                if pd.notna(tstr_value)
                and pd.notna(trtr_value)
                and trtr_value != 0
                else np.nan
            )

            if pd.notna(tstr_value) and pd.notna(trtr_value) and trtr_value != 0:
                preservation_scores.append(tstr_value / trtr_value)

        row["mean_primary_utility_preservation_ratio"] = (
            float(np.mean(preservation_scores))
            if preservation_scores else np.nan
        )
        row["mean_primary_utility_preservation_percent"] = (
            row["mean_primary_utility_preservation_ratio"] * 100
            if pd.notna(row["mean_primary_utility_preservation_ratio"])
            else np.nan
        )

        comparison_rows.append(row)

MODEL_COMPARISON_DF = pd.DataFrame(comparison_rows)

MODEL_COMPARISON_DF.to_csv(
    NB15_ROOT / "aggregations" / "model_comparison.csv",
    index=False,
)

print(MODEL_COMPARISON_DF.to_string(index=False))

    dataset_id            model_id comparison_status  tstr_accuracy  trtr_accuracy  accuracy_absolute_difference  accuracy_degradation_percent  accuracy_preservation_percent  tstr_macro_f1  trtr_macro_f1  macro_f1_absolute_difference  macro_f1_degradation_percent  macro_f1_preservation_percent  tstr_weighted_f1  trtr_weighted_f1  weighted_f1_absolute_difference  weighted_f1_degradation_percent  weighted_f1_preservation_percent  mean_primary_utility_preservation_ratio  mean_primary_utility_preservation_percent
  adult_income logistic_regression              PASS       0.847414       0.798417                      0.048997                     -6.136752                     106.136752       0.778885       0.761460                      0.017425                     -2.288405                     102.288405          0.843079          0.810424                         0.032655                        -4.029381                        104.029381                                 1.041515              

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [104]:

# ==================================================================================================
# SECTION 18 — SAVE UTILITY RESULTS
# ==================================================================================================

from pathlib import Path
import hashlib
import pandas as pd

print("=" * 100)
print("SECTION 18 — SAVE UTILITY RESULTS")
print("=" * 100)

# --------------------------------------------------------------------------------
# 1. DEFINE CANONICAL PATHS
# --------------------------------------------------------------------------------

NB15_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15"
)

RESULTS_DIR = NB15_ROOT / "results"
LOGS_DIR = NB15_ROOT / "logs"
AGGREGATIONS_DIR = NB15_ROOT / "aggregations"
VALIDATION_DIR = NB15_ROOT / "validation"
INPUTS_DIR = NB15_ROOT / "inputs"
CONFIG_DIR = NB15_ROOT / "config"

# Create required directories without deleting existing artifacts.
for directory in [
    RESULTS_DIR,
    LOGS_DIR,
    AGGREGATIONS_DIR,
    VALIDATION_DIR,
    INPUTS_DIR,
    CONFIG_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------------------
# 2. DEFINE OUTPUT ARTIFACT REGISTRY
# --------------------------------------------------------------------------------

OUTPUT_FILES = {
    "raw_results": RESULTS_DIR / "ml_utility_results.csv",
    "run_log": LOGS_DIR / "model_run_log.csv",
    "accuracy": RESULTS_DIR / "accuracy_results.csv",
    "macro_f1": RESULTS_DIR / "macro_f1_results.csv",
    "weighted_f1": RESULTS_DIR / "weighted_f1_results.csv",
    "additional_metrics": RESULTS_DIR / "additional_metrics_results.csv",
    "runtime": RESULTS_DIR / "runtime_results.csv",
    "runtime_summary": AGGREGATIONS_DIR / "runtime_summary.csv",
    "repetition_aggregation": AGGREGATIONS_DIR / "repetition_aggregation.csv",
    "model_comparison": AGGREGATIONS_DIR / "model_comparison.csv",
    "task_validation": VALIDATION_DIR / "task_validation.csv",
    "real_data_audit": INPUTS_DIR / "real_data_audit.csv",
    "synthetic_data_audit": INPUTS_DIR / "synthetic_data_audit.csv",
    "additional_metric_coverage": VALIDATION_DIR / "additional_metric_coverage.csv",
}

# --------------------------------------------------------------------------------
# 3. SAVE AUTHORITATIVE RAW RESULTS
# --------------------------------------------------------------------------------
# IMPORTANT:
# Use the persisted Section 10 results as the source of truth.
# Do not overwrite successful persisted results with a potentially stale RESULTS_DF.

SOURCE_RESULTS_PATH = RESULTS_DIR / "tstr_trtr_results.csv"
RAW_RESULTS_PATH = OUTPUT_FILES["raw_results"]

if not SOURCE_RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"Authoritative Section 10 results not found: {SOURCE_RESULTS_PATH}"
    )

AUTHORITATIVE_RESULTS_DF = pd.read_csv(SOURCE_RESULTS_PATH)

required_raw_columns = {
    "dataset",
    "model",
    "evaluation",
    "seed",
    "status",
    "accuracy",
    "macro_f1",
    "weighted_f1",
}

missing_raw_columns = required_raw_columns - set(
    AUTHORITATIVE_RESULTS_DF.columns
)

if missing_raw_columns:
    raise ValueError(
        "Authoritative results are missing required columns: "
        f"{sorted(missing_raw_columns)}"
    )

# Validate successful records before saving.
status_series = (
    AUTHORITATIVE_RESULTS_DF["status"]
    .astype(str)
    .str.strip()
    .str.upper()
)

successful_count = int(status_series.eq("SUCCESS").sum())

if len(AUTHORITATIVE_RESULTS_DF) != 90 or successful_count != 90:
    raise ValueError(
        "Unexpected raw-result status. Expected exactly 90 records, "
        f"all successful; found {len(AUTHORITATIVE_RESULTS_DF)} records "
        f"and {successful_count} successful records."
    )

# Validate experiment coverage.
expected_datasets = {
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
}
expected_models = {
    "logistic_regression",
    "random_forest",
    "xgboost",
}
expected_protocols = {"TRTR", "TSTR"}
expected_seeds = {3026, 3027, 3028, 3029, 3030}

actual_datasets = set(
    AUTHORITATIVE_RESULTS_DF["dataset"].dropna().astype(str)
)
actual_models = set(
    AUTHORITATIVE_RESULTS_DF["model"].dropna().astype(str)
)
actual_protocols = set(
    AUTHORITATIVE_RESULTS_DF["evaluation"]
    .dropna()
    .astype(str)
    .str.upper()
)
actual_seeds = set(
    pd.to_numeric(
        AUTHORITATIVE_RESULTS_DF["seed"], errors="raise"
    ).astype(int)
)

if actual_datasets != expected_datasets:
    raise ValueError(
        f"Unexpected dataset coverage: {sorted(actual_datasets)}"
    )

if actual_models != expected_models:
    raise ValueError(
        f"Unexpected model coverage: {sorted(actual_models)}"
    )

if actual_protocols != expected_protocols:
    raise ValueError(
        f"Unexpected protocol coverage: {sorted(actual_protocols)}"
    )

if actual_seeds != expected_seeds:
    raise ValueError(
        f"Unexpected seed coverage: {sorted(actual_seeds)}"
    )

# Each dataset/model/protocol/seed combination should occur once.
duplicate_keys = ["dataset", "model", "evaluation", "seed"]

if AUTHORITATIVE_RESULTS_DF.duplicated(duplicate_keys).any():
    raise ValueError(
        "Duplicate dataset/model/protocol/seed records detected."
    )

# Save a validated copy of the authoritative results.
AUTHORITATIVE_RESULTS_DF.to_csv(RAW_RESULTS_PATH, index=False)

# Verify the saved raw results by reloading them.
SAVED_RAW_RESULTS_DF = pd.read_csv(RAW_RESULTS_PATH)

if len(SAVED_RAW_RESULTS_DF) != 90:
    raise ValueError("Saved raw-results file failed row-count verification.")

if not (
    SAVED_RAW_RESULTS_DF["status"]
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("SUCCESS")
    .all()
):
    raise ValueError("Saved raw-results file contains unsuccessful records.")

print("Authoritative raw results saved:", RAW_RESULTS_PATH)
print("Raw result records:", len(SAVED_RAW_RESULTS_DF))
print("Successful records:", successful_count)

# --------------------------------------------------------------------------------
# 4. VERIFY ALL CONFIGURED OUTPUT ARTIFACTS
# --------------------------------------------------------------------------------

missing_artifacts = []
empty_artifacts = []

for artifact_name, artifact_path in OUTPUT_FILES.items():
    if not artifact_path.exists():
        missing_artifacts.append((artifact_name, artifact_path))
    elif artifact_path.stat().st_size == 0:
        empty_artifacts.append((artifact_name, artifact_path))

if missing_artifacts:
    details = "\n".join(
        f"- {name}: {path}" for name, path in missing_artifacts
    )
    raise FileNotFoundError(
        "Expected output artifacts are missing:\n" + details
    )

if empty_artifacts:
    details = "\n".join(
        f"- {name}: {path}" for name, path in empty_artifacts
    )
    raise ValueError(
        "Expected output artifacts are empty:\n" + details
    )

# --------------------------------------------------------------------------------
# 5. DEFINE CONFIGURATION ARTIFACTS
# --------------------------------------------------------------------------------

CONFIG_FILES = {
    "configuration": CONFIG_PATH,
    "tstr_protocol": CONFIG_DIR / "tstr_protocol.json",
    "trtr_protocol": CONFIG_DIR / "trtr_protocol.json",
    "model_registry": MODEL_REGISTRY_PATH,
}

for artifact_name, artifact_path in CONFIG_FILES.items():
    if not isinstance(artifact_path, Path):
        artifact_path = Path(artifact_path)
        CONFIG_FILES[artifact_name] = artifact_path

    if not artifact_path.exists():
        raise FileNotFoundError(
            f"Required configuration not found ({artifact_name}): "
            f"{artifact_path}"
        )

    if artifact_path.stat().st_size == 0:
        raise ValueError(
            f"Required configuration is empty ({artifact_name}): "
            f"{artifact_path}"
        )

# --------------------------------------------------------------------------------
# 6. DEFINE SHA-256 FINGERPRINT FUNCTION
# --------------------------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    """
    Calculate the SHA-256 digest of a file using bounded memory.
    """
    digest = hashlib.sha256()

    with open(path, "rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()

# --------------------------------------------------------------------------------
# 7. GENERATE FINGERPRINT RECORDS
# --------------------------------------------------------------------------------

FINGERPRINT_ROWS = []

ALL_ARTIFACTS = {
    **OUTPUT_FILES,
    **CONFIG_FILES,
}

for artifact_name, artifact_path in ALL_ARTIFACTS.items():

    if not artifact_path.exists():
        raise FileNotFoundError(
            f"Artifact disappeared during verification: {artifact_path}"
        )

    size_bytes = artifact_path.stat().st_size

    if size_bytes == 0:
        raise ValueError(
            f"Artifact became empty during verification: {artifact_path}"
        )

    FINGERPRINT_ROWS.append({
        "artifact_name": artifact_name,
        "path": str(artifact_path),
        "size_bytes": size_bytes,
        "sha256": sha256_file(artifact_path),
        "status": "PASS",
    })

FINGERPRINT_DF = pd.DataFrame(FINGERPRINT_ROWS)

# --------------------------------------------------------------------------------
# 8. SAVE FINGERPRINT MANIFEST
# --------------------------------------------------------------------------------

FINGERPRINT_PATH = VALIDATION_DIR / "artifact_fingerprints.csv"

FINGERPRINT_DF.to_csv(FINGERPRINT_PATH, index=False)

# Verify the manifest after writing it.
if not FINGERPRINT_PATH.exists() or FINGERPRINT_PATH.stat().st_size == 0:
    raise IOError(
        f"Fingerprint manifest was not saved correctly: {FINGERPRINT_PATH}"
    )

VERIFIED_FINGERPRINT_DF = pd.read_csv(FINGERPRINT_PATH)

expected_fingerprint_count = len(ALL_ARTIFACTS)

if len(VERIFIED_FINGERPRINT_DF) != expected_fingerprint_count:
    raise ValueError(
        f"Expected {expected_fingerprint_count} fingerprint records, "
        f"found {len(VERIFIED_FINGERPRINT_DF)}."
    )

if not VERIFIED_FINGERPRINT_DF["status"].eq("PASS").all():
    raise ValueError("One or more fingerprint records failed verification.")

# --------------------------------------------------------------------------------
# 9. FINAL SUMMARY
# --------------------------------------------------------------------------------

print("-" * 100)
print("SECTION 18 — SAVE AND FINGERPRINT SUMMARY")
print("-" * 100)

print("Configured output artifacts :", len(OUTPUT_FILES))
print("Configuration artifacts     :", len(CONFIG_FILES))
print("Fingerprint records         :", len(VERIFIED_FINGERPRINT_DF))
print("Successful raw records      :", successful_count)
print("Unique datasets             :", len(actual_datasets))
print("Unique models               :", len(actual_models))
print("Evaluation protocols        :", sorted(actual_protocols))
print("Experiment seeds            :", len(actual_seeds))
print("Fingerprint manifest        :", FINGERPRINT_PATH)
print("Save verification           : PASS")
print("Section 18 status           : PASS")


SECTION 18 — SAVE UTILITY RESULTS
Authoritative raw results saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/results/ml_utility_results.csv
Raw result records: 90
Successful records: 90
----------------------------------------------------------------------------------------------------
SECTION 18 — SAVE AND FINGERPRINT SUMMARY
----------------------------------------------------------------------------------------------------
Configured output artifacts : 14
Configuration artifacts     : 4
Fingerprint records         : 18
Successful raw records      : 90
Unique datasets             : 3
Unique models               : 3
Evaluation protocols        : ['TRTR', 'TSTR']
Experiment seeds            : 5
Fingerprint manifest        : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/validation/artifact_fingerprints.csv
Save verification           : PASS
Section 18 status           : PASS


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [106]:

# ==================================================================================================
# SECTION 19 — VALIDATE COVERAGE
# ==================================================================================================

from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 19 — VALIDATE COVERAGE")
print("=" * 100)

# --------------------------------------------------------------------------------
# 1. CANONICAL PATHS AND EXPERIMENT CONFIGURATION
# --------------------------------------------------------------------------------

NB15_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15"
)

RESULTS_DIR = NB15_ROOT / "results"
VALIDATION_DIR = NB15_ROOT / "validation"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_RESULTS_PATH = RESULTS_DIR / "ml_utility_results.csv"
if not SOURCE_RESULTS_PATH.exists():
    SOURCE_RESULTS_PATH = RESULTS_DIR / "tstr_trtr_results.csv"

RUNTIME_PATH = RESULTS_DIR / "runtime_results.csv"
VALIDATION_PATH = VALIDATION_DIR / "utility_coverage_validation.csv"

DATASETS_EXPECTED = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

MODELS_EXPECTED = [
    "logistic_regression",
    "random_forest",
    "xgboost",
]

PROTOCOLS_EXPECTED = ["TSTR", "TRTR"]

SEED_TO_REPETITION = {
    3026: 1,
    3027: 2,
    3028: 3,
    3029: 4,
    3030: 5,
}

N_REPETITIONS = 5

PRIMARY_METRICS = [
    "accuracy",
    "macro_f1",
    "weighted_f1",
]

ADDITIONAL_METRICS = [
    "precision_macro",
    "recall_macro",
    "roc_auc_ovr",
]

ALL_METRICS = PRIMARY_METRICS + ADDITIONAL_METRICS

VALIDATION_ROWS = []

def add_check(
    check,
    expected,
    observed,
    status,
    dataset_id=None,
    protocol=None,
    model_id=None,
    artifact_name=None,
    path=None,
    invalid_values=None,
    details=None,
):
    VALIDATION_ROWS.append({
        "check": check,
        "dataset_id": dataset_id,
        "protocol": protocol,
        "model_id": model_id,
        "artifact_name": artifact_name,
        "path": str(path) if path is not None else None,
        "expected": expected,
        "observed": observed,
        "invalid_values": invalid_values,
        "details": details,
        "status": status,
    })

# --------------------------------------------------------------------------------
# 2. LOAD AUTHORITATIVE PERSISTED RESULTS
# --------------------------------------------------------------------------------

if not SOURCE_RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"Authoritative utility results not found: {SOURCE_RESULTS_PATH}"
    )

RESULTS_VALIDATION_DF = pd.read_csv(SOURCE_RESULTS_PATH)

# Normalize source column names to the canonical validation schema.
column_aliases = {
    "dataset": "dataset_id",
    "model": "model_id",
    "evaluation": "protocol",
    "macro_precision": "precision_macro",
    "macro_recall": "recall_macro",
}

for old_name, new_name in column_aliases.items():
    if old_name in RESULTS_VALIDATION_DF.columns:
        if new_name not in RESULTS_VALIDATION_DF.columns:
            RESULTS_VALIDATION_DF = RESULTS_VALIDATION_DF.rename(
                columns={old_name: new_name}
            )

required_columns = {
    "dataset_id",
    "model_id",
    "protocol",
    "seed",
    "status",
    *ALL_METRICS,
}

missing_columns = required_columns - set(RESULTS_VALIDATION_DF.columns)

if missing_columns:
    raise ValueError(
        "Persisted results are missing required columns: "
        f"{sorted(missing_columns)}"
    )

# Normalize identifiers and status.
for col in ["dataset_id", "model_id"]:
    RESULTS_VALIDATION_DF[col] = (
        RESULTS_VALIDATION_DF[col].astype("string").str.strip()
    )

RESULTS_VALIDATION_DF["protocol"] = (
    RESULTS_VALIDATION_DF["protocol"]
    .astype("string")
    .str.strip()
    .str.upper()
)

RESULTS_VALIDATION_DF["status"] = (
    RESULTS_VALIDATION_DF["status"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# SUCCESS is the persisted source convention; PASS is used internally
# for the canonical validation status.
RESULTS_VALIDATION_DF["validation_status"] = (
    RESULTS_VALIDATION_DF["status"].map({
        "SUCCESS": "PASS",
        "PASS": "PASS",
    }).fillna("FAIL")
)

RESULTS_VALIDATION_DF["seed"] = pd.to_numeric(
    RESULTS_VALIDATION_DF["seed"], errors="coerce"
)

if RESULTS_VALIDATION_DF["seed"].isna().any():
    raise ValueError("Missing or invalid experiment seed values detected.")

RESULTS_VALIDATION_DF["seed"] = (
    RESULTS_VALIDATION_DF["seed"].astype(int)
)

RESULTS_VALIDATION_DF["repetition"] = (
    RESULTS_VALIDATION_DF["seed"].map(SEED_TO_REPETITION)
)

if RESULTS_VALIDATION_DF["repetition"].isna().any():
    bad_seeds = sorted(
        RESULTS_VALIDATION_DF.loc[
            RESULTS_VALIDATION_DF["repetition"].isna(), "seed"
        ].unique().tolist()
    )
    raise ValueError(f"Unexpected experiment seeds: {bad_seeds}")

RESULTS_VALIDATION_DF["repetition"] = (
    RESULTS_VALIDATION_DF["repetition"].astype(int)
)

for metric in ALL_METRICS:
    RESULTS_VALIDATION_DF[metric] = pd.to_numeric(
        RESULTS_VALIDATION_DF[metric], errors="coerce"
    )

# --------------------------------------------------------------------------------
# 3. EXPECTED RUN COUNT AND SUCCESSFUL RUNS
# --------------------------------------------------------------------------------

expected_runs = (
    len(DATASETS_EXPECTED)
    * len(PROTOCOLS_EXPECTED)
    * len(MODELS_EXPECTED)
    * N_REPETITIONS
)

actual_runs = len(RESULTS_VALIDATION_DF)

successful_mask = RESULTS_VALIDATION_DF["validation_status"].eq("PASS")
successful_results = RESULTS_VALIDATION_DF.loc[successful_mask].copy()

successful_runs = len(successful_results)
failed_runs = actual_runs - successful_runs

add_check(
    check="expected_run_count",
    expected=expected_runs,
    observed=actual_runs,
    status="PASS" if actual_runs == expected_runs else "FAIL",
)

add_check(
    check="successful_run_count",
    expected=expected_runs,
    observed=successful_runs,
    status="PASS" if successful_runs == expected_runs else "FAIL",
    details="SUCCESS/PASS source statuses are normalized to canonical PASS.",
)

# Validate the full expected dataset/model/protocol coverage.
observed_datasets = set(
    RESULTS_VALIDATION_DF["dataset_id"].dropna().tolist()
)
observed_models = set(
    RESULTS_VALIDATION_DF["model_id"].dropna().tolist()
)
observed_protocols = set(
    RESULTS_VALIDATION_DF["protocol"].dropna().tolist()
)

add_check(
    "dataset_coverage",
    sorted(DATASETS_EXPECTED),
    sorted(observed_datasets),
    "PASS" if observed_datasets == set(DATASETS_EXPECTED) else "FAIL",
)

add_check(
    "model_coverage",
    sorted(MODELS_EXPECTED),
    sorted(observed_models),
    "PASS" if observed_models == set(MODELS_EXPECTED) else "FAIL",
)

add_check(
    "protocol_coverage",
    sorted(PROTOCOLS_EXPECTED),
    sorted(observed_protocols),
    "PASS" if observed_protocols == set(PROTOCOLS_EXPECTED) else "FAIL",
)

# --------------------------------------------------------------------------------
# 4. DUPLICATE RUN-KEY VALIDATION
# --------------------------------------------------------------------------------

expected_keys = [
    "dataset_id",
    "protocol",
    "model_id",
    "repetition",
]

duplicate_count = int(
    RESULTS_VALIDATION_DF.duplicated(expected_keys).sum()
)

add_check(
    check="duplicate_run_keys",
    expected=0,
    observed=duplicate_count,
    status="PASS" if duplicate_count == 0 else "FAIL",
)

# --------------------------------------------------------------------------------
# 5. REPETITION COVERAGE FOR EACH EXPERIMENT GROUP
# --------------------------------------------------------------------------------

for dataset_id in DATASETS_EXPECTED:
    for protocol in PROTOCOLS_EXPECTED:
        for model_id in MODELS_EXPECTED:

            subset = RESULTS_VALIDATION_DF[
                (RESULTS_VALIDATION_DF["dataset_id"] == dataset_id)
                & (RESULTS_VALIDATION_DF["protocol"] == protocol)
                & (RESULTS_VALIDATION_DF["model_id"] == model_id)
            ]

            valid_repetitions = int(
                subset.loc[
                    subset["validation_status"].eq("PASS"),
                    "repetition",
                ].nunique()
            )

            group_status = (
                "PASS"
                if valid_repetitions == N_REPETITIONS
                and len(subset) == N_REPETITIONS
                else "FAIL"
            )

            add_check(
                check="repetition_coverage",
                dataset_id=dataset_id,
                protocol=protocol,
                model_id=model_id,
                expected=N_REPETITIONS,
                observed=valid_repetitions,
                status=group_status,
                details=f"Records in group: {len(subset)}",
            )

# --------------------------------------------------------------------------------
# 6. PRIMARY AND ADDITIONAL METRIC VALIDATION
# --------------------------------------------------------------------------------

for metric in ALL_METRICS:
    values = successful_results[metric]

    non_missing = values.dropna()
    finite_mask = np.isfinite(non_missing.to_numpy(dtype=float))
    non_finite_count = int((~finite_mask).sum())

    out_of_range_count = int(
        ((non_missing < 0) | (non_missing > 1)).sum()
    )

    missing_count = int(values.isna().sum())
    invalid_count = non_finite_count + out_of_range_count

    metric_status = (
        "PASS"
        if successful_runs > 0
        and invalid_count == 0
        and missing_count == 0
        else "FAIL"
    )

    add_check(
        check=f"{metric}_validity",
        expected="Finite, non-missing values within [0, 1]",
        observed=len(non_missing),
        invalid_values=invalid_count + missing_count,
        status=metric_status,
        details=(
            f"Missing={missing_count}; "
            f"non-finite={non_finite_count}; "
            f"out-of-range={out_of_range_count}"
        ),
    )

# --------------------------------------------------------------------------------
# 7. RUNTIME VALIDATION — CHECK THE RUNTIME ARTIFACT SEPARATELY
# --------------------------------------------------------------------------------

if not RUNTIME_PATH.exists():
    add_check(
        check="runtime_artifact_available",
        expected="Existing non-empty runtime_results.csv",
        observed="MISSING",
        status="FAIL",
        path=RUNTIME_PATH,
    )
else:
    runtime_df = pd.read_csv(RUNTIME_PATH)

    runtime_aliases = {
        "dataset": "dataset_id",
        "model": "model_id",
        "evaluation": "protocol",
        "runtime": "runtime_seconds",
        "mean_runtime_seconds": "runtime_seconds",
    }

    for old_name, new_name in runtime_aliases.items():
        if old_name in runtime_df.columns and new_name not in runtime_df.columns:
            runtime_df = runtime_df.rename(
                columns={old_name: new_name}
            )

    if "runtime_seconds" not in runtime_df.columns:
        add_check(
            check="runtime_validity",
            expected="Finite, non-negative runtime values",
            observed="runtime_seconds column missing",
            status="FAIL",
            path=RUNTIME_PATH,
        )
    else:
        runtime_values = pd.to_numeric(
            runtime_df["runtime_seconds"], errors="coerce"
        )

        missing_runtime = int(runtime_values.isna().sum())
        finite_runtime = np.isfinite(
            runtime_values.dropna().to_numpy(dtype=float)
        )
        non_finite_runtime = int((~finite_runtime).sum())
        negative_runtime = int(
            (runtime_values.dropna() < 0).sum()
        )

        invalid_runtime = (
            missing_runtime + non_finite_runtime + negative_runtime
        )

        add_check(
            check="runtime_validity",
            expected="Finite, non-negative runtime values",
            observed=int(runtime_values.notna().sum()),
            invalid_values=invalid_runtime,
            status="PASS" if invalid_runtime == 0
            and len(runtime_values) > 0 else "FAIL",
            path=RUNTIME_PATH,
            details=(
                f"Missing={missing_runtime}; "
                f"non-finite={non_finite_runtime}; "
                f"negative={negative_runtime}"
            ),
        )

# --------------------------------------------------------------------------------
# 8. CONFIGURED OUTPUT ARTIFACT VALIDATION
# --------------------------------------------------------------------------------

OUTPUT_FILES = {
    "raw_results": RESULTS_DIR / "ml_utility_results.csv",
    "run_log": NB15_ROOT / "logs" / "model_run_log.csv",
    "accuracy": RESULTS_DIR / "accuracy_results.csv",
    "macro_f1": RESULTS_DIR / "macro_f1_results.csv",
    "weighted_f1": RESULTS_DIR / "weighted_f1_results.csv",
    "additional_metrics": RESULTS_DIR / "additional_metrics_results.csv",
    "runtime": RESULTS_DIR / "runtime_results.csv",
    "runtime_summary": NB15_ROOT / "aggregations" / "runtime_summary.csv",
    "repetition_aggregation": NB15_ROOT / "aggregations" / "repetition_aggregation.csv",
    "model_comparison": NB15_ROOT / "aggregations" / "model_comparison.csv",
    "task_validation": VALIDATION_DIR / "task_validation.csv",
    "real_data_audit": NB15_ROOT / "inputs" / "real_data_audit.csv",
    "synthetic_data_audit": NB15_ROOT / "inputs" / "synthetic_data_audit.csv",
    "additional_metric_coverage": VALIDATION_DIR / "additional_metric_coverage.csv",
}

artifact_failures = 0

for artifact_name, artifact_path in OUTPUT_FILES.items():
    exists = artifact_path.exists()
    nonempty = exists and artifact_path.stat().st_size > 0
    ok = bool(exists and nonempty)

    artifact_failures += int(not ok)

    add_check(
        check="artifact_exists",
        artifact_name=artifact_name,
        path=artifact_path,
        expected="PRESENT and non-empty",
        observed=(
            "PRESENT"
            if ok
            else "MISSING"
            if not exists
            else "EMPTY"
        ),
        status="PASS" if ok else "FAIL",
    )

# --------------------------------------------------------------------------------
# 9. BUILD AND SAVE VALIDATION REPORT
# --------------------------------------------------------------------------------

VALIDATION_DF = pd.DataFrame(VALIDATION_ROWS)

failed_checks = int(
    VALIDATION_DF["status"].ne("PASS").sum()
)

if failed_checks == 0 and failed_runs == 0:
    validation_status = "PASS"
elif failed_checks == 0:
    validation_status = "PARTIAL"
else:
    validation_status = "FAIL"

VALIDATION_DF.to_csv(VALIDATION_PATH, index=False)

# Verify that the report was actually persisted.
if not VALIDATION_PATH.exists() or VALIDATION_PATH.stat().st_size == 0:
    raise IOError(
        f"Validation report was not saved correctly: {VALIDATION_PATH}"
    )

# --------------------------------------------------------------------------------
# 10. FINAL SUMMARY
# --------------------------------------------------------------------------------

print("-" * 100)
print("SECTION 19 — COVERAGE VALIDATION SUMMARY")
print("-" * 100)

print("Authoritative source:", SOURCE_RESULTS_PATH)
print("Expected runs:", expected_runs)
print("Actual runs:", actual_runs)
print("Successful runs:", successful_runs)
print("Failed runs:", failed_runs)
print("Duplicate run keys:", duplicate_count)
print("Expected experiment groups:", (
    len(DATASETS_EXPECTED)
    * len(PROTOCOLS_EXPECTED)
    * len(MODELS_EXPECTED)
))
print("Artifact failures:", artifact_failures)
print("Failed validation checks:", failed_checks)
print("Validation report:", VALIDATION_PATH)
print("Validation status:", validation_status)

print("\nValidation report:")
print(VALIDATION_DF.to_string(index=False))

if validation_status == "FAIL":
    raise RuntimeError(
        "Notebook 15 coverage validation failed. "
        f"Inspect {VALIDATION_PATH} for the specific failed checks."
    )


SECTION 19 — VALIDATE COVERAGE
----------------------------------------------------------------------------------------------------
SECTION 19 — COVERAGE VALIDATION SUMMARY
----------------------------------------------------------------------------------------------------
Authoritative source: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/results/ml_utility_results.csv
Expected runs: 90
Actual runs: 90
Successful runs: 90
Failed runs: 0
Duplicate run keys: 0
Expected experiment groups: 18
Artifact failures: 0
Failed validation checks: 0
Validation report: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/validation/utility_coverage_validation.csv
Validation status: PASS

Validation report:
                   check     dataset_id protocol            model_id              artifact_name                                                                                                            path                                       expected  

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [108]:

# ==================================================================================================
# SECTION 20 — COMPLETION SUMMARY
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import platform
import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 20 — COMPLETION SUMMARY")
print("=" * 100)

# --------------------------------------------------------------------------------
# 1. Resolve authoritative artifacts
# --------------------------------------------------------------------------------

RESULTS_PATH = NB15_ROOT / "results" / "ml_utility_results.csv"
VALIDATION_PATH = (
    NB15_ROOT / "validation" / "utility_coverage_validation.csv"
)
SUMMARY_PATH = NB15_ROOT / "notebook_15_completion_summary.json"

if not RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"Authoritative results file not found: {RESULTS_PATH}"
    )

if not VALIDATION_PATH.exists():
    raise FileNotFoundError(
        f"Section 19 validation report not found: {VALIDATION_PATH}"
    )

# Always reload persisted results. Do not rely on stale RESULTS_DF.
COMPLETION_RESULTS_DF = pd.read_csv(RESULTS_PATH)
VALIDATION_REPORT_DF = pd.read_csv(VALIDATION_PATH)

# --------------------------------------------------------------------------------
# 2. Normalize and validate run statuses
# --------------------------------------------------------------------------------

if "status" not in COMPLETION_RESULTS_DF.columns:
    raise ValueError("Authoritative results have no 'status' column.")

status_values = (
    COMPLETION_RESULTS_DF["status"]
    .astype("string")
    .str.strip()
    .str.upper()
)

success_mask = status_values.isin(["SUCCESS", "PASS"])
failure_mask = ~success_mask

successful_runs = int(success_mask.sum())
failed_runs = int(failure_mask.sum())
actual_runs = int(len(COMPLETION_RESULTS_DF))

# Use the existing experiment design.
expected_runs = int(
    len(DATASETS) * len(ACTIVE_MODELS) * 2 * N_REPETITIONS
)

if actual_runs != expected_runs:
    raise ValueError(
        f"Run-count mismatch: expected {expected_runs}, observed {actual_runs}."
    )

# --------------------------------------------------------------------------------
# 3. Confirm Section 19 validation result
# --------------------------------------------------------------------------------

if "status" not in VALIDATION_REPORT_DF.columns:
    raise ValueError("Validation report has no 'status' column.")

validation_status_values = (
    VALIDATION_REPORT_DF["status"]
    .astype("string")
    .str.strip()
    .str.upper()
)

failed_validation_checks = int(
    (~validation_status_values.isin(["PASS", "SUCCESS"])).sum()
)

validation_status = (
    "PASS"
    if len(VALIDATION_REPORT_DF) > 0
    and failed_validation_checks == 0
    else "FAIL"
)

# --------------------------------------------------------------------------------
# 4. Determine completion status
# --------------------------------------------------------------------------------

completion_status = (
    "COMPLETED"
    if failed_runs == 0 and validation_status == "PASS"
    else "PARTIAL"
)

# --------------------------------------------------------------------------------
# 5. Build completion summary
# --------------------------------------------------------------------------------

summary = {
    "notebook_id": NOTEBOOK_ID,
    "notebook_title": NOTEBOOK_TITLE,
    "status": completion_status,
    "validation_status": validation_status,
    "failed_validation_checks": failed_validation_checks,
    "project_root": str(PROJECT_ROOT),
    "notebook_root": str(NB15_ROOT),
    "datasets": list(DATASETS.keys()),
    "targets": {
        dataset_id: metadata["target"]
        for dataset_id, metadata in DATASETS.items()
    },
    "protocols": ["TSTR", "TRTR"],
    "models_configured": list(MODEL_REGISTRY.keys()),
    "models_executed": list(ACTIVE_MODELS),
    "primary_metrics": list(PRIMARY_METRICS),
    "additional_metrics": [
        "precision_macro",
        "recall_macro",
        "roc_auc_ovr",
    ],
    "expected_runs": expected_runs,
    "actual_runs": actual_runs,
    "successful_runs": successful_runs,
    "failed_runs": failed_runs,
    "repetition_seeds": list(REPETITION_SEEDS),
    "synthetic_generation_repetitions": 1,
    "predictive_model_repetitions": N_REPETITIONS,
    "synthetic_repetition_note": (
        "Five predictive-model seed repetitions use one canonical synthetic "
        "dataset per source dataset. Independent synthetic-generation "
        "variability is not estimated."
    ),
    "real_test_policy": (
        "The same held-out real test split is used for TSTR and TRTR; "
        "test data are excluded from fitting."
    ),
    "configuration_path": str(CONFIG_PATH),
    "raw_results_path": str(RESULTS_PATH),
    "repetition_aggregation_path": str(
        NB15_ROOT / "aggregations" / "repetition_aggregation.csv"
    ),
    "model_comparison_path": str(
        NB15_ROOT / "aggregations" / "model_comparison.csv"
    ),
    "validation_report_path": str(VALIDATION_PATH),
    "fingerprint_manifest_path": str(FINGERPRINT_PATH),
    "completed_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
}

# --------------------------------------------------------------------------------
# 6. Save and reload verification
# --------------------------------------------------------------------------------

NB15_ROOT.mkdir(parents=True, exist_ok=True)

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)

with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
    reloaded_summary = json.load(f)

if reloaded_summary["notebook_id"] != NOTEBOOK_ID:
    raise ValueError("Completion summary notebook ID verification failed.")

if reloaded_summary["status"] != completion_status:
    raise ValueError("Completion summary status verification failed.")

if reloaded_summary["successful_runs"] != successful_runs:
    raise ValueError("Completion summary successful-run verification failed.")

if reloaded_summary["failed_runs"] != failed_runs:
    raise ValueError("Completion summary failed-run verification failed.")

if reloaded_summary["validation_status"] != validation_status:
    raise ValueError("Completion summary validation-status verification failed.")

# --------------------------------------------------------------------------------
# 7. Final report
# --------------------------------------------------------------------------------

print("-" * 100)
print("NOTEBOOK 15 COMPLETION SUMMARY")
print("-" * 100)
print(f"Authoritative results : {RESULTS_PATH}")
print(f"Expected runs         : {expected_runs}")
print(f"Actual runs           : {actual_runs}")
print(f"Successful runs       : {successful_runs}")
print(f"Failed runs           : {failed_runs}")
print(f"Validation status     : {validation_status}")
print(f"Failed validation checks: {failed_validation_checks}")
print(f"Completion status     : {completion_status}")
print(f"Summary saved         : {SUMMARY_PATH}")
print("Summary reload verification: PASS")

print("\nSaved summary JSON:")
print(json.dumps(reloaded_summary, indent=2))

if completion_status != "COMPLETED":
    raise RuntimeError(
        "Notebook 15 is PARTIAL. Review the run statuses and validation report."
    )

print("\nSECTION 20 STATUS: PASS")


SECTION 20 — COMPLETION SUMMARY
----------------------------------------------------------------------------------------------------
NOTEBOOK 15 COMPLETION SUMMARY
----------------------------------------------------------------------------------------------------
Authoritative results : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/results/ml_utility_results.csv
Expected runs         : 90
Actual runs           : 90
Successful runs       : 90
Failed runs           : 0
Validation status     : PASS
Failed validation checks: 0
Completion status     : COMPLETED
Summary saved         : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15/notebook_15_completion_summary.json
Summary reload verification: PASS

Saved summary JSON:
{
  "notebook_id": "notebook_15",
  "notebook_title": "ML Utility / TSTR Evaluation",
  "status": "COMPLETED",
  "validation_status": "PASS",
  "failed_validation_checks": 0,
  "project_root": "/content/drive/MyDrive/SPP_GAN_Re

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [109]:

from pathlib import Path
import json
import pandas as pd

NB15_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_15"
)

required_files = [
    NB15_ROOT / "results" / "ml_utility_results.csv",
    NB15_ROOT / "aggregations" / "repetition_aggregation.csv",
    NB15_ROOT / "aggregations" / "model_comparison.csv",
    NB15_ROOT / "validation" / "utility_coverage_validation.csv",
    NB15_ROOT / "validation" / "artifact_fingerprints.csv",
    NB15_ROOT / "notebook_15_completion_summary.json",
]

print("=" * 90)
print("NOTEBOOK 15 — PERSISTENCE AUDIT")
print("=" * 90)

missing = []
for path in required_files:
    valid = path.is_file() and path.stat().st_size > 0
    print(f"{'PASS' if valid else 'FAIL'} | {path.name}")
    if not valid:
        missing.append(str(path))

if missing:
    raise FileNotFoundError(
        "Required Notebook 15 artifacts are missing:\n" + "\n".join(missing)
    )

with open(
    NB15_ROOT / "notebook_15_completion_summary.json",
    "r",
    encoding="utf-8",
) as f:
    summary = json.load(f)

results = pd.read_csv(NB15_ROOT / "results" / "ml_utility_results.csv")
status = results["status"].astype(str).str.strip().str.upper()

assert summary["status"] == "COMPLETED"
assert len(results) == 90
assert status.isin(["SUCCESS", "PASS"]).all()

fingerprints = pd.read_csv(
    NB15_ROOT / "validation" / "artifact_fingerprints.csv"
)

print("\nResults records:", len(results))
print("Successful records:", int(status.isin(["SUCCESS", "PASS"]).sum()))
print("Fingerprint manifest rows:", len(fingerprints))
print("Completion summary:", summary["status"])
print("\nPERSISTENCE AUDIT: PASS")


NOTEBOOK 15 — PERSISTENCE AUDIT
PASS | ml_utility_results.csv
PASS | repetition_aggregation.csv
PASS | model_comparison.csv
PASS | utility_coverage_validation.csv
PASS | artifact_fingerprints.csv
PASS | notebook_15_completion_summary.json

Results records: 90
Successful records: 90
Fingerprint manifest rows: 18
Completion summary: COMPLETED

PERSISTENCE AUDIT: PASS


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
